# FIT5196 Assessment 1 - Solution Notebook

**Group:** Group050

**Team members:**  
Xingao Zhan ID: 36354775; <br>
Yu Wang ID: 26900867; <br>
Keshu Zhang ID: 36436763; <br>
Qingzhuo Zhao ID: 26662841

**Stage label:** Specification Tasks 1–4 complete  
**Version date:** 2026-08-25  
**Status:** Six canonical tables, fixed Task 3 text interface and the integrated Task 4 validation register are executable from a fresh kernel.  
**Scope:** Task 5 and Task 6 are not implemented in this notebook checkpoint.


## 0. Configuration and reproducibility

Keep all configurable paths in this section. The final notebook must run with
**Restart and Run All** without manual file edits or network access.


In [1]:
from pathlib import Path
import sys

GROUP_ID = "Group050"
PROJECT_DIR = Path.cwd()

# Official default: support files and raw_input are beside the notebook.
# Development fallback: keep the supplied package read-only in a nearby
# Group050_A1 folder while all generated and edited files stay here.
# Search only relative parent levels so live and stage-backup copies run
# without embedding any machine-specific absolute path.
PACKAGE_DIR = PROJECT_DIR
if not (PACKAGE_DIR / "raw_input").is_dir():
    for parent_dir in PROJECT_DIR.parents:
        candidate_package = parent_dir / "Group050_A1"
        if (candidate_package / "raw_input").is_dir():
            PACKAGE_DIR = candidate_package
            break

INPUT_DIR = PACKAGE_DIR / "raw_input"
OUTPUT_DIR = PROJECT_DIR / "outputs"
TEMPLATE_DIR = PACKAGE_DIR

JSON_PATH = INPUT_DIR / f"{GROUP_ID}_commerce.json"
XML_PATH = INPUT_DIR / f"{GROUP_ID}_operations.xml"
DATA_DICTIONARY_PATH = TEMPLATE_DIR / "public_data_dictionary.csv"
MAPPING_TEMPLATE_PATH = TEMPLATE_DIR / "A1_source_to_target_mapping_template.csv"
MAPPING_OUTPUT_PATH = PROJECT_DIR / f"{GROUP_ID}_source_to_target_mapping.csv"
PUBLIC_TEXT_TESTS_PATH = TEMPLATE_DIR / "A1_public_text_test_cases.csv"

required_paths = {
    "JSON source": JSON_PATH,
    "XML source": XML_PATH,
    "public data dictionary": DATA_DICTIONARY_PATH,
    "mapping template": MAPPING_TEMPLATE_PATH,
    "public text tests": PUBLIC_TEXT_TESTS_PATH,
}
missing_paths = {
    label: path for label, path in required_paths.items() if not path.is_file()
}
if missing_paths:
    missing_text = "\n".join(
        f"- {label}: {path}" for label, path in missing_paths.items()
    )
    raise FileNotFoundError(f"Required assignment files are missing:\n{missing_text}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

package_display = "." if PACKAGE_DIR == PROJECT_DIR else "Group050_A1 (relative parent search)"
print("Working directory: .")
print(f"Read-only package directory: {package_display}")
print(f"JSON source: {JSON_PATH.name}")
print(f"XML source: {XML_PATH.name}")
print("Output directory: outputs")


Working directory: .
Read-only package directory: Group050_A1 (relative parent search)
JSON source: Group050_commerce.json
XML source: Group050_operations.xml
Output directory: outputs


### 0.1 Environment and dependencies

Import the libraries used by your submitted workflow. Record non-standard
dependencies in `requirements.txt`.


In [2]:
# EVIDENCE: SEC-0.1-ENVIRONMENT
import json
import platform
import sys
import xml.etree.ElementTree as ET
from collections import Counter

import numpy as np
import pandas as pd

def show(frame, title=None):
    if title:
        print(f"\n{title}")
    print(frame.to_string(index=False))

environment = pd.DataFrame(
    [
        {"component": "Python", "version": platform.python_version()},
        {"component": "pandas", "version": pd.__version__},
        {"component": "NumPy", "version": np.__version__},
        {"component": "XML parser", "version": "xml.etree.ElementTree (stdlib)"},
        {"component": "JSON parser", "version": "json (stdlib)"},
    ]
)
show(environment, "Environment and dependencies")



Environment and dependencies
  component                        version
     Python                         3.14.6
     pandas                          3.0.5
      NumPy                          2.5.1
 XML parser xml.etree.ElementTree (stdlib)
JSON parser                  json (stdlib)


## 1. Parse and profile the two sources

Use structured JSON and XML parsers. Record source grains, nested/repeated
structures, candidate keys, formats, missing-value conventions and evidence of
within-source or cross-source overlap.


### 1.1 JSON structure and profile


In [3]:
# EVIDENCE: SEC-1.1-JSON-PROFILE
with JSON_PATH.open(encoding="utf-8") as handle:
    json_source = json.load(handle)

json_customers = json_source["customerProfiles"]
json_orders = json_source["orders"]
json_items = [item for order in json_orders for item in order["shoppingCart"]]
json_deliveries = [order["delivery"] for order in json_orders]
json_reviews = json_source["productReviews"]

def profile_records(source_collection, grain, records, key_field):
    keys = [record.get(key_field) for record in records]
    non_missing = [key for key in keys if key not in (None, "")]
    frequencies = Counter(non_missing)
    fingerprints = Counter(
        json.dumps(record, sort_keys=True, ensure_ascii=False) for record in records
    )
    return {
        "source_collection": source_collection,
        "grain": grain,
        "candidate_key": key_field,
        "rows": len(records),
        "missing_key": len(keys) - len(non_missing),
        "unique_key": len(frequencies),
        "duplicate_key_groups": sum(count > 1 for count in frequencies.values()),
        "duplicate_extra_rows": len(non_missing) - len(frequencies),
        "exact_duplicate_groups": sum(count > 1 for count in fingerprints.values()),
        "exact_duplicate_extra_rows": len(records) - len(fingerprints),
    }

json_profile = pd.DataFrame(
    [
        profile_records("customerProfiles[]", "one customer profile", json_customers, "customerID"),
        profile_records("orders[]", "one source order", [order["header"] for order in json_orders], "orderID"),
        profile_records("orders[].shoppingCart[]", "one source order item", json_items, "orderItemID"),
        profile_records("orders[].delivery", "one source delivery", json_deliveries, "deliveryID"),
        profile_records("productReviews[]", "one source review", json_reviews, "reviewID"),
    ]
)
show(json_profile, "JSON collection profile")

json_structure = pd.DataFrame(
    [
        {"path": "customerProfiles[]", "representation": "repeated root array", "fields": ", ".join(json_customers[0].keys())},
        {"path": "orders[]", "representation": "repeated root array with header, shoppingCart[] and delivery", "fields": ", ".join(json_orders[0].keys())},
        {"path": "orders[].header", "representation": "nested object", "fields": ", ".join(json_orders[0]["header"].keys())},
        {"path": "orders[].shoppingCart[]", "representation": "nested repeated array", "fields": ", ".join(json_items[0].keys())},
        {"path": "orders[].delivery", "representation": "nested object", "fields": ", ".join(json_deliveries[0].keys())},
        {"path": "productReviews[]", "representation": "repeated root array", "fields": ", ".join(json_reviews[0].keys())},
    ]
)
show(json_structure, "JSON nesting and fields")

first_header = json_orders[0]["header"]
first_delivery = json_orders[0]["delivery"]
json_formats = pd.DataFrame(
    [
        {"concept": "timestamp", "path": "orders[].header.orderTimestamp", "example": repr(first_header["orderTimestamp"]), "Python type": type(first_header["orderTimestamp"]).__name__},
        {"concept": "date", "path": "orders[].delivery.dispatchDate", "example": repr(first_delivery["dispatchDate"]), "Python type": type(first_delivery["dispatchDate"]).__name__},
        {"concept": "boolean", "path": "orders[].header.expeditedDelivery", "example": repr(first_header["expeditedDelivery"]), "Python type": type(first_header["expeditedDelivery"]).__name__},
        {"concept": "currency value", "path": "orders[].header.orderPrice", "example": repr(first_header["orderPrice"]), "Python type": type(first_header["orderPrice"]).__name__},
        {"concept": "percentage points", "path": "orders[].header.couponDiscount", "example": repr(first_header["couponDiscount"]), "Python type": type(first_header["couponDiscount"]).__name__},
        {"concept": "missing optional string", "path": "orders[].header.couponCode", "example": "empty string count=" + str(sum(order["header"].get("couponCode") == "" for order in json_orders)), "Python type": "str"},
    ]
)
show(json_formats, "JSON source-format evidence")

json_collections = {
    "customerProfiles[]": json_customers,
    "orders[].header": [order["header"] for order in json_orders],
    "orders[].shoppingCart[]": json_items,
    "orders[].delivery": json_deliveries,
    "productReviews[]": json_reviews,
}
json_missing_rows = []
for source_collection, records in json_collections.items():
    fields = sorted({field for record in records for field in record})
    for field in fields:
        absent_count = sum(field not in record for record in records)
        null_count = sum(record.get(field) is None for record in records if field in record)
        empty_string_count = sum(record.get(field) == "" for record in records if field in record)
        if absent_count or null_count or empty_string_count:
            json_missing_rows.append(
                {
                    "source_collection": source_collection,
                    "field": field,
                    "absent_key": absent_count,
                    "JSON null": null_count,
                    "empty_string": empty_string_count,
                }
            )
json_missing_profile = pd.DataFrame(json_missing_rows)
show(json_missing_profile, "JSON missing-value conventions (non-zero counts only)")



JSON collection profile
      source_collection                 grain candidate_key  rows  missing_key  unique_key  duplicate_key_groups  duplicate_extra_rows  exact_duplicate_groups  exact_duplicate_extra_rows
     customerProfiles[]  one customer profile    customerID   500            0         500                     0                     0                       0                           0
               orders[]      one source order       orderID  2818            0        2750                    68                    68                      68                          68
orders[].shoppingCart[] one source order item   orderItemID  8884            0        8666                   218                   218                     218                         218
      orders[].delivery   one source delivery    deliveryID  2818            0        2750                    68                    68                      68                          68
       productReviews[]     one source r

### 1.2 XML structure and profile


In [4]:
# EVIDENCE: SEC-1.2-XML-PROFILE
xml_tree = ET.parse(XML_PATH)
xml_root = xml_tree.getroot()

xml_orders = xml_root.findall("./Orders/Order")
xml_headers = [order.find("Header") for order in xml_orders]
xml_items = xml_root.findall("./Orders/Order/Shopping_Cart/Item")
xml_deliveries = xml_root.findall("./Orders/Order/Delivery")
xml_products = xml_root.findall("./ProductCatalogue/Product")
xml_reviews = xml_root.findall("./ProductReviews/Review")
xml_warehouses = list(xml_root.find("WarehouseDirectory"))

def element_records(elements):
    return [{child.tag: child.text or "" for child in element} for element in elements]

xml_header_records = element_records(xml_headers)
xml_item_records = element_records(xml_items)
xml_delivery_records = element_records(xml_deliveries)
xml_product_records = element_records(xml_products)
xml_review_records = element_records(xml_reviews)
xml_warehouse_records = element_records(xml_warehouses)

xml_profile = pd.DataFrame(
    [
        profile_records("/OperationsExport/Orders/Order/Header", "one source order", xml_header_records, "Order_ID"),
        profile_records("/OperationsExport/Orders/Order/Shopping_Cart/Item", "one source order item", xml_item_records, "Order_Item_ID"),
        profile_records("/OperationsExport/Orders/Order/Delivery", "one source delivery", xml_delivery_records, "Delivery_ID"),
        profile_records("/OperationsExport/ProductCatalogue/Product", "one product", xml_product_records, "Product_ID"),
        profile_records("/OperationsExport/ProductReviews/Review", "one source review", xml_review_records, "Review_ID"),
        profile_records("/OperationsExport/WarehouseDirectory/*", "one warehouse reference", xml_warehouse_records, "Name"),
    ]
)
show(xml_profile, "XML collection profile")

xml_structure = pd.DataFrame(
    [
        {"path": "/OperationsExport", "representation": "root element", "child elements": ", ".join(child.tag for child in xml_root)},
        {"path": "/OperationsExport/Orders/Order", "representation": "repeated element with Header, Shopping_Cart and Delivery", "child elements": ", ".join(child.tag for child in xml_orders[0])},
        {"path": "/OperationsExport/Orders/Order/Shopping_Cart/Item", "representation": "nested repeated element", "child elements": ", ".join(xml_item_records[0].keys())},
        {"path": "/OperationsExport/ProductCatalogue/Product", "representation": "repeated element", "child elements": ", ".join(xml_product_records[0].keys())},
        {"path": "/OperationsExport/ProductReviews/Review", "representation": "repeated element", "child elements": ", ".join(xml_review_records[0].keys())},
        {"path": "/OperationsExport/WarehouseDirectory/*", "representation": "source-specific reference collection", "child elements": ", ".join(xml_warehouse_records[0].keys())},
    ]
)
show(xml_structure, "XML nesting and fields")

first_xml_header = xml_header_records[0]
first_xml_delivery = xml_delivery_records[0]
xml_formats = pd.DataFrame(
    [
        {"concept": "timestamp", "path": ".../Header/Order_Timestamp", "example": repr(first_xml_header["Order_Timestamp"]), "XML representation": "text"},
        {"concept": "date", "path": ".../Delivery/Dispatch_Date", "example": repr(first_xml_delivery["Dispatch_Date"]), "XML representation": "text"},
        {"concept": "boolean", "path": ".../Header/Expedited_Delivery", "example": repr(first_xml_header["Expedited_Delivery"]), "XML representation": "Y/N text"},
        {"concept": "currency value", "path": ".../Header/Order_Price", "example": repr(first_xml_header["Order_Price"]), "XML representation": "currency-labelled text"},
        {"concept": "percentage", "path": ".../Header/Coupon_Discount", "example": repr(first_xml_header["Coupon_Discount"]), "XML representation": "percent-labelled text"},
        {"concept": "missing optional string", "path": ".../Header/Coupon_Code", "example": "empty element count=" + str(sum(record["Coupon_Code"] == "" for record in xml_header_records)), "XML representation": "empty element"},
    ]
)
show(xml_formats, "XML source-format evidence")

xml_collections = {
    "/OperationsExport/Orders/Order/Header": xml_header_records,
    "/OperationsExport/Orders/Order/Shopping_Cart/Item": xml_item_records,
    "/OperationsExport/Orders/Order/Delivery": xml_delivery_records,
    "/OperationsExport/ProductCatalogue/Product": xml_product_records,
    "/OperationsExport/ProductReviews/Review": xml_review_records,
}
xml_missing_rows = []
for source_collection, records in xml_collections.items():
    fields = sorted({field for record in records for field in record})
    for field in fields:
        absent_element_count = sum(field not in record for record in records)
        empty_element_count = sum(record.get(field) == "" for record in records)
        if absent_element_count or empty_element_count:
            xml_missing_rows.append(
                {
                    "source_collection": source_collection,
                    "field": field,
                    "absent_element": absent_element_count,
                    "empty_element": empty_element_count,
                }
            )
xml_missing_profile = pd.DataFrame(xml_missing_rows)
show(xml_missing_profile, "XML missing-value conventions (non-zero counts only)")



XML collection profile
                                source_collection                   grain candidate_key  rows  missing_key  unique_key  duplicate_key_groups  duplicate_extra_rows  exact_duplicate_groups  exact_duplicate_extra_rows
            /OperationsExport/Orders/Order/Header        one source order      Order_ID  2818            0        2750                    68                    68                      68                          68
/OperationsExport/Orders/Order/Shopping_Cart/Item   one source order item Order_Item_ID  8833            0        8622                   211                   211                     211                         211
          /OperationsExport/Orders/Order/Delivery     one source delivery   Delivery_ID  2818            0        2750                    68                    68                      68                          68
       /OperationsExport/ProductCatalogue/Product             one product    Product_ID  1000            0        10

### 1.3 Source comparison and assumptions


In [5]:
# EVIDENCE: SEC-1.3-SOURCE-COMPARISON
json_key_sets = {
    "orders": {record["orderID"] for record in [order["header"] for order in json_orders]},
    "order_items": {record["orderItemID"] for record in json_items},
    "deliveries": {record["deliveryID"] for record in json_deliveries},
    "product_reviews": {record["reviewID"] for record in json_reviews},
}
xml_key_sets = {
    "orders": {record["Order_ID"] for record in xml_header_records},
    "order_items": {record["Order_Item_ID"] for record in xml_item_records},
    "deliveries": {record["Delivery_ID"] for record in xml_delivery_records},
    "product_reviews": {record["Review_ID"] for record in xml_review_records},
}

overlap_rows = []
for entity in json_key_sets:
    json_keys = json_key_sets[entity]
    xml_keys = xml_key_sets[entity]
    overlap = json_keys & xml_keys
    overlap_rows.append(
        {
            "entity": entity,
            "JSON unique keys": len(json_keys),
            "XML unique keys": len(xml_keys),
            "cross-source overlap": len(overlap),
            "JSON only": len(json_keys - xml_keys),
            "XML only": len(xml_keys - json_keys),
            "union before field reconciliation": len(json_keys | xml_keys),
        }
    )
overlap_profile = pd.DataFrame(overlap_rows)
show(overlap_profile, "Cross-source key overlap")

combined_repeat_profile = pd.concat(
    [
        json_profile.assign(source="JSON"),
        xml_profile.assign(source="XML"),
    ],
    ignore_index=True,
)[
    ["source", "source_collection", "grain", "candidate_key", "rows", "missing_key", "unique_key", "duplicate_key_groups", "duplicate_extra_rows", "exact_duplicate_groups", "exact_duplicate_extra_rows"]
]
show(combined_repeat_profile, "Within-source repeat evidence")

source_coverage = pd.DataFrame(
    [
        {"target entity": "orders", "JSON evidence": "orders[].header", "XML evidence": "/OperationsExport/Orders/Order/Header", "coverage": "both"},
        {"target entity": "order_items", "JSON evidence": "orders[].shoppingCart[]", "XML evidence": "/OperationsExport/Orders/Order/Shopping_Cart/Item", "coverage": "both"},
        {"target entity": "customers", "JSON evidence": "customerProfiles[]", "XML evidence": "no full customer collection", "coverage": "JSON only"},
        {"target entity": "deliveries", "JSON evidence": "orders[].delivery", "XML evidence": "/OperationsExport/Orders/Order/Delivery", "coverage": "both"},
        {"target entity": "products", "JSON evidence": "product IDs only in related records", "XML evidence": "/OperationsExport/ProductCatalogue/Product", "coverage": "XML only for full entity"},
        {"target entity": "product_reviews", "JSON evidence": "productReviews[]", "XML evidence": "/OperationsExport/ProductReviews/Review", "coverage": "both"},
        {"target entity": "warehouse reference", "JSON evidence": "warehouse name in order header", "XML evidence": "/OperationsExport/WarehouseDirectory/*", "coverage": "source-specific helper"},
    ]
)
show(source_coverage, "Source coverage and source-specific collections")

key_relationships = pd.DataFrame(
    [
        {"entity / grain": "orders / one order", "candidate primary key": "order_id", "candidate foreign keys": "customer_id -> customers.customer_id", "source evidence": "header customer and order identifiers in both sources"},
        {"entity / grain": "order_items / one line item", "candidate primary key": "order_item_id", "candidate foreign keys": "order_id -> orders.order_id | product_id -> products.product_id", "source evidence": "nested shopping-cart identifiers in both sources"},
        {"entity / grain": "customers / one customer", "candidate primary key": "customer_id", "candidate foreign keys": "none", "source evidence": "customerProfiles[].customerID"},
        {"entity / grain": "deliveries / one delivery", "candidate primary key": "delivery_id", "candidate foreign keys": "order_id -> orders.order_id", "source evidence": "nested delivery identifiers in both sources"},
        {"entity / grain": "products / one product", "candidate primary key": "product_id", "candidate foreign keys": "none in the six-table target contract", "source evidence": "/OperationsExport/ProductCatalogue/Product/Product_ID"},
        {"entity / grain": "product_reviews / one review", "candidate primary key": "review_id", "candidate foreign keys": "order_id -> orders.order_id | order_item_id -> order_items.order_item_id | product_id -> products.product_id | customer_id -> customers.customer_id", "source evidence": "review identifiers in both sources"},
        {"entity / grain": "warehouse reference / one warehouse", "candidate primary key": "Name", "candidate foreign keys": "referenced by orders.nearest_warehouse", "source evidence": "/OperationsExport/WarehouseDirectory/*/Name"},
    ]
)
show(key_relationships, "Candidate keys and relationships to verify after reconciliation")

assumptions = pd.DataFrame(
    [
        {"assumption_id": "ASM-01", "decision before transformation": "Use the published table primary key as the stable business key at each target grain; do not invent identifiers."},
        {"assumption_id": "ASM-02", "decision before transformation": "Normalise comparable types and strings before comparing duplicates or cross-source overlap."},
        {"assumption_id": "ASM-03", "decision before transformation": "Retain one canonical row when all normalised non-missing values agree; record any disagreement as validation evidence rather than applying source precedence."},
        {"assumption_id": "ASM-04", "decision before transformation": "Preserve identifier leading zeros and source case; do not lower-case structured categories."},
        {"assumption_id": "ASM-05", "decision before transformation": "Parse JSON/XML structurally before applying regex to bounded narrative fields."},
        {"assumption_id": "ASM-06", "decision before transformation": "Use literal NaN only for prescribed missing string outputs; never substitute it into required numeric, boolean, primary-key or foreign-key fields."},
        {"assumption_id": "ASM-07", "decision before transformation": "Recompute line and order arithmetic in the published sequence and use reported source amounts only as validation evidence."},
        {"assumption_id": "ASM-08", "decision before transformation": "Flatten repeated items only into their target one-to-many table; do not flatten all entities into one wide table."},
        {"assumption_id": "ASM-09", "decision before transformation": "Treat the observed counts as profiling evidence only; derive every output and validation result from the current allocated files."},
    ]
)
show(assumptions, "Pre-transformation assumption register")

assert combined_repeat_profile["missing_key"].sum() == 0, "A candidate source key is missing"
duplicate_keys_are_exact = (
    combined_repeat_profile["duplicate_key_groups"]
    == combined_repeat_profile["exact_duplicate_groups"]
).all()
assert duplicate_keys_are_exact, "A repeated business key is not an exact source duplicate"
print(f"Repeated business-key groups are exact source duplicates: {duplicate_keys_are_exact}")
print("\nTask 1 source-profile checks: PASS")



Cross-source key overlap
         entity  JSON unique keys  XML unique keys  cross-source overlap  JSON only  XML only  union before field reconciliation
         orders              2750             2750                   500       2250      2250                               5000
    order_items              8666             8622                  1577       7089      7045                              15711
     deliveries              2750             2750                   500       2250      2250                               5000
product_reviews              3850             3850                   700       3150      3150                               7000

Within-source repeat evidence
source                                 source_collection                   grain candidate_key  rows  missing_key  unique_key  duplicate_key_groups  duplicate_extra_rows  exact_duplicate_groups  exact_duplicate_extra_rows
  JSON                                customerProfiles[]    one customer pro

## 2. Source-to-target mapping

The completed field-lineage artifact is
`Group050_source_to_target_mapping.csv` in the submission root. It preserves
the template's target rows and records applicable JSON/XML structural paths,
transformation or derivation, overlap/conflict handling and stable notebook
evidence for every target field. A source path may remain blank only when that
source is not applicable, as indicated by `source_format`.

The executable audit below checks template and dictionary alignment, applicable
path completeness, actual path existence in the parsed sources, evidence IDs,
placeholder absence, unique mapping IDs and UTF-8 encoding.


In [6]:
# EVIDENCE: SEC-2-MAPPING-AUDIT
import re

mapping_template = pd.read_csv(
    MAPPING_TEMPLATE_PATH, keep_default_na=False, dtype=str, encoding="utf-8-sig"
)
mapping = pd.read_csv(
    MAPPING_OUTPUT_PATH, keep_default_na=False, dtype=str, encoding="utf-8-sig"
)
data_dictionary = pd.read_csv(
    DATA_DICTIONARY_PATH, keep_default_na=False, dtype=str, encoding="utf-8-sig"
)

expected_columns = [
    "mapping_id",
    "output_table",
    "target_field",
    "source_format",
    "json_source_path",
    "xml_source_path",
    "transformation_or_derivation",
    "overlap_or_conflict_rule",
    "notebook_evidence",
]
prefilled_columns = ["mapping_id", "output_table", "target_field"]
required_text_columns = [
    "source_format",
    "transformation_or_derivation",
    "overlap_or_conflict_rule",
    "notebook_evidence",
]
completed_columns = [
    "source_format",
    "json_source_path",
    "xml_source_path",
    "transformation_or_derivation",
    "overlap_or_conflict_rule",
    "notebook_evidence",
]
allowed_source_formats = {"JSON", "XML", "both", "derived"}
defined_evidence_ids = {
    "SEC-1.1-JSON-PROFILE",
    "SEC-1.2-XML-PROFILE",
    "SEC-1.3-SOURCE-COMPARISON",
    "SEC-2-MAPPING-AUDIT",
}

audit_rows = []

def add_mapping_check(check_id, check, observed, passed):
    audit_rows.append(
        {
            "check_id": check_id,
            "check": check,
            "observed": observed,
            "status": "PASS" if passed else "FAIL",
        }
    )

add_mapping_check(
    "MAP-AUDIT-01",
    "Required mapping filename and file presence",
    MAPPING_OUTPUT_PATH.name,
    MAPPING_OUTPUT_PATH.is_file()
    and MAPPING_OUTPUT_PATH.name == f"{GROUP_ID}_source_to_target_mapping.csv",
)
add_mapping_check(
    "MAP-AUDIT-02",
    "Exact nine-column contract",
    list(mapping.columns),
    list(mapping.columns) == expected_columns,
)
template_alignment = (
    len(mapping) == len(mapping_template)
    and mapping[prefilled_columns].equals(mapping_template[prefilled_columns])
)
add_mapping_check(
    "MAP-AUDIT-03",
    "All pre-filled template rows preserved in order",
    f"mapping={len(mapping)}, template={len(mapping_template)}",
    template_alignment,
)
dictionary_targets = data_dictionary[["output_table", "field_name"]].rename(
    columns={"field_name": "target_field"}
)
dictionary_alignment = (
    len(mapping) == len(data_dictionary)
    and mapping[["output_table", "target_field"]].equals(dictionary_targets)
)
add_mapping_check(
    "MAP-AUDIT-04",
    "Mapping targets match the public data dictionary",
    f"mapping={len(mapping)}, dictionary={len(data_dictionary)}",
    dictionary_alignment,
)
mapping_id_ok = (
    mapping["mapping_id"].is_unique
    and mapping["mapping_id"].str.fullmatch(r"MAP-[a-z_]+-\d{2}").all()
)
add_mapping_check(
    "MAP-AUDIT-05",
    "Stable and unique MAP IDs",
    f"unique={mapping['mapping_id'].nunique()}",
    mapping_id_ok,
)
source_format_ok = set(mapping["source_format"]) <= allowed_source_formats
add_mapping_check(
    "MAP-AUDIT-06",
    "Allowed source_format values only",
    sorted(mapping["source_format"].unique()),
    source_format_ok,
)
blank_required_text = [
    (row.mapping_id, column)
    for row in mapping.itertuples(index=False)
    for column in required_text_columns
    if not str(getattr(row, column)).strip()
]
add_mapping_check(
    "MAP-AUDIT-07",
    "All non-path mapping fields completed",
    f"blank cells={len(blank_required_text)}",
    not blank_required_text,
)
applicable_path_missing = []
for row in mapping.itertuples(index=False):
    json_path_present = bool(row.json_source_path.strip())
    xml_path_present = bool(row.xml_source_path.strip())
    if row.source_format in {"JSON", "both"} and not json_path_present:
        applicable_path_missing.append((row.mapping_id, "json_source_path"))
    if row.source_format in {"XML", "both"} and not xml_path_present:
        applicable_path_missing.append((row.mapping_id, "xml_source_path"))
    if row.source_format == "derived" and not (json_path_present or xml_path_present):
        applicable_path_missing.append((row.mapping_id, "derived source path"))
add_mapping_check(
    "MAP-AUDIT-08",
    "Every applicable JSON/XML path is present",
    f"missing applicable paths={len(applicable_path_missing)}",
    not applicable_path_missing,
)

json_actual_paths = set()
for prefix, records in json_collections.items():
    for record in records:
        json_actual_paths.update(f"{prefix}.{field}" for field in record)

xml_actual_paths = set()
def collect_xml_paths(element, parent_path):
    for child in element:
        child_path = f"{parent_path}/{child.tag}"
        xml_actual_paths.add(child_path)
        collect_xml_paths(child, child_path)

collect_xml_paths(xml_root, f"/{xml_root.tag}")

def split_mapping_paths(value):
    return [part.strip() for part in value.split("|") if part.strip()]

invalid_path_references = []
for row in mapping.itertuples(index=False):
    for path in split_mapping_paths(row.json_source_path):
        if path not in json_actual_paths:
            invalid_path_references.append((row.mapping_id, "JSON", path))
    for path in split_mapping_paths(row.xml_source_path):
        if path not in xml_actual_paths:
            invalid_path_references.append((row.mapping_id, "XML", path))
add_mapping_check(
    "MAP-AUDIT-09",
    "Every listed structural path exists in the parsed sources",
    f"invalid path references={len(invalid_path_references)}",
    not invalid_path_references,
)
evidence_tokens = {
    token.strip()
    for value in mapping["notebook_evidence"]
    for token in value.split("|")
    if token.strip()
}
unknown_evidence_ids = sorted(evidence_tokens - defined_evidence_ids)
all_rows_cite_mapping_audit = mapping["notebook_evidence"].str.contains(
    "SEC-2-MAPPING-AUDIT", regex=False
).all()
add_mapping_check(
    "MAP-AUDIT-10",
    "Evidence IDs are defined and every row cites this audit",
    f"unknown={unknown_evidence_ids}, all cite audit={all_rows_cite_mapping_audit}",
    not unknown_evidence_ids and all_rows_cite_mapping_audit,
)
placeholder_pattern = re.compile(r"\b(?:replace|todo|tbd|example)\b", re.IGNORECASE)
placeholder_rows = mapping[completed_columns].apply(
    lambda column: column.map(lambda value: bool(placeholder_pattern.search(value)))
).any(axis=1)
add_mapping_check(
    "MAP-AUDIT-11",
    "No template placeholders remain in completed fields",
    f"placeholder rows={int(placeholder_rows.sum())}",
    not placeholder_rows.any(),
)
utf8_bom_present = MAPPING_OUTPUT_PATH.read_bytes().startswith(b"\xef\xbb\xbf")
add_mapping_check(
    "MAP-AUDIT-12",
    "CSV is UTF-8 with BOM for portable spreadsheet display",
    f"UTF-8 BOM={utf8_bom_present}",
    utf8_bom_present,
)

mapping_audit = pd.DataFrame(audit_rows)
show(mapping_audit, "Task 1 mapping audit register")
mapping_coverage = (
    mapping.groupby(["output_table", "source_format"], sort=False)
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
show(mapping_coverage, "Mapping coverage by target table and source format")
failed_mapping_checks = mapping_audit[mapping_audit["status"] != "PASS"]
assert failed_mapping_checks.empty, failed_mapping_checks.to_string(index=False)
print(f"\nFINAL_TASK1_MAPPING_AUDIT_PASS True; rows={len(mapping)}")



Task 1 mapping audit register
    check_id                                                     check                                                                                                                                                              observed status
MAP-AUDIT-01               Required mapping filename and file presence                                                                                                                                 Group050_source_to_target_mapping.csv   PASS
MAP-AUDIT-02                                Exact nine-column contract [mapping_id, output_table, target_field, source_format, json_source_path, xml_source_path, transformation_or_derivation, overlap_or_conflict_rule, notebook_evidence]   PASS
MAP-AUDIT-03           All pre-filled template rows preserved in order                                                                                                                                             mapping=111, template=111 

## 3. Text and regex functions

The six pure text functions are imported from `Group050_text_functions.py` and
tested against both the supplied public cases and student-designed boundary
cases before any dependent table is built.


### 3.1 Cleaning and extraction implementation


In [7]:
# EVIDENCE: SEC-3.1-TEXT-FUNCTIONS
from Group050_text_functions import (
    build_latin_analysis,
    clean_narrative_text,
    contains_non_latin_script,
    extract_order_reference,
    extract_product_sku,
    extract_promo_code,
)

TASK3_FUNCTIONS = {
    "clean_narrative_text": clean_narrative_text,
    "extract_order_reference": extract_order_reference,
    "extract_product_sku": extract_product_sku,
    "extract_promo_code": extract_promo_code,
    "build_latin_analysis": build_latin_analysis,
    "contains_non_latin_script": contains_non_latin_script,
}

print("TASK3_FUNCTION_INTERFACE_LOADED 6/6")


TASK3_FUNCTION_INTERFACE_LOADED 6/6


### 3.2 Public and student-designed tests


In [8]:
# EVIDENCE: SEC-3.2-TEXT-FUNCTION-TESTS
import inspect

public_text_cases = pd.read_csv(
    PUBLIC_TEXT_TESTS_PATH,
    keep_default_na=False,
    dtype=str,
    encoding="utf-8-sig",
)


def _task3_expected_value(function_name, expected_text):
    if function_name == "contains_non_latin_script":
        return expected_text.strip().lower() == "true"
    return expected_text


public_test_rows = []
for case in public_text_cases.to_dict("records"):
    function_name = case["function"]
    function = TASK3_FUNCTIONS[function_name]
    expected = _task3_expected_value(
        function_name,
        case["expected_output"],
    )
    observed = function(case["input_value"])
    public_test_rows.append(
        {
            "case_id": case["case_id"],
            "function": function_name,
            "expected": expected,
            "observed": observed,
            "passed": observed == expected,
        }
    )

task3_public_results = pd.DataFrame(public_test_rows)

student_cases = [
    ("STU-CLEAN-01", "clean_narrative_text", None, "NaN", "missing input"),
    ("STU-CLEAN-02", "clean_narrative_text", "NaN", "NaN", "literal sentinel"),
    (
        "STU-CLEAN-03",
        "clean_narrative_text",
        "[SYSTEM] https://example.com 😊",
        "NaN",
        "noise-only input",
    ),
    (
        "STU-CLEAN-04",
        "clean_narrative_text",
        "[NOTE] Café 包装很好",
        "[note] café 包装很好",
        "preserve non-removable and multilingual text",
    ),
    (
        "STU-ORDER-01",
        "extract_order_reference",
        "reference: cord000042.",
        "CORD000042",
        "case and leading zeroes",
    ),
    (
        "STU-ORDER-02",
        "extract_order_reference",
        "XHORD123456",
        "NaN",
        "embedded prefix near-match",
    ),
    (
        "STU-ORDER-03",
        "extract_order_reference",
        "HORD1234567",
        "NaN",
        "wrong digit count",
    ),
    (
        "STU-ORDER-04",
        "extract_order_reference",
        "HORD123456-extra",
        "NaN",
        "malformed extension",
    ),
    (
        "STU-ORDER-05",
        "extract_order_reference",
        "包装HORD123456很好",
        "NaN",
        "multilingual embedding",
    ),
    (
        "STU-SKU-01",
        "extract_product_sku",
        "(sku-a1B2).",
        "SKU-A1B2",
        "case normalisation",
    ),
    (
        "STU-SKU-02",
        "extract_product_sku",
        "SKU-ABC123-extra",
        "NaN",
        "malformed extension",
    ),
    (
        "STU-SKU-03",
        "extract_product_sku",
        "SKU-ABCé",
        "NaN",
        "ASCII token boundary",
    ),
    (
        "STU-PROMO-01",
        "extract_promo_code",
        "promo=b1save-00.",
        "B1SAVE-00",
        "valid lower boundary",
    ),
    (
        "STU-PROMO-02",
        "extract_promo_code",
        "B6SAVE-24",
        "NaN",
        "invalid promotion range",
    ),
    (
        "STU-PROMO-03",
        "extract_promo_code",
        "B3SAVE-244",
        "NaN",
        "wrong digit count",
    ),
    (
        "STU-LATIN-01",
        "build_latin_analysis",
        "service était bon 包装很好",
        "service était bon",
        "mixed script",
    ),
    (
        "STU-LATIN-02",
        "build_latin_analysis",
        "包装很好",
        "NaN",
        "no Latin letter",
    ),
    (
        "STU-LATIN-03",
        "build_latin_analysis",
        "cafe\u0301 κόσμος 42!",
        "café 42!",
        "NFC and punctuation",
    ),
    (
        "STU-SCRIPT-01",
        "contains_non_latin_script",
        "café déjà vu",
        False,
        "Latin diacritics are Latin",
    ),
    (
        "STU-SCRIPT-02",
        "contains_non_latin_script",
        "café 包装",
        True,
        "detect non-Latin letters",
    ),
    (
        "STU-SCRIPT-03",
        "contains_non_latin_script",
        "NaN",
        False,
        "literal sentinel",
    ),
]

student_test_rows = []
for case_id, function_name, value, expected, purpose in student_cases:
    observed = TASK3_FUNCTIONS[function_name](value)
    student_test_rows.append(
        {
            "case_id": case_id,
            "function": function_name,
            "expected": expected,
            "observed": observed,
            "passed": observed == expected,
            "purpose": purpose,
        }
    )

task3_student_results = pd.DataFrame(student_test_rows)
task3_interface_results = pd.DataFrame(
    [
        {
            "function": name,
            "callable": callable(function),
            "argument_count": len(inspect.signature(function).parameters),
            "passed": callable(function)
            and len(inspect.signature(function).parameters) == 1,
        }
        for name, function in TASK3_FUNCTIONS.items()
    ]
)

assert task3_public_results["passed"].all(), "A public Task 3 case failed"
assert task3_student_results["passed"].all(), "A student Task 3 case failed"
assert task3_interface_results["passed"].all(), "The fixed Task 3 interface failed"

show(task3_public_results, "Task 3 public text cases")
show(task3_student_results, "Task 3 student-designed text cases")
show(task3_interface_results, "Task 3 fixed interface")
print(
    "TASK3_COMPLETE True; "
    f"public={int(task3_public_results['passed'].sum())}/"
    f"{len(task3_public_results)}; "
    f"student={int(task3_student_results['passed'].sum())}/"
    f"{len(task3_student_results)}; interface=6/6"
)



Task 3 public text cases
case_id                  function                               expected                               observed  passed
 TXT-01      clean_narrative_text                     leave at reception                     leave at reception    True
 TXT-02        extract_promo_code                              B3SAVE-24                              B3SAVE-24    True
 TXT-03      clean_narrative_text                    café setup was easy                    café setup was easy    True
 TXT-04   extract_order_reference                             HORD123456                             HORD123456    True
 TXT-05       extract_product_sku                             SKU-ABC123                             SKU-ABC123    True
 TXT-06   extract_order_reference                                    NaN                                    NaN    True
 TXT-07      build_latin_analysis                      service était bon                      service était bon    True
 TXT-08 contai

## 4. Build the six standardised relational tables

Show the transformation and row-flow evidence for each table. Keep helper
columns inside the workflow; export only fields in the public data dictionary.


### 4.1 `orders`


In [9]:
# EVIDENCE: SEC-4.1-ORDERS-INTEGRATED
# Integrated from the supplied orders/order-items member notebook.
try:
    display
except NameError:
    display = lambda value: print(value)

# SHARED HELPER: Convert AUD currency text to numbers

def parse_aud_amount(values):
    cleaned_values = (
        values.astype("string")
        .str.replace(
            "AUD",
            "",
            regex=False,
        )
        .str.replace(
            ",",
            "",
            regex=False,
        )
        .str.strip()
    )

    return pd.to_numeric(
        cleaned_values,
        errors="raise",
    ).round(2)

# EVIDENCE: SEC-4.1-JSON-ORDERS-EXTRACTION

json_orders_records = []

for order in json_orders:
    order_header = order["header"]
    json_orders_records.append(order_header)

json_orders_raw = pd.DataFrame(
    json_orders_records
)

print(
    "JSON orders extracted:",
    len(json_orders_raw),
    "rows and",
    len(json_orders_raw.columns),
    "columns",
)

print("\nColumn names:")
print(json_orders_raw.columns.tolist())

display(
    json_orders_raw[
        [
            "orderID",
            "customerID",
            "orderTimestamp",
            "orderPrice",
            "orderTotal",
        ]
    ].head()
)

# EVIDENCE: SEC-4.1-XML-ORDERS-EXTRACTION

xml_orders_raw = pd.DataFrame(
    xml_header_records
)

print(
    "XML orders extracted:",
    len(xml_orders_raw),
    "rows and",
    len(xml_orders_raw.columns),
    "columns",
)

print("\nColumn names:")
print(xml_orders_raw.columns.tolist())

display(
    xml_orders_raw[
        [
            "Order_ID",
            "Customer_ID",
            "Order_Timestamp",
            "Order_Price",
            "Order_Total",
        ]
    ].head()
)

# STEP 4A: Standardise JSON order column names

json_orders_named = (
    json_orders_raw.rename(
        columns={
            "orderID": "order_id",
            "sourceSystemRecordID": (
                "source_system_record_id"
            ),
            "customerID": "customer_id",
            "orderTimestamp": "order_timestamp",
            "salesChannel": "sales_channel",
            "paymentMethod": "payment_method",
            "currency": "currency",
            "nearestWarehouse": (
                "nearest_warehouse"
            ),
            "orderStatus": "order_status",
            "orderPrice": (
                "reported_order_price"
            ),
            "deliveryCharges": (
                "delivery_charges"
            ),
            "couponCode": "coupon_code",
            "couponDiscount": (
                "coupon_discount"
            ),
            "taxAmount": (
                "reported_tax_amount"
            ),
            "orderTotal": (
                "reported_order_total"
            ),
            "season": "season",
            "expeditedDelivery": (
                "expedited_delivery"
            ),
            "customerLat": "customer_lat",
            "customerLong": "customer_long",
            "deviceType": "device_type",
            "referralSource": (
                "referral_source"
            ),
            "customerNote": (
                "raw_customer_note"
            ),
        }
    ).copy()
)

json_orders_named["source_format"] = "JSON"

json_orders_named = json_orders_named[
    [
        "order_id",
        "source_system_record_id",
        "customer_id",
        "order_timestamp",
        "sales_channel",
        "payment_method",
        "currency",
        "nearest_warehouse",
        "order_status",
        "reported_order_price",
        "delivery_charges",
        "coupon_code",
        "coupon_discount",
        "reported_tax_amount",
        "reported_order_total",
        "season",
        "expedited_delivery",
        "customer_lat",
        "customer_long",
        "device_type",
        "referral_source",
        "raw_customer_note",
        "source_format",
    ]
]

print(
    "JSON order columns renamed:",
    len(json_orders_named),
    "rows and",
    len(json_orders_named.columns),
    "columns",
)

print("\nNew column names:")
print(json_orders_named.columns.tolist())

display(
    json_orders_named[
        [
            "order_id",
            "customer_id",
            "order_timestamp",
            "reported_order_price",
            "reported_order_total",
            "source_format",
        ]
    ].head()
)

# STEP 4B: Standardise XML order column names

xml_orders_named = (
    xml_orders_raw.rename(
        columns={
            "Order_ID": "order_id",
            "Source_System_Record_ID": (
                "source_system_record_id"
            ),
            "Customer_ID": "customer_id",
            "Order_Timestamp": (
                "order_timestamp"
            ),
            "Sales_Channel": "sales_channel",
            "Payment_Method": (
                "payment_method"
            ),
            "Currency": "currency",
            "Nearest_Warehouse": (
                "nearest_warehouse"
            ),
            "Order_Status": "order_status",
            "Order_Price": (
                "reported_order_price"
            ),
            "Delivery_Charges": (
                "delivery_charges"
            ),
            "Coupon_Code": "coupon_code",
            "Coupon_Discount": (
                "coupon_discount"
            ),
            "Tax_Amount": (
                "reported_tax_amount"
            ),
            "Order_Total": (
                "reported_order_total"
            ),
            "Season": "season",
            "Expedited_Delivery": (
                "expedited_delivery"
            ),
            "Customer_Lat": "customer_lat",
            "Customer_Long": "customer_long",
            "Device_Type": "device_type",
            "Referral_Source": (
                "referral_source"
            ),
            "Customer_Note": (
                "raw_customer_note"
            ),
        }
    ).copy()
)

xml_orders_named["source_format"] = "XML"

xml_orders_named = xml_orders_named[
    [
        "order_id",
        "source_system_record_id",
        "customer_id",
        "order_timestamp",
        "sales_channel",
        "payment_method",
        "currency",
        "nearest_warehouse",
        "order_status",
        "reported_order_price",
        "delivery_charges",
        "coupon_code",
        "coupon_discount",
        "reported_tax_amount",
        "reported_order_total",
        "season",
        "expedited_delivery",
        "customer_lat",
        "customer_long",
        "device_type",
        "referral_source",
        "raw_customer_note",
        "source_format",
    ]
]

print(
    "XML order columns renamed:",
    len(xml_orders_named),
    "rows and",
    len(xml_orders_named.columns),
    "columns",
)

print("\nNew column names:")
print(xml_orders_named.columns.tolist())

display(
    xml_orders_named[
        [
            "order_id",
            "customer_id",
            "order_timestamp",
            "reported_order_price",
            "reported_order_total",
            "source_format",
        ]
    ].head()
)

# STEP 5A: Standardise JSON order text and timestamp fields

import unicodedata


def normalise_text_values(values):
    text_values = values.astype("string")

    normalised_values = text_values.map(
        lambda value: (
            unicodedata.normalize(
                "NFC",
                value.strip(),
            )
            if pd.notna(value)
            else value
        )
    )

    return normalised_values.astype("string")


json_orders_text_standardised = (
    json_orders_named.copy()
)

identifier_columns = [
    "order_id",
    "source_system_record_id",
    "customer_id",
]

category_columns = [
    "sales_channel",
    "payment_method",
    "currency",
    "nearest_warehouse",
    "order_status",
    "season",
    "device_type",
    "referral_source",
]

for column in (
    identifier_columns
    + category_columns
):
    json_orders_text_standardised[column] = (
        normalise_text_values(
            json_orders_text_standardised[column]
        )
    )

json_orders_text_standardised[
    "order_timestamp"
] = (
    pd.to_datetime(
        json_orders_text_standardised[
            "order_timestamp"
        ],
        format="%Y-%m-%d %H:%M:%S",
        errors="raise",
    )
    .dt.strftime("%Y-%m-%d %H:%M:%S")
    .astype("string")
)

json_orders_text_standardised[
    "coupon_code"
] = normalise_text_values(
    json_orders_text_standardised[
        "coupon_code"
    ]
)

json_orders_text_standardised[
    "coupon_code"
] = (
    json_orders_text_standardised[
        "coupon_code"
    ]
    .fillna("")
    .mask(
        json_orders_text_standardised[
            "coupon_code"
        ].fillna("").eq(""),
        "NaN",
    )
)

json_orders_text_standardised[
    "raw_customer_note"
] = (
    json_orders_text_standardised[
        "raw_customer_note"
    ].astype("string")
)

json_orders_text_standardised[
    "source_format"
] = (
    json_orders_text_standardised[
        "source_format"
    ].astype("string")
)

required_text_columns = (
    identifier_columns
    + category_columns
    + ["order_timestamp"]
)

missing_required_text_values = (
    json_orders_text_standardised[
        required_text_columns
    ].isna().sum().sum()
)

empty_required_text_values = (
    json_orders_text_standardised[
        required_text_columns
    ]
    .eq("")
    .sum()
    .sum()
)

print(
    "JSON orders after text standardisation:",
    len(json_orders_text_standardised),
    "rows and",
    len(
        json_orders_text_standardised.columns
    ),
    "columns",
)

print(
    "Missing required text values:",
    missing_required_text_values,
)

print(
    "Empty required text values:",
    empty_required_text_values,
)

print(
    "Literal NaN coupon codes:",
    json_orders_text_standardised[
        "coupon_code"
    ].eq("NaN").sum(),
)

display(
    json_orders_text_standardised[
        [
            "order_id",
            "customer_id",
            "order_timestamp",
            "sales_channel",
            "coupon_code",
        ]
    ].head()
)

# STEP 5B: Standardise JSON order numeric and boolean fields

json_orders_clean = (
    json_orders_text_standardised.copy()
)

order_money_columns = [
    "reported_order_price",
    "delivery_charges",
    "reported_tax_amount",
    "reported_order_total",
]

for column in order_money_columns:
    json_orders_clean[column] = (
        pd.to_numeric(
            json_orders_clean[column],
            errors="raise",
        ).round(2)
    )

json_orders_clean[
    "coupon_discount"
] = (
    pd.to_numeric(
        json_orders_clean[
            "coupon_discount"
        ],
        errors="raise",
    ).astype("float64")
)

coordinate_columns = [
    "customer_lat",
    "customer_long",
]

for column in coordinate_columns:
    json_orders_clean[column] = (
        pd.to_numeric(
            json_orders_clean[column],
            errors="raise",
        )
    )

valid_json_booleans = (
    json_orders_clean[
        "expedited_delivery"
    ].isin([True, False]).all()
)

if not valid_json_booleans:
    raise ValueError(
        "Unexpected JSON expedited-delivery value"
    )

json_orders_clean[
    "expedited_delivery"
] = (
    json_orders_clean[
        "expedited_delivery"
    ].astype("bool")
)

required_numeric_columns = (
    order_money_columns
    + [
        "coupon_discount",
        "customer_lat",
        "customer_long",
    ]
)

missing_required_numeric_values = (
    json_orders_clean[
        required_numeric_columns
    ].isna().sum().sum()
)

print("JSON order numeric data types:")
print(
    json_orders_clean[
        required_numeric_columns
    ].dtypes
)

print(
    "\nExpedited-delivery data type:",
    json_orders_clean[
        "expedited_delivery"
    ].dtype,
)

print(
    "Valid JSON boolean values:",
    valid_json_booleans,
)

print(
    "Missing required numeric values:",
    missing_required_numeric_values,
)

print("\nExpedited-delivery values:")
print(
    json_orders_clean[
        "expedited_delivery"
    ].value_counts()
)

display(
    json_orders_clean[
        [
            "order_id",
            "reported_order_price",
            "delivery_charges",
            "coupon_discount",
            "reported_tax_amount",
            "reported_order_total",
            "expedited_delivery",
        ]
    ].head()
)

# STEP 5C: Standardise XML order text and timestamp fields

xml_orders_text_standardised = (
    xml_orders_named.copy()
)

for column in (
    identifier_columns
    + category_columns
):
    xml_orders_text_standardised[column] = (
        normalise_text_values(
            xml_orders_text_standardised[column]
        )
    )

xml_orders_text_standardised[
    "order_timestamp"
] = (
    pd.to_datetime(
        xml_orders_text_standardised[
            "order_timestamp"
        ],
        format="%d/%m/%Y %H:%M:%S",
        errors="raise",
    )
    .dt.strftime("%Y-%m-%d %H:%M:%S")
    .astype("string")
)

xml_orders_text_standardised[
    "coupon_code"
] = normalise_text_values(
    xml_orders_text_standardised[
        "coupon_code"
    ]
)

xml_orders_text_standardised[
    "coupon_code"
] = (
    xml_orders_text_standardised[
        "coupon_code"
    ]
    .fillna("")
    .mask(
        xml_orders_text_standardised[
            "coupon_code"
        ].fillna("").eq(""),
        "NaN",
    )
)

xml_orders_text_standardised[
    "raw_customer_note"
] = (
    xml_orders_text_standardised[
        "raw_customer_note"
    ].astype("string")
)

xml_orders_text_standardised[
    "source_format"
] = (
    xml_orders_text_standardised[
        "source_format"
    ].astype("string")
)

xml_missing_required_text_values = (
    xml_orders_text_standardised[
        required_text_columns
    ].isna().sum().sum()
)

xml_empty_required_text_values = (
    xml_orders_text_standardised[
        required_text_columns
    ]
    .eq("")
    .sum()
    .sum()
)

print(
    "XML orders after text standardisation:",
    len(xml_orders_text_standardised),
    "rows and",
    len(
        xml_orders_text_standardised.columns
    ),
    "columns",
)

print(
    "Missing required text values:",
    xml_missing_required_text_values,
)

print(
    "Empty required text values:",
    xml_empty_required_text_values,
)

print(
    "Literal NaN coupon codes:",
    xml_orders_text_standardised[
        "coupon_code"
    ].eq("NaN").sum(),
)

display(
    xml_orders_text_standardised[
        [
            "order_id",
            "customer_id",
            "order_timestamp",
            "sales_channel",
            "coupon_code",
        ]
    ].head()
)

# STEP 5D: Standardise XML order numeric and boolean fields

xml_orders_clean = (
    xml_orders_text_standardised.copy()
)

for column in order_money_columns:
    xml_orders_clean[column] = (
        parse_aud_amount(
            xml_orders_clean[column]
        ).astype("float64")
    )

xml_orders_clean[
    "coupon_discount"
] = (
    pd.to_numeric(
        xml_orders_clean[
            "coupon_discount"
        ]
        .astype("string")
        .str.replace(
            "%",
            "",
            regex=False,
        )
        .str.strip(),
        errors="raise",
    ).astype("float64")
)

for column in coordinate_columns:
    xml_orders_clean[column] = (
        pd.to_numeric(
            xml_orders_clean[column],
            errors="raise",
        )
    )

xml_expedited_text = (
    xml_orders_clean[
        "expedited_delivery"
    ]
    .astype("string")
    .str.strip()
    .str.upper()
)

unexpected_xml_boolean_values = sorted(
    set(
        xml_expedited_text
        .dropna()
        .tolist()
    )
    - {"Y", "N"}
)

valid_xml_booleans = (
    xml_expedited_text
    .isin(["Y", "N"])
    .all()
)

if not valid_xml_booleans:
    raise ValueError(
        "Unexpected XML expedited-delivery "
        f"values: {unexpected_xml_boolean_values}"
    )

xml_orders_clean[
    "expedited_delivery"
] = (
    xml_expedited_text
    .map(
        {
            "Y": True,
            "N": False,
        }
    )
    .astype("bool")
)

xml_missing_required_numeric_values = (
    xml_orders_clean[
        required_numeric_columns
    ]
    .isna()
    .sum()
    .sum()
)

print("XML order numeric data types:")
print(
    xml_orders_clean[
        required_numeric_columns
    ].dtypes
)

print(
    "\nExpedited-delivery data type:",
    xml_orders_clean[
        "expedited_delivery"
    ].dtype,
)

print(
    "Valid XML boolean values:",
    valid_xml_booleans,
)

print(
    "Unexpected XML boolean values:",
    unexpected_xml_boolean_values,
)

print(
    "Missing required numeric values:",
    xml_missing_required_numeric_values,
)

print("\nExpedited-delivery values:")
print(
    xml_orders_clean[
        "expedited_delivery"
    ].value_counts()
)

display(
    xml_orders_clean[
        [
            "order_id",
            "reported_order_price",
            "delivery_charges",
            "coupon_discount",
            "reported_tax_amount",
            "reported_order_total",
            "expedited_delivery",
        ]
    ].head()
)

# STEP 6A: Remove repeated JSON orders safely
# EVIDENCE: SEC-4.1-JSON-WITHIN-SOURCE-DEDUP

order_comparison_columns = [
    column
    for column in json_orders_clean.columns
    if column not in [
        "order_id",
        "source_format",
    ]
]

json_duplicate_order_rows = (
    json_orders_clean[
        json_orders_clean.duplicated(
            subset=["order_id"],
            keep=False,
        )
    ].copy()
)

json_duplicate_order_comparison = (
    json_duplicate_order_rows
    .groupby("order_id")[
        order_comparison_columns
    ]
    .nunique(dropna=False)
)

json_conflicting_order_ids = (
    json_duplicate_order_comparison[
        json_duplicate_order_comparison
        .gt(1)
        .any(axis=1)
    ]
    .index
    .tolist()
)

if json_conflicting_order_ids:
    raise ValueError(
        "Conflicting JSON order duplicates: "
        f"{json_conflicting_order_ids[:10]}"
    )

json_orders_deduplicated = (
    json_orders_clean
    .drop_duplicates(
        subset=["order_id"],
        keep="first",
    )
    .copy()
)

json_order_rows_removed = (
    len(json_orders_clean)
    - len(json_orders_deduplicated)
)

print(
    "JSON repeated order_id values:",
    json_duplicate_order_rows[
        "order_id"
    ].nunique(),
)

print(
    "JSON conflicting duplicate IDs:",
    len(json_conflicting_order_ids),
)

print(
    "JSON rows before deduplication:",
    len(json_orders_clean),
)

print(
    "JSON rows removed:",
    json_order_rows_removed,
)

print(
    "JSON rows after deduplication:",
    len(json_orders_deduplicated),
)

print(
    "JSON order_id is now unique:",
    json_orders_deduplicated[
        "order_id"
    ].is_unique,
)

# STEP 6B: Remove repeated XML orders safely
# EVIDENCE: SEC-4.1-XML-WITHIN-SOURCE-DEDUP

xml_duplicate_order_rows = (
    xml_orders_clean[
        xml_orders_clean.duplicated(
            subset=["order_id"],
            keep=False,
        )
    ].copy()
)

xml_duplicate_order_comparison = (
    xml_duplicate_order_rows
    .groupby("order_id")[
        order_comparison_columns
    ]
    .nunique(dropna=False)
)

xml_conflicting_order_ids = (
    xml_duplicate_order_comparison[
        xml_duplicate_order_comparison
        .gt(1)
        .any(axis=1)
    ]
    .index
    .tolist()
)

if xml_conflicting_order_ids:
    raise ValueError(
        "Conflicting XML order duplicates: "
        f"{xml_conflicting_order_ids[:10]}"
    )

xml_orders_deduplicated = (
    xml_orders_clean
    .drop_duplicates(
        subset=["order_id"],
        keep="first",
    )
    .copy()
)

xml_order_rows_removed = (
    len(xml_orders_clean)
    - len(xml_orders_deduplicated)
)

print(
    "XML repeated order_id values:",
    xml_duplicate_order_rows[
        "order_id"
    ].nunique(),
)

print(
    "XML conflicting duplicate IDs:",
    len(xml_conflicting_order_ids),
)

print(
    "XML rows before deduplication:",
    len(xml_orders_clean),
)

print(
    "XML rows removed:",
    xml_order_rows_removed,
)

print(
    "XML rows after deduplication:",
    len(xml_orders_deduplicated),
)

print(
    "XML order_id is now unique:",
    xml_orders_deduplicated[
        "order_id"
    ].is_unique,
)

# STEP 7A: Locate orders shared by JSON and XML

json_order_id_set = set(
    json_orders_deduplicated[
        "order_id"
    ]
)

xml_order_id_set = set(
    xml_orders_deduplicated[
        "order_id"
    ]
)

overlapping_order_ids = sorted(
    json_order_id_set
    & xml_order_id_set
)

json_only_order_ids = sorted(
    json_order_id_set
    - xml_order_id_set
)

xml_only_order_ids = sorted(
    xml_order_id_set
    - json_order_id_set
)

all_order_ids = sorted(
    json_order_id_set
    | xml_order_id_set
)

orders_overlap = (
    json_orders_deduplicated
    .merge(
        xml_orders_deduplicated,
        on="order_id",
        how="inner",
        suffixes=(
            "_json",
            "_xml",
        ),
        validate="one_to_one",
    )
    .sort_values("order_id")
    .reset_index(drop=True)
)

print(
    "Unique JSON orders:",
    len(json_order_id_set),
)

print(
    "Unique XML orders:",
    len(xml_order_id_set),
)

print(
    "Orders present in both sources:",
    len(overlapping_order_ids),
)

print(
    "JSON-only orders:",
    len(json_only_order_ids),
)

print(
    "XML-only orders:",
    len(xml_only_order_ids),
)

print(
    "Total distinct order IDs:",
    len(all_order_ids),
)

print(
    "Rows in overlap comparison table:",
    len(orders_overlap),
)

display(
    orders_overlap[
        [
            "order_id",
            "customer_id_json",
            "customer_id_xml",
            "reported_order_price_json",
            "reported_order_price_xml",
        ]
    ].head()
)

# STEP 7B: Compare exact fields for shared orders
# EVIDENCE: SEC-4.1-CROSS-SOURCE-EXACT-COMPARISON

order_exact_comparison_columns = [
    "source_system_record_id",
    "customer_id",
    "order_timestamp",
    "sales_channel",
    "payment_method",
    "currency",
    "nearest_warehouse",
    "order_status",
    "coupon_code",
    "season",
    "expedited_delivery",
    "device_type",
    "referral_source",
    "raw_customer_note",
]

exact_order_comparison_results = []
exact_order_mismatches = {}

for column in order_exact_comparison_columns:
    json_values = orders_overlap[
        f"{column}_json"
    ]

    xml_values = orders_overlap[
        f"{column}_xml"
    ]

    values_match = (
        json_values.eq(xml_values)
        | (
            json_values.isna()
            & xml_values.isna()
        )
    ).fillna(False)

    mismatch_ids = (
        orders_overlap.loc[
            ~values_match,
            "order_id",
        ]
        .tolist()
    )

    exact_order_comparison_results.append(
        {
            "column": column,
            "mismatching_orders": (
                len(mismatch_ids)
            ),
        }
    )

    if mismatch_ids:
        exact_order_mismatches[
            column
        ] = mismatch_ids

exact_order_comparison_summary = (
    pd.DataFrame(
        exact_order_comparison_results
    )
)

orders_with_exact_conflicts = sorted(
    {
        order_id
        for mismatch_ids
        in exact_order_mismatches.values()
        for order_id in mismatch_ids
    }
)

display(exact_order_comparison_summary)

print(
    "Exact fields checked:",
    len(order_exact_comparison_columns),
)

print(
    "Exact fields with conflicts:",
    len(exact_order_mismatches),
)

print(
    "Orders with exact-field conflicts:",
    len(orders_with_exact_conflicts),
)

if exact_order_mismatches:
    raise ValueError(
        "JSON and XML shared orders have "
        "conflicting exact fields: "
        f"{exact_order_mismatches}"
    )

# STEP 7C: Compare numeric fields for shared orders
# EVIDENCE: SEC-4.1-CROSS-SOURCE-NUMERIC-COMPARISON

order_numeric_tolerances = {
    "reported_order_price": 0.01,
    "delivery_charges": 0.01,
    "coupon_discount": 0.000001,
    "reported_tax_amount": 0.01,
    "reported_order_total": 0.01,
    "customer_lat": 0.000001,
    "customer_long": 0.000001,
}

numeric_order_comparison_results = []
numeric_order_mismatches = {}

for (
    column,
    allowed_difference,
) in order_numeric_tolerances.items():
    json_values = pd.to_numeric(
        orders_overlap[
            f"{column}_json"
        ],
        errors="raise",
    )

    xml_values = pd.to_numeric(
        orders_overlap[
            f"{column}_xml"
        ],
        errors="raise",
    )

    absolute_difference = (
        json_values
        .sub(xml_values)
        .abs()
    )

    values_match = (
        absolute_difference.le(
            allowed_difference
            + 0.000000001
        )
        | (
            json_values.isna()
            & xml_values.isna()
        )
    ).fillna(False)

    mismatch_ids = (
        orders_overlap.loc[
            ~values_match,
            "order_id",
        ]
        .tolist()
    )

    numeric_order_comparison_results.append(
        {
            "column": column,
            "allowed_difference": (
                allowed_difference
            ),
            "maximum_difference": (
                absolute_difference.max()
            ),
            "mismatching_orders": (
                len(mismatch_ids)
            ),
        }
    )

    if mismatch_ids:
        numeric_order_mismatches[
            column
        ] = mismatch_ids

numeric_order_comparison_summary = (
    pd.DataFrame(
        numeric_order_comparison_results
    )
)

orders_with_numeric_conflicts = sorted(
    {
        order_id
        for mismatch_ids
        in numeric_order_mismatches.values()
        for order_id in mismatch_ids
    }
)

display(numeric_order_comparison_summary)

print(
    "Numeric fields checked:",
    len(order_numeric_tolerances),
)

print(
    "Numeric fields with conflicts:",
    len(numeric_order_mismatches),
)

print(
    "Orders with numeric-field conflicts:",
    len(orders_with_numeric_conflicts),
)

if numeric_order_mismatches:
    raise ValueError(
        "JSON and XML shared orders have "
        "conflicting numeric fields: "
        f"{numeric_order_mismatches}"
    )

# STEP 7D: Reconcile JSON and XML orders
# EVIDENCE: SEC-4.1-CROSS-SOURCE-RECONCILIATION

if (
    exact_order_mismatches
    or numeric_order_mismatches
):
    raise ValueError(
        "Orders cannot be reconciled while "
        "cross-source conflicts remain"
    )

json_orders_for_union = (
    json_orders_deduplicated.copy()
)

xml_orders_for_union = (
    xml_orders_deduplicated.copy()
)

json_orders_for_union.loc[
    json_orders_for_union[
        "order_id"
    ].isin(overlapping_order_ids),
    "source_format",
] = "both"

xml_orders_for_union.loc[
    xml_orders_for_union[
        "order_id"
    ].isin(overlapping_order_ids),
    "source_format",
] = "both"

orders_before_cross_source_collapse = (
    pd.concat(
        [
            json_orders_for_union,
            xml_orders_for_union,
        ],
        ignore_index=True,
    )
)

orders_reconciled = (
    orders_before_cross_source_collapse
    .sort_values(
        "order_id",
        kind="mergesort",
    )
    .drop_duplicates(
        subset=["order_id"],
        keep="first",
    )
    .reset_index(drop=True)
)

orders_reconciled[
    "source_format"
] = (
    orders_reconciled[
        "source_format"
    ].astype("string")
)

orders_cross_source_rows_removed = (
    len(orders_before_cross_source_collapse)
    - len(orders_reconciled)
)

canonical_order_id_set = set(
    orders_reconciled[
        "order_id"
    ]
)

canonical_orders_match_union = (
    canonical_order_id_set
    == set(all_order_ids)
)

if not orders_reconciled[
    "order_id"
].is_unique:
    raise ValueError(
        "Reconciled order_id values are not unique"
    )

if not canonical_orders_match_union:
    raise ValueError(
        "Reconciled orders do not match the "
        "JSON/XML order-ID union"
    )

print(
    "Rows before cross-source collapse:",
    len(orders_before_cross_source_collapse),
)

print(
    "Shared rows collapsed:",
    orders_cross_source_rows_removed,
)

print(
    "Reconciled order rows:",
    len(orders_reconciled),
)

print(
    "Reconciled order_id is unique:",
    orders_reconciled[
        "order_id"
    ].is_unique,
)

print(
    "Reconciled IDs match source union:",
    canonical_orders_match_union,
)

print("\nSource-format counts:")
print(
    orders_reconciled[
        "source_format"
    ]
    .value_counts()
    .sort_index()
)

display(
    orders_reconciled[
        [
            "order_id",
            "customer_id",
            "order_timestamp",
            "reported_order_price",
            "reported_order_total",
            "source_format",
        ]
    ].head()
)


JSON orders extracted: 2818 rows and 22 columns

Column names:
['couponCode', 'couponDiscount', 'currency', 'customerID', 'customerLat', 'customerLong', 'customerNote', 'deliveryCharges', 'deviceType', 'expeditedDelivery', 'nearestWarehouse', 'orderID', 'orderPrice', 'orderStatus', 'orderTimestamp', 'orderTotal', 'paymentMethod', 'referralSource', 'salesChannel', 'season', 'sourceSystemRecordID', 'taxAmount']


,orderID,customerID,orderTimestamp,orderPrice,orderTotal
0,HORD002902,CUS00249,2018-07-25 13:54:00,1716.89,1645.73
1,HORD002426,CUS00448,2018-10-28 21:40:00,291.07,274.33
2,HORD003877,CUS00132,2018-12-18 09:43:00,2393.43,2047.64
3,HORD001912,CUS00130,2018-12-11 12:22:00,3833.63,3846.75
4,HORD002339,CUS00045,2018-06-27 08:33:00,456.45,449.27


XML orders extracted: 2818 rows and 22 columns

Column names:
['Order_ID', 'Source_System_Record_ID', 'Customer_ID', 'Order_Timestamp', 'Sales_Channel', 'Payment_Method', 'Currency', 'Nearest_Warehouse', 'Order_Status', 'Order_Price', 'Delivery_Charges', 'Coupon_Code', 'Coupon_Discount', 'Tax_Amount', 'Order_Total', 'Season', 'Expedited_Delivery', 'Customer_Lat', 'Customer_Long', 'Device_Type', 'Referral_Source', 'Customer_Note']


,Order_ID,Customer_ID,Order_Timestamp,Order_Price,Order_Total
0,HORD001093,CUS00442,15/04/2018 11:46:00,"AUD 1,862.19","AUD 1,872.67"
1,HORD003432,CUS00449,22/12/2018 11:24:00,"AUD 2,261.50","AUD 2,047.52"
2,HORD000963,CUS00327,24/02/2018 13:02:00,"AUD 8,285.06","AUD 7,884.33"
3,HORD000540,CUS00081,17/08/2018 14:48:00,"AUD 7,515.92","AUD 7,165.56"
4,HORD000464,CUS00117,01/08/2018 17:51:00,"AUD 1,598.26","AUD 1,529.01"


JSON order columns renamed: 2818 rows and 23 columns

New column names:
['order_id', 'source_system_record_id', 'customer_id', 'order_timestamp', 'sales_channel', 'payment_method', 'currency', 'nearest_warehouse', 'order_status', 'reported_order_price', 'delivery_charges', 'coupon_code', 'coupon_discount', 'reported_tax_amount', 'reported_order_total', 'season', 'expedited_delivery', 'customer_lat', 'customer_long', 'device_type', 'referral_source', 'raw_customer_note', 'source_format']


...

,column,mismatching_orders
0,source_system_record_id,0
1,customer_id,0
2,order_timestamp,0
3,sales_channel,0
4,payment_method,0
5,currency,0
6,nearest_warehouse,0
7,order_status,0
8,coupon_code,0
9,season,0


Exact fields checked: 14
Exact fields with conflicts: 0
Orders with exact-field conflicts: 0


,column,allowed_difference,maximum_difference,mismatching_orders
0,reported_order_price,0.010000,0.0,0
1,delivery_charges,0.010000,0.0,0
2,coupon_discount,0.000001,0.0,0
3,reported_tax_amount,0.010000,0.0,0
4,reported_order_total,0.010000,0.0,0
5,customer_lat,0.000001,0.0,0
6,customer_long,0.000001,0.0,0


Numeric fields checked: 7
Numeric fields with conflicts: 0
Orders with numeric-field conflicts: 0
Rows before cross-source collapse: 5500
Shared rows collapsed: 500
Reconciled order rows: 5000
Reconciled order_id is unique: True
Reconciled IDs match source union: True

Source-format counts:
source_format
JSON    2250
XML     2250
both     500
Name: count, dtype: Int64


,order_id,customer_id,order_timestamp,reported_order_price,reported_order_total,source_format
0,HORD000001,CUS00191,2018-11-12 19:33:00,2047.92,1955.24,both
1,HORD000002,CUS00096,2018-04-19 19:08:00,6549.78,6234.97,JSON
2,HORD000003,CUS00197,2018-03-08 20:08:00,3047.54,2904.73,XML
3,HORD000004,CUS00365,2018-01-30 20:39:00,584.43,597.10,JSON
4,HORD000005,CUS00248,2018-03-11 09:48:00,2280.74,2293.20,XML


### 4.2 `order_items`


In [10]:
# EVIDENCE: SEC-4.2-ORDER-ITEMS-INTEGRATED
# Integrated from the supplied orders/order-items member notebook.

# EVIDENCE: SEC-4.2-JSON-ORDER-ITEMS-EXTRACTION

json_order_items_records = []

for order in json_orders:
    shopping_cart = order["shoppingCart"]

    for item in shopping_cart:
        json_order_items_records.append(item)

json_order_items_raw = pd.DataFrame(
    json_order_items_records
)

print(
    "JSON order items extracted:",
    len(json_order_items_raw),
    "rows and",
    len(json_order_items_raw.columns),
    "columns",
)

print("\nColumn names:")
print(json_order_items_raw.columns.tolist())

display(json_order_items_raw.head())

# EVIDENCE: SEC-4.2-XML-ORDER-ITEMS-EXTRACTION

xml_order_items_raw = pd.DataFrame(
    xml_item_records
)

print(
    "XML order items extracted:",
    len(xml_order_items_raw),
    "rows and",
    len(xml_order_items_raw.columns),
    "columns",
)

print("\nColumn names:")
print(xml_order_items_raw.columns.tolist())

display(xml_order_items_raw.head())

# STEP 2A: Standardise JSON order-item column names

json_order_items_named = (
    json_order_items_raw.rename(
        columns={
            "orderItemID": "order_item_id",
            "orderID": "order_id",
            "productID": "product_id",
            "quantity": "quantity",
            "unitPrice": "unit_price",
            "lineRevenue": "reported_line_revenue",
        }
    ).copy()
)

json_order_items_named["source_format"] = "JSON"

json_order_items_named = json_order_items_named[
    [
        "order_item_id",
        "order_id",
        "product_id",
        "quantity",
        "unit_price",
        "reported_line_revenue",
        "source_format",
    ]
]

print(
    "JSON order-item columns renamed:",
    len(json_order_items_named),
    "rows and",
    len(json_order_items_named.columns),
    "columns",
)

print("\nNew column names:")
print(json_order_items_named.columns.tolist())

display(json_order_items_named.head())

# STEP 2B: Standardise XML order-item column names

xml_order_items_named = (
    xml_order_items_raw.rename(
        columns={
            "Order_Item_ID": "order_item_id",
            "Order_ID": "order_id",
            "Product_ID": "product_id",
            "Quantity": "quantity",
            "Unit_Price": "unit_price",
            "Line_Revenue": "reported_line_revenue",
        }
    ).copy()
)

xml_order_items_named["source_format"] = "XML"

xml_order_items_named = xml_order_items_named[
    [
        "order_item_id",
        "order_id",
        "product_id",
        "quantity",
        "unit_price",
        "reported_line_revenue",
        "source_format",
    ]
]

print(
    "XML order-item columns renamed:",
    len(xml_order_items_named),
    "rows and",
    len(xml_order_items_named.columns),
    "columns",
)

print("\nNew column names:")
print(xml_order_items_named.columns.tolist())

display(xml_order_items_named.head())

# STEP 3A: Standardise JSON order-item data types

json_order_items_clean = (
    json_order_items_named.copy()
)

identifier_columns = [
    "order_item_id",
    "order_id",
    "product_id",
]

for column in identifier_columns:
    json_order_items_clean[column] = (
        json_order_items_clean[column]
        .astype("string")
        .str.strip()
    )

json_order_items_clean["quantity"] = (
    pd.to_numeric(
        json_order_items_clean["quantity"],
        errors="raise",
    ).astype("int64")
)

json_order_items_clean["unit_price"] = (
    pd.to_numeric(
        json_order_items_clean["unit_price"],
        errors="raise",
    ).round(2)
)

json_order_items_clean[
    "reported_line_revenue"
] = (
    pd.to_numeric(
        json_order_items_clean[
            "reported_line_revenue"
        ],
        errors="raise",
    ).round(2)
)

json_order_items_clean["source_format"] = (
    json_order_items_clean["source_format"]
    .astype("string")
)

required_columns = [
    "order_item_id",
    "order_id",
    "product_id",
    "quantity",
    "unit_price",
    "reported_line_revenue",
]

missing_required_values = (
    json_order_items_clean[
        required_columns
    ].isna().sum().sum()
)

print("JSON order-item data types:")
print(json_order_items_clean.dtypes)

print(
    "\nMissing required values:",
    missing_required_values,
)

display(json_order_items_clean.head())

# STEP 3B: Standardise XML order-item data types

def parse_aud_amount(values):
    cleaned_values = (
        values.astype("string")
        .str.replace(
            "AUD",
            "",
            regex=False,
        )
        .str.replace(
            ",",
            "",
            regex=False,
        )
        .str.strip()
    )

    return pd.to_numeric(
        cleaned_values,
        errors="raise",
    ).round(2)


xml_order_items_clean = (
    xml_order_items_named.copy()
)

identifier_columns = [
    "order_item_id",
    "order_id",
    "product_id",
]

for column in identifier_columns:
    xml_order_items_clean[column] = (
        xml_order_items_clean[column]
        .astype("string")
        .str.strip()
    )

xml_order_items_clean["quantity"] = (
    pd.to_numeric(
        xml_order_items_clean["quantity"],
        errors="raise",
    ).astype("int64")
)

xml_order_items_clean["unit_price"] = (
    parse_aud_amount(
        xml_order_items_clean["unit_price"]
    )
)

xml_order_items_clean[
    "reported_line_revenue"
] = (
    parse_aud_amount(
        xml_order_items_clean[
            "reported_line_revenue"
        ]
    )
)

xml_order_items_clean["source_format"] = (
    xml_order_items_clean["source_format"]
    .astype("string")
)

required_columns = [
    "order_item_id",
    "order_id",
    "product_id",
    "quantity",
    "unit_price",
    "reported_line_revenue",
]

xml_missing_required_values = (
    xml_order_items_clean[
        required_columns
    ].isna().sum().sum()
)

print("XML order-item data types:")
print(xml_order_items_clean.dtypes)

print(
    "\nMissing required values:",
    xml_missing_required_values,
)

display(xml_order_items_clean.head())

# EVIDENCE: SEC-4.2-JSON-WITHIN-SOURCE-DEDUP

order_item_comparison_columns = [
    "order_id",
    "product_id",
    "quantity",
    "unit_price",
    "reported_line_revenue",
]

json_duplicate_item_rows = (
    json_order_items_clean[
        json_order_items_clean.duplicated(
            subset=["order_item_id"],
            keep=False,
        )
    ].copy()
)

json_duplicate_comparison = (
    json_duplicate_item_rows
    .groupby("order_item_id")[
        order_item_comparison_columns
    ]
    .nunique(dropna=False)
)

json_conflicting_item_ids = (
    json_duplicate_comparison[
        json_duplicate_comparison
        .gt(1)
        .any(axis=1)
    ]
    .index
    .tolist()
)

if json_conflicting_item_ids:
    raise ValueError(
        "Conflicting JSON order-item duplicates: "
        f"{json_conflicting_item_ids[:10]}"
    )

json_order_items_deduplicated = (
    json_order_items_clean
    .drop_duplicates(
        subset=["order_item_id"],
        keep="first",
    )
    .copy()
)

json_rows_removed = (
    len(json_order_items_clean)
    - len(json_order_items_deduplicated)
)

print(
    "JSON repeated order_item_id values:",
    json_duplicate_item_rows[
        "order_item_id"
    ].nunique(),
)

print(
    "JSON conflicting duplicate IDs:",
    len(json_conflicting_item_ids),
)

print(
    "JSON rows before deduplication:",
    len(json_order_items_clean),
)

print(
    "JSON rows removed:",
    json_rows_removed,
)

print(
    "JSON rows after deduplication:",
    len(json_order_items_deduplicated),
)

# EVIDENCE: SEC-4.2-XML-WITHIN-SOURCE-DEDUP

xml_duplicate_item_rows = (
    xml_order_items_clean[
        xml_order_items_clean.duplicated(
            subset=["order_item_id"],
            keep=False,
        )
    ].copy()
)

xml_duplicate_comparison = (
    xml_duplicate_item_rows
    .groupby("order_item_id")[
        order_item_comparison_columns
    ]
    .nunique(dropna=False)
)

xml_conflicting_item_ids = (
    xml_duplicate_comparison[
        xml_duplicate_comparison
        .gt(1)
        .any(axis=1)
    ]
    .index
    .tolist()
)

if xml_conflicting_item_ids:
    raise ValueError(
        "Conflicting XML order-item duplicates: "
        f"{xml_conflicting_item_ids[:10]}"
    )

xml_order_items_deduplicated = (
    xml_order_items_clean
    .drop_duplicates(
        subset=["order_item_id"],
        keep="first",
    )
    .copy()
)

xml_rows_removed = (
    len(xml_order_items_clean)
    - len(xml_order_items_deduplicated)
)

print(
    "XML repeated order_item_id values:",
    xml_duplicate_item_rows[
        "order_item_id"
    ].nunique(),
)

print(
    "XML conflicting duplicate IDs:",
    len(xml_conflicting_item_ids),
)

print(
    "XML rows before deduplication:",
    len(xml_order_items_clean),
)

print(
    "XML rows removed:",
    xml_rows_removed,
)

print(
    "XML rows after deduplication:",
    len(xml_order_items_deduplicated),
)

# EVIDENCE: SEC-4.2-CROSS-SOURCE-RECONCILIATION

money_tolerance = 0.01

order_item_overlap = (
    json_order_items_deduplicated.merge(
        xml_order_items_deduplicated,
        on="order_item_id",
        how="inner",
        suffixes=("_json", "_xml"),
        validate="one_to_one",
    )
)

exact_conflict = (
    (
        order_item_overlap["order_id_json"]
        != order_item_overlap["order_id_xml"]
    )
    | (
        order_item_overlap["product_id_json"]
        != order_item_overlap["product_id_xml"]
    )
    | (
        order_item_overlap["quantity_json"]
        != order_item_overlap["quantity_xml"]
    )
)

money_conflict = (
    (
        order_item_overlap["unit_price_json"]
        - order_item_overlap["unit_price_xml"]
    ).abs()
    > money_tolerance
) | (
    (
        order_item_overlap[
            "reported_line_revenue_json"
        ]
        - order_item_overlap[
            "reported_line_revenue_xml"
        ]
    ).abs()
    > money_tolerance
)

order_item_overlap["has_conflict"] = (
    exact_conflict | money_conflict
)

conflicting_overlap_items = (
    order_item_overlap[
        order_item_overlap["has_conflict"]
    ].copy()
)

if not conflicting_overlap_items.empty:
    raise ValueError(
        "Cross-source order-item conflicts found: "
        + str(
            conflicting_overlap_items[
                "order_item_id"
            ].head(10).tolist()
        )
    )

json_item_ids = set(
    json_order_items_deduplicated[
        "order_item_id"
    ]
)

xml_item_ids = set(
    xml_order_items_deduplicated[
        "order_item_id"
    ]
)

overlap_item_ids = (
    json_item_ids & xml_item_ids
)

json_only_item_ids = (
    json_item_ids - xml_item_ids
)

xml_only_item_ids = (
    xml_item_ids - json_item_ids
)

order_items_reconciled = pd.concat(
    [
        json_order_items_deduplicated,
        xml_order_items_deduplicated,
    ],
    ignore_index=True,
)

order_items_reconciled = (
    order_items_reconciled
    .drop_duplicates(
        subset=["order_item_id"],
        keep="first",
    )
    .copy()
)

order_items_reconciled.loc[
    order_items_reconciled[
        "order_item_id"
    ].isin(overlap_item_ids),
    "source_format",
] = "both"

order_items_reconciled = (
    order_items_reconciled
    .sort_values("order_item_id")
    .reset_index(drop=True)
)

print(
    "Cross-source overlapping order items:",
    len(overlap_item_ids),
)

print(
    "Cross-source conflicts:",
    len(conflicting_overlap_items),
)

print(
    "JSON-only order items:",
    len(json_only_item_ids),
)

print(
    "XML-only order items:",
    len(xml_only_item_ids),
)

print(
    "Canonical order items:",
    len(order_items_reconciled),
)

print("\nSource coverage:")
print(
    order_items_reconciled[
        "source_format"
    ].value_counts()
)

# EVIDENCE: SEC-4.2-LINE-REVENUE-DERIVATION

calculated_line_revenue = (
    order_items_reconciled["quantity"]
    * order_items_reconciled["unit_price"]
).round(2)

line_revenue_difference = (
    calculated_line_revenue
    - order_items_reconciled[
        "reported_line_revenue"
    ]
).abs()

line_revenue_mismatch_mask = (
    line_revenue_difference
    > money_tolerance
)

line_revenue_mismatches = (
    order_items_reconciled[
        line_revenue_mismatch_mask
    ].copy()
)

if not line_revenue_mismatches.empty:
    raise ValueError(
        "Calculated line revenue does not match "
        "the reported source value for: "
        + str(
            line_revenue_mismatches[
                "order_item_id"
            ].head(10).tolist()
        )
    )

order_items_standardised = (
    order_items_reconciled.copy()
)

order_items_standardised[
    "line_revenue"
] = calculated_line_revenue

order_items_standardised = (
    order_items_standardised[
        [
            "order_item_id",
            "order_id",
            "product_id",
            "quantity",
            "unit_price",
            "line_revenue",
        ]
    ].copy()
)

print(
    "Line-revenue mismatches:",
    len(line_revenue_mismatches),
)

print(
    "Maximum absolute difference:",
    line_revenue_difference.max(),
)

print(
    "Standardised order_items:",
    len(order_items_standardised),
    "rows and",
    len(order_items_standardised.columns),
    "columns",
)

display(order_items_standardised.head())

# STEP 8A: Validate order_items to orders relationship
# EVIDENCE: SEC-4.2-ORDER-FOREIGN-KEY-VALIDATION

order_item_order_id_set = set(
    order_items_standardised[
        "order_id"
    ]
)

reconciled_order_id_set = set(
    orders_reconciled[
        "order_id"
    ]
)

order_ids_missing_from_orders = sorted(
    order_item_order_id_set
    - reconciled_order_id_set
)

orders_without_order_items = sorted(
    reconciled_order_id_set
    - order_item_order_id_set
)

order_item_order_foreign_key_valid = (
    len(order_ids_missing_from_orders)
    == 0
)

all_orders_have_order_items = (
    len(orders_without_order_items)
    == 0
)

if not order_item_order_foreign_key_valid:
    raise ValueError(
        "Some order_items refer to missing orders: "
        f"{order_ids_missing_from_orders[:10]}"
    )

if not all_orders_have_order_items:
    raise ValueError(
        "Some orders have no order_items: "
        f"{orders_without_order_items[:10]}"
    )

print(
    "Order-item rows:",
    len(order_items_standardised),
)

print(
    "Distinct order IDs in order_items:",
    len(order_item_order_id_set),
)

print(
    "Distinct order IDs in orders:",
    len(reconciled_order_id_set),
)

print(
    "Item order IDs missing from orders:",
    len(order_ids_missing_from_orders),
)

print(
    "Orders without order_items:",
    len(orders_without_order_items),
)

print(
    "Order-item to order foreign key valid:",
    order_item_order_foreign_key_valid,
)

print(
    "Every order has at least one item:",
    all_orders_have_order_items,
)

print(
    "order_item_id is unique:",
    order_items_standardised[
        "order_item_id"
    ].is_unique,
)

# STEP 8B: Derive order_price from order_items
# EVIDENCE: SEC-4.1-ORDER-PRICE-DERIVATION

order_price_from_items = (
    order_items_standardised
    .groupby(
        "order_id",
        as_index=False,
        sort=True,
    )["line_revenue"]
    .sum()
    .rename(
        columns={
            "line_revenue": "order_price",
        }
    )
)

order_price_from_items[
    "order_price"
] = (
    order_price_from_items[
        "order_price"
    ].round(2)
)

orders_with_calculated_price = (
    orders_reconciled
    .merge(
        order_price_from_items,
        on="order_id",
        how="left",
        validate="one_to_one",
    )
)

missing_calculated_order_prices = (
    orders_with_calculated_price[
        "order_price"
    ].isna().sum()
)

order_price_difference = (
    orders_with_calculated_price[
        "order_price"
    ]
    .sub(
        orders_with_calculated_price[
            "reported_order_price"
        ]
    )
    .abs()
)

order_price_mismatch_rows = (
    orders_with_calculated_price[
        order_price_difference.gt(
            0.01 + 0.000000001
        )
    ]
)

if missing_calculated_order_prices:
    raise ValueError(
        "Some orders have no calculated order_price"
    )

if not order_price_mismatch_rows.empty:
    raise ValueError(
        "Calculated order_price conflicts with "
        "reported source values for: "
        f"{order_price_mismatch_rows['order_id'].head(10).tolist()}"
    )

print(
    "Orders calculated from order_items:",
    len(order_price_from_items),
)

print(
    "Orders after price merge:",
    len(orders_with_calculated_price),
)

print(
    "Missing calculated order prices:",
    missing_calculated_order_prices,
)

print(
    "Order-price mismatches:",
    len(order_price_mismatch_rows),
)

print(
    "Maximum order-price difference:",
    order_price_difference.max(),
)

display(
    orders_with_calculated_price[
        [
            "order_id",
            "reported_order_price",
            "order_price",
        ]
    ].head()
)

# STEP 8C: Derive tax_amount from order_price
# EVIDENCE: SEC-4.1-TAX-AMOUNT-DERIVATION

orders_with_calculated_tax = (
    orders_with_calculated_price.copy()
)

orders_with_calculated_tax[
    "tax_amount"
] = (
    orders_with_calculated_tax[
        "order_price"
    ]
    .div(11)
    .round(2)
)

tax_amount_difference = (
    orders_with_calculated_tax[
        "tax_amount"
    ]
    .sub(
        orders_with_calculated_tax[
            "reported_tax_amount"
        ]
    )
    .abs()
)

tax_amount_mismatch_rows = (
    orders_with_calculated_tax[
        tax_amount_difference.gt(
            0.01 + 0.000000001
        )
    ]
)

missing_calculated_tax_amounts = (
    orders_with_calculated_tax[
        "tax_amount"
    ].isna().sum()
)

if missing_calculated_tax_amounts:
    raise ValueError(
        "Some orders have no calculated tax_amount"
    )

if not tax_amount_mismatch_rows.empty:
    raise ValueError(
        "Calculated tax_amount conflicts with "
        "reported source values for: "
        f"{tax_amount_mismatch_rows['order_id'].head(10).tolist()}"
    )

print(
    "Orders with calculated tax:",
    len(orders_with_calculated_tax),
)

print(
    "Missing calculated tax amounts:",
    missing_calculated_tax_amounts,
)

print(
    "Tax-amount mismatches:",
    len(tax_amount_mismatch_rows),
)

print(
    "Maximum tax-amount difference:",
    tax_amount_difference.max(),
)

display(
    orders_with_calculated_tax[
        [
            "order_id",
            "order_price",
            "reported_tax_amount",
            "tax_amount",
        ]
    ].head()
)

# STEP 8D: Derive order_total
# EVIDENCE: SEC-4.1-ORDER-TOTAL-DERIVATION

orders_with_calculated_total = (
    orders_with_calculated_tax.copy()
)

orders_with_calculated_total[
    "discounted_order_price"
] = (
    orders_with_calculated_total[
        "order_price"
    ]
    * (
        1
        - (
            orders_with_calculated_total[
                "coupon_discount"
            ]
            / 100
        )
    )
)

orders_with_calculated_total[
    "order_total"
] = (
    orders_with_calculated_total[
        "discounted_order_price"
    ]
    .add(
        orders_with_calculated_total[
            "delivery_charges"
        ]
    )
    .round(2)
)

order_total_difference = (
    orders_with_calculated_total[
        "order_total"
    ]
    .sub(
        orders_with_calculated_total[
            "reported_order_total"
        ]
    )
    .abs()
)

order_total_mismatch_rows = (
    orders_with_calculated_total[
        order_total_difference.gt(
            0.01 + 0.000000001
        )
    ]
)

missing_calculated_order_totals = (
    orders_with_calculated_total[
        "order_total"
    ].isna().sum()
)

if missing_calculated_order_totals:
    raise ValueError(
        "Some orders have no calculated order_total"
    )

if not order_total_mismatch_rows.empty:
    raise ValueError(
        "Calculated order_total conflicts with "
        "reported source values for: "
        f"{order_total_mismatch_rows['order_id'].head(10).tolist()}"
    )

print(
    "Orders with calculated total:",
    len(orders_with_calculated_total),
)

print(
    "Missing calculated order totals:",
    missing_calculated_order_totals,
)

print(
    "Order-total mismatches:",
    len(order_total_mismatch_rows),
)

print(
    "Maximum order-total difference:",
    order_total_difference.max(),
)

display(
    orders_with_calculated_total[
        [
            "order_id",
            "order_price",
            "coupon_discount",
            "discounted_order_price",
            "delivery_charges",
            "reported_order_total",
            "order_total",
        ]
    ].head()
)

# STEP 9A: Prepare orders for Task 3 text functions

order_non_text_output_columns = [
    "order_id",
    "source_system_record_id",
    "customer_id",
    "order_timestamp",
    "sales_channel",
    "payment_method",
    "currency",
    "nearest_warehouse",
    "order_status",
    "order_price",
    "delivery_charges",
    "coupon_code",
    "coupon_discount",
    "tax_amount",
    "order_total",
    "season",
    "expedited_delivery",
    "customer_lat",
    "customer_long",
    "device_type",
    "referral_source",
]

missing_non_text_order_columns = sorted(
    set(order_non_text_output_columns)
    - set(
        orders_with_calculated_total.columns
    )
)

if missing_non_text_order_columns:
    raise ValueError(
        "Missing required order columns: "
        f"{missing_non_text_order_columns}"
    )

orders_ready_for_text = (
    orders_with_calculated_total[
        order_non_text_output_columns
        + ["raw_customer_note"]
    ]
    .copy()
    .sort_values("order_id")
    .reset_index(drop=True)
)

pending_task3_order_columns = [
    "customer_note_clean",
    "promo_code",
]

print(
    "Orders ready for Task 3:",
    len(orders_ready_for_text),
    "rows and",
    len(orders_ready_for_text.columns),
    "columns",
)

print(
    "Missing required non-text columns:",
    missing_non_text_order_columns,
)

print(
    "Columns still supplied by Task 3:",
    pending_task3_order_columns,
)

display(
    orders_ready_for_text[
        [
            "order_id",
            "order_price",
            "tax_amount",
            "order_total",
            "raw_customer_note",
        ]
    ].head()
)

# STEP 9B: Prepare the final order_items table

order_item_output_columns = [
    "order_item_id",
    "order_id",
    "product_id",
    "quantity",
    "unit_price",
    "line_revenue",
]

order_items_output_ready = (
    order_items_standardised[
        order_item_output_columns
    ]
    .copy()
    .sort_values("order_item_id")
    .reset_index(drop=True)
)

order_item_column_order_correct = (
    order_items_output_ready.columns.tolist()
    == order_item_output_columns
)

order_item_missing_required_values = (
    order_items_output_ready[
        order_item_output_columns
    ]
    .isna()
    .sum()
    .sum()
)

invalid_quantity_rows = (
    order_items_output_ready[
        order_items_output_ready[
            "quantity"
        ].le(0)
    ]
)

invalid_order_item_amount_rows = (
    order_items_output_ready[
        order_items_output_ready[
            "unit_price"
        ].lt(0)
        | order_items_output_ready[
            "line_revenue"
        ].lt(0)
    ]
)

recalculated_line_revenue = (
    order_items_output_ready[
        "quantity"
    ]
    .mul(
        order_items_output_ready[
            "unit_price"
        ]
    )
    .round(2)
)

final_line_revenue_difference = (
    order_items_output_ready[
        "line_revenue"
    ]
    .sub(recalculated_line_revenue)
    .abs()
)

final_line_revenue_mismatch_rows = (
    order_items_output_ready[
        final_line_revenue_difference.gt(
            0.01 + 0.000000001
        )
    ]
)

if not order_item_column_order_correct:
    raise ValueError(
        "order_items columns are not in "
        "the required order"
    )

if order_item_missing_required_values:
    raise ValueError(
        "order_items contains missing "
        "required values"
    )

if not order_items_output_ready[
    "order_item_id"
].is_unique:
    raise ValueError(
        "order_item_id is not unique"
    )

if not invalid_quantity_rows.empty:
    raise ValueError(
        "order_items contains invalid quantities"
    )

if not invalid_order_item_amount_rows.empty:
    raise ValueError(
        "order_items contains invalid monetary values"
    )

if not final_line_revenue_mismatch_rows.empty:
    raise ValueError(
        "Final line_revenue formula check failed"
    )

print(
    "Order-items output rows:",
    len(order_items_output_ready),
)

print(
    "Order-items output columns:",
    len(order_items_output_ready.columns),
)

print(
    "Column order correct:",
    order_item_column_order_correct,
)

print(
    "Missing required values:",
    order_item_missing_required_values,
)

print(
    "order_item_id is unique:",
    order_items_output_ready[
        "order_item_id"
    ].is_unique,
)

print(
    "Invalid quantity rows:",
    len(invalid_quantity_rows),
)

print(
    "Invalid monetary rows:",
    len(invalid_order_item_amount_rows),
)

print(
    "Line-revenue formula mismatches:",
    len(final_line_revenue_mismatch_rows),
)

print(
    "Order foreign key valid:",
    order_item_order_foreign_key_valid,
)

print("\nFinal data types:")
print(order_items_output_ready.dtypes)

display(order_items_output_ready.head())

# STEP 9C: Validate product references against source catalogue

source_catalogue_product_ids = {
    str(
        product_record.get(
            "Product_ID",
            "",
        )
    ).strip()
    for product_record in xml_product_records
    if str(
        product_record.get(
            "Product_ID",
            "",
        )
    ).strip()
}

order_item_product_id_set = set(
    order_items_output_ready[
        "product_id"
    ]
)

product_ids_missing_from_catalogue = sorted(
    order_item_product_id_set
    - source_catalogue_product_ids
)

catalogue_products_without_order_items = sorted(
    source_catalogue_product_ids
    - order_item_product_id_set
)

source_product_reference_valid = (
    len(
        product_ids_missing_from_catalogue
    )
    == 0
)

if not source_product_reference_valid:
    raise ValueError(
        "Some order_items refer to products "
        "missing from the source catalogue: "
        f"{product_ids_missing_from_catalogue[:10]}"
    )

print(
    "Distinct product IDs in order_items:",
    len(order_item_product_id_set),
)

print(
    "Distinct product IDs in source catalogue:",
    len(source_catalogue_product_ids),
)

print(
    "Item product IDs missing from catalogue:",
    len(product_ids_missing_from_catalogue),
)

print(
    "Catalogue products without order_items:",
    len(catalogue_products_without_order_items),
)

print(
    "Source product references valid:",
    source_product_reference_valid,
)

# Finalise the two official target tables using the shared Task 2 text logic.
_orders_raw_notes = orders_ready_for_text["raw_customer_note"].copy()
orders = (
    orders_ready_for_text.drop(columns="raw_customer_note")
    .assign(
        customer_note_clean=_orders_raw_notes.map(clean_narrative_text),
        promo_code=_orders_raw_notes.map(extract_promo_code),
    )
)
_order_target_columns = (
    data_dictionary.loc[data_dictionary["output_table"].eq("orders")]
    .assign(_position=lambda frame: pd.to_numeric(frame["position"]))
    .sort_values("_position")["field_name"]
    .tolist()
)
_order_item_target_columns = (
    data_dictionary.loc[data_dictionary["output_table"].eq("order_items")]
    .assign(_position=lambda frame: pd.to_numeric(frame["position"]))
    .sort_values("_position")["field_name"]
    .tolist()
)
orders = orders.loc[:, _order_target_columns].sort_values(
    "order_id", kind="mergesort"
).reset_index(drop=True)
orders["order_timestamp"] = pd.to_datetime(
    orders["order_timestamp"]
).dt.strftime("%Y-%m-%d %H:%M:%S")
order_items = order_items_output_ready.loc[
    :, _order_item_target_columns
].sort_values("order_item_id", kind="mergesort").reset_index(drop=True)
show(orders.head(5), "Canonical orders preview")
show(order_items.head(5), "Canonical order_items preview")
print(f"ORDERS_TASK2_BUILD_PASS rows={len(orders)}")
print(f"ORDER_ITEMS_TASK2_BUILD_PASS rows={len(order_items)}")


JSON order items extracted: 8884 rows and 6 columns

Column names:
['lineRevenue', 'orderID', 'orderItemID', 'productID', 'quantity', 'unitPrice']


,lineRevenue,orderID,orderItemID,productID,quantity,unitPrice
0,318.00,HORD002902,HITM0009045,PRD0759,3,106.00
1,491.21,HORD002902,HITM0009046,PRD0192,1,491.21
2,834.78,HORD002902,HITM0009047,PRD0075,2,417.39
3,72.90,HORD002902,HITM0009048,PRD0009,1,72.90
4,291.07,HORD002426,HITM0007559,PRD0013,1,291.07


XML order items extracted: 8833 rows and 6 columns

Column names:
['Order_Item_ID', 'Order_ID', 'Product_ID', 'Quantity', 'Unit_Price', 'Line_Revenue']


,Order_Item_ID,Order_ID,Product_ID,Quantity,Unit_Price,Line_Revenue
0,HITM0003384,HORD001093,PRD0243,1,AUD 734.66,AUD 734.66
1,HITM0003385,HORD001093,PRD0387,1,"AUD 1,127.53","AUD 1,127.53"
2,HITM0010711,HORD003432,PRD0716,1,AUD 744.09,AUD 744.09
3,HITM0010712,HORD003432,PRD0343,1,"AUD 1,454.71","AUD 1,454.71"
4,HITM0010713,HORD003432,PRD0576,2,AUD 31.35,AUD 62.70


JSON order-item columns renamed: 8884 rows and 7 columns

New column names:
['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'reported_line_revenue', 'source_format']


...

,order_id,order_price,tax_amount,order_total,raw_customer_note
0,HORD000001,2047.92,186.17,1955.24,[SYSTEM] <p>Call before delivery</p> https://o...
1,HORD000002,6549.78,595.43,6234.97,[SYSTEM] <p>Please leave at reception</p> http...
2,HORD000003,3047.54,277.05,2904.73,[SYSTEM] <p>Please leave at reception</p> http...
3,HORD000004,584.43,53.13,597.1,[SYSTEM] <p>Please use minimal packaging</p> h...
4,HORD000005,2280.74,207.34,2293.2,[SYSTEM] <p>Gift purchase for a family member<...


Order-items output rows: 15711
Order-items output columns: 6
Column order correct: True
Missing required values: 0
order_item_id is unique: True
Invalid quantity rows: 0
Invalid monetary rows: 0
Line-revenue formula mismatches: 0
Order foreign key valid: True

Final data types:
order_item_id     string
order_id          string
product_id        string
quantity           int64
unit_price       Float64
line_revenue     Float64
dtype: object


,order_item_id,order_id,product_id,quantity,unit_price,line_revenue
0,HITM0000001,HORD000001,PRD0837,1,1658.39,1658.39
1,HITM0000002,HORD000001,PRD0776,1,133.13,133.13
2,HITM0000003,HORD000001,PRD0180,1,256.4,256.4
3,HITM0000004,HORD000002,PRD0726,1,830.43,830.43
4,HITM0000005,HORD000002,PRD0626,2,362.78,725.56


Distinct product IDs in order_items: 1000
Distinct product IDs in source catalogue: 1000
Item product IDs missing from catalogue: 0
Catalogue products without order_items: 0
Source product references valid: True



Canonical orders preview
  order_id source_system_record_id customer_id     order_timestamp sales_channel payment_method currency nearest_warehouse order_status  order_price  delivery_charges coupon_code  coupon_discount  tax_amount  order_total season  expedited_delivery  customer_lat  customer_long device_type referral_source               customer_note_clean promo_code
HORD000001        SRC-050-H-000001    CUS00191 2018-11-12 19:33:00         Store         PayPal      AUD          Thompson    Completed      2047.92              9.72         NaN              5.0      186.17      1955.24 Spring               False    -37.839582     144.941232      Tablet         Organic              call before delivery        NaN
HORD000002        SRC-050-H-000002    CUS00096 2018-04-19 19:08:00        Mobile           Card      AUD            Bakers    Completed      6549.78             12.68         NaN              5.0      595.43      6234.97 Autumn               False    -37.751456     145.0322

### 4.3 `customers`


In [11]:
# Integrated member component: customers
# EVIDENCE: SEC-4.3-CUSTOMERS
# TASK SCOPE: Build the Task 2 standardised customers table.
#
# REQUIREMENT CROSSWALK
# REQ-01/02: Build exactly one row per customer at the published grain.
# REQ-03: customer_id is the required, non-null primary key. The customers
#            table itself has no foreign key; FK checks belong to orders and
#            product_reviews after all six tables have been integrated.
# REQ-04: Obtain field names, order, data types and nullability from the
#            public data dictionary instead of manually defining the output.
# REQ-05: customerProfiles[] is already one repeated record per customer; it
#            contains no nested one-to-many array that needs flattening.
# REQ-06: Trim/NFC-normalise identifiers and categories without changing case
#            or converting identifiers/postcodes to numbers (leading zeroes).
# REQ-07: Validate and standardise signup_date as YYYY-MM-DD.
# REQ-08: Keep marketing_consent as a real bool for True/False CSV output.
# REQ-09: Remove commas before numeric conversion if a numeric source value
#            contains a thousands separator. No customer field has a currency
#            label, but the numeric parser still rejects malformed values.
# REQ-10: No customer target field is a percentage; not applicable here.
# REQ-11: No customer revenue/order/tax amount is derived; not applicable.
# REQ-12: Every customer string is non-nullable in the dictionary, so missing
#            input is an error. Literal 'NaN' applies only to prescribed
#            nullable string outputs elsewhere in the six-table model.
# REQ-13/14: Reconcile duplicate customer_id groups field by field, record all
#            conflicts, and never apply silent first/last-source precedence.
# REQ-15: Derive row counts and conflict counts from the current input; do not
#            hard-code row counts or expected values.
# REQ-16: Keep helper fields internal; customers contains dictionary fields
#            only. The shared final cell must export Group050_customers.csv.

from datetime import datetime
from decimal import Decimal, InvalidOperation, ROUND_HALF_UP
import unicodedata

import pandas as pd


# ---------------------------------------------------------------------------
# 1. Confirm that the Task 1 parsing and dictionary cells were run first.
# ---------------------------------------------------------------------------
required_customer_inputs = ("data_dictionary", "json_customers", "show")
missing_customer_inputs = [
    name for name in required_customer_inputs if name not in globals()
]
if missing_customer_inputs:
    raise RuntimeError(
        "Run all Task 1 cells before SEC-4.3-CUSTOMERS. "
        f"Missing objects: {missing_customer_inputs}"
    )


# ---------------------------------------------------------------------------
# 2. Read the customer contract from public_data_dictionary.csv.  [REQ-04]
# ---------------------------------------------------------------------------
customer_dictionary = (
    data_dictionary.loc[
        data_dictionary["output_table"].eq("customers")
    ]
    .assign(_position=lambda frame: frame["position"].astype(int))
    .sort_values("_position", kind="stable")
    .drop(columns="_position")
    .reset_index(drop=True)
)

if customer_dictionary.empty:
    raise ValueError("The public data dictionary has no customers contract.")

customer_columns = customer_dictionary["field_name"].tolist()
customer_type_by_field = customer_dictionary.set_index("field_name")[
    "data_type"
].to_dict()
customer_nullable_by_field = customer_dictionary.set_index("field_name")[
    "nullable"
].astype(str).str.strip().str.lower().eq("true").to_dict()


# Source-to-target lineage. The insertion order is intentionally the published
# customer field order, but the output is still reordered from the dictionary.
CUSTOMER_SOURCE_FIELDS = {
    "customer_id": "customerID",
    "signup_date": "signupDate",
    "loyalty_tier": "loyaltyTier",
    "customer_segment": "customerSegment",
    "age_band": "ageBand",
    "preferred_channel": "preferredChannel",
    "home_suburb": "homeSuburb",
    "prior_12m_orders": "prior12MOrders",
    "lifetime_value_before_period": "lifetimeValueBeforePeriod",
    "marketing_consent": "marketingConsent",
    "home_postcode": "homePostcode",
    "home_state": "homeState",
    "home_country": "homeCountry",
    "preferred_language": "preferredLanguage",
    "acquisition_source": "acquisitionSource",
    "account_status": "accountStatus",
    "preferred_device": "preferredDevice",
    "email_domain": "emailDomain",
    "household_size_band": "householdSizeBand",
    "contact_frequency_preference": "contactFrequencyPreference",
}

if set(CUSTOMER_SOURCE_FIELDS) != set(customer_columns):
    raise AssertionError(
        "Customer lineage does not match the public dictionary: "
        f"lineage_only={sorted(set(CUSTOMER_SOURCE_FIELDS) - set(customer_columns))}, "
        f"dictionary_only={sorted(set(customer_columns) - set(CUSTOMER_SOURCE_FIELDS))}"
    )


# ---------------------------------------------------------------------------
# 3. Small strict parsers: fail early instead of silently coercing bad data.
# ---------------------------------------------------------------------------
def _required_customer_text(value, source_field):
    """Apply REQ-06 while preserving identifier/category case and zeroes."""
    if value is None:
        raise ValueError(f"{source_field}: required text is missing")
    result = unicodedata.normalize("NFC", str(value)).strip()
    if not result:
        raise ValueError(f"{source_field}: required text is empty")
    return result


def _customer_date(value, source_field):
    """Apply REQ-07: validate and return an exact YYYY-MM-DD string."""
    text = _required_customer_text(value, source_field)
    try:
        parsed = datetime.strptime(text, "%Y-%m-%d")
    except ValueError as exc:
        raise ValueError(
            f"{source_field}: invalid YYYY-MM-DD date {value!r}"
        ) from exc
    return parsed.strftime("%Y-%m-%d")


def _customer_integer(value, source_field):
    """Convert an integral value without rounding a fractional value."""
    if value is None or type(value) is bool:
        raise ValueError(f"{source_field}: invalid integer {value!r}")
    try:
        number = Decimal(str(value).replace(",", "").strip())  # REQ-09
    except InvalidOperation as exc:
        raise ValueError(f"{source_field}: invalid integer {value!r}") from exc
    if not number.is_finite() or number != number.to_integral_value():
        raise ValueError(f"{source_field}: expected an integer, got {value!r}")
    return int(number)


def _customer_number_2dp(value, source_field):
    """Convert a number and apply its published 0.01 comparison precision."""
    if value is None or type(value) is bool:
        raise ValueError(f"{source_field}: invalid number {value!r}")
    try:
        number = Decimal(str(value).replace(",", "").strip())  # REQ-09
    except InvalidOperation as exc:
        raise ValueError(f"{source_field}: invalid number {value!r}") from exc
    if not number.is_finite():
        raise ValueError(f"{source_field}: non-finite number {value!r}")
    return float(number.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP))


def _customer_boolean(value, source_field):
    """Apply REQ-08: accept only a real JSON bool, not truthy strings/ints."""
    if type(value) is not bool:
        raise ValueError(
            f"{source_field}: expected JSON True/False, got {value!r}"
        )
    return value


CUSTOMER_PARSERS = {
    "signup_date": _customer_date,
    "prior_12m_orders": _customer_integer,
    "lifetime_value_before_period": _customer_number_2dp,
    "marketing_consent": _customer_boolean,
}


def _standardise_customer_record(record, source_row_number):
    """Map one customerProfiles[] object to one target-grain row. [REQ-01/05]"""
    standardised = {}
    for target_field in customer_columns:
        source_field = CUSTOMER_SOURCE_FIELDS[target_field]
        raw_value = record.get(source_field)
        parser = CUSTOMER_PARSERS.get(target_field, _required_customer_text)
        standardised[target_field] = parser(raw_value, source_field)

    # Helper is retained only until reconciliation and is never exported.
    standardised["_source_row_number"] = source_row_number
    return standardised


# ---------------------------------------------------------------------------
# 4. Transform the source array without hard-coded row counts. [REQ-05/15]
# ---------------------------------------------------------------------------
customer_staging = pd.DataFrame(
    [
        _standardise_customer_record(record, source_row_number)
        for source_row_number, record in enumerate(json_customers, start=1)
    ]
)


# ---------------------------------------------------------------------------
# 5. Reconcile duplicate IDs and explicitly report conflicts. [REQ-13/14]
# ---------------------------------------------------------------------------
customer_conflicts = []
canonical_customer_rows = []

for customer_id, group in customer_staging.groupby(
    "customer_id", sort=False, dropna=False
):
    canonical = {"customer_id": customer_id}
    for field in customer_columns[1:]:
        distinct_values = list(pd.unique(group[field].dropna()))
        if len(distinct_values) > 1:
            customer_conflicts.append(
                {
                    "customer_id": customer_id,
                    "field": field,
                    "values": repr(distinct_values),
                    "source_rows": repr(group["_source_row_number"].tolist()),
                }
            )
        elif distinct_values:
            canonical[field] = distinct_values[0]
        else:
            canonical[field] = None
    canonical_customer_rows.append(canonical)

customer_conflict_register = pd.DataFrame(
    customer_conflicts,
    columns=["customer_id", "field", "values", "source_rows"],
)

# Never choose a conflicting source value silently. No source-precedence rule
# is justified for this JSON-only table, so a genuine conflict stops the build.
if not customer_conflict_register.empty:
    show(customer_conflict_register, "Customer conflict register")
    raise ValueError(
        "Conflicting customer duplicates require investigation before export."
    )


# REQ-16: output only dictionary fields, in dictionary order; helper columns
# such as _source_row_number stay in customer_staging and cannot leak to CSV.
customers = pd.DataFrame(canonical_customer_rows, columns=customer_columns)


# Apply stable pandas dtypes that match the dictionary contract. [REQ-04]
for field, data_type in customer_type_by_field.items():
    if data_type == "string":
        customers[field] = customers[field].astype("string")
    elif data_type == "boolean":
        customers[field] = customers[field].astype("bool")
    elif data_type == "number" and field == "prior_12m_orders":
        customers[field] = customers[field].astype("int64")
    elif data_type == "number":
        customers[field] = customers[field].astype("float64")
    elif data_type == "date":
        # Keep the exported representation as an exact YYYY-MM-DD string.
        customers[field] = customers[field].astype("string")
    else:
        raise ValueError(
            f"Unsupported customer dictionary type {data_type!r} for {field!r}"
        )


# ---------------------------------------------------------------------------
# 6. Executable validation evidence tied to the requirements.
# ---------------------------------------------------------------------------
required_fields = [
    field
    for field, nullable in customer_nullable_by_field.items()
    if not nullable
]
string_fields = [
    field
    for field, data_type in customer_type_by_field.items()
    if data_type in {"string", "date"}
]

null_count = int(customers[required_fields].isna().sum().sum())
blank_string_count = int(
    sum(customers[field].str.strip().eq("").sum() for field in string_fields)
)
date_format_ok = customers["signup_date"].str.fullmatch(
    r"\d{4}-\d{2}-\d{2}"
).all()
date_parse_ok = pd.to_datetime(
    customers["signup_date"], format="%Y-%m-%d", errors="coerce"
).notna().all()

# Compare normalised source identifiers/postcodes with the outputs. This is a
# direct leading-zero/case preservation check, not merely a regex check.
source_customer_ids = {
    _required_customer_text(record.get("customerID"), "customerID")
    for record in json_customers
}
source_postcodes_by_customer = {
    _required_customer_text(record.get("customerID"), "customerID"):
    _required_customer_text(record.get("homePostcode"), "homePostcode")
    for record in json_customers
}
identifier_preservation_ok = set(customers["customer_id"]) == source_customer_ids
postcode_preservation_ok = all(
    row.home_postcode == source_postcodes_by_customer[row.customer_id]
    for row in customers[["customer_id", "home_postcode"]].itertuples(index=False)
)

customer_checks = pd.DataFrame(
    [
        {
            "check_id": "T2-CUSTOMERS-REQ-01-04",
            "requirement": "exact grain/schema/order/types",
            "observed": (
                f"rows={len(customers)}, columns={len(customers.columns)}, "
                f"exact_order={list(customers.columns) == customer_columns}"
            ),
            "passed": list(customers.columns) == customer_columns,
        },
        {
            "check_id": "T2-CUSTOMERS-REQ-03",
            "requirement": "non-null unique customer_id primary key",
            "observed": (
                f"null={customers['customer_id'].isna().sum()}, "
                f"duplicate={customers['customer_id'].duplicated().sum()}"
            ),
            "passed": customers["customer_id"].notna().all()
            and customers["customer_id"].is_unique,
        },
        {
            "check_id": "T2-CUSTOMERS-REQ-06",
            "requirement": "identifier/case/leading-zero preservation",
            "observed": (
                f"customer_id_match={identifier_preservation_ok}, "
                f"postcode_match={postcode_preservation_ok}"
            ),
            "passed": identifier_preservation_ok and postcode_preservation_ok,
        },
        {
            "check_id": "T2-CUSTOMERS-REQ-07",
            "requirement": "signup_date is valid YYYY-MM-DD",
            "observed": f"format={date_format_ok}, parseable={date_parse_ok}",
            "passed": bool(date_format_ok and date_parse_ok),
        },
        {
            "check_id": "T2-CUSTOMERS-REQ-08",
            "requirement": "marketing_consent contains real booleans",
            "observed": repr(sorted(customers["marketing_consent"].unique().tolist())),
            "passed": customers["marketing_consent"].map(
                lambda value: type(value) is bool
            ).all(),
        },
        {
            "check_id": "T2-CUSTOMERS-REQ-12",
            "requirement": "required fields have no null/empty values",
            "observed": f"null={null_count}, empty_string={blank_string_count}",
            "passed": null_count == 0 and blank_string_count == 0,
        },
        {
            "check_id": "T2-CUSTOMERS-REQ-13-14",
            "requirement": "duplicate reconciliation has no unresolved conflict",
            "observed": (
                f"source_rows={len(customer_staging)}, "
                f"unique_ids={customer_staging['customer_id'].nunique()}, "
                f"conflicts={len(customer_conflict_register)}"
            ),
            "passed": customer_conflict_register.empty,
        },
        {
            "check_id": "T2-CUSTOMERS-REQ-15-16",
            "requirement": "reproducible row flow and no helper output columns",
            "observed": (
                f"source_rows={len(json_customers)}, output_rows={len(customers)}, "
                f"helper_columns={sorted(set(customers.columns) - set(customer_columns))}"
            ),
            "passed": len(customers) == customer_staging["customer_id"].nunique()
            and list(customers.columns) == customer_columns,
        },
    ]
)
customer_checks["status"] = customer_checks["passed"].map(
    {True: "PASS", False: "FAIL"}
)

failed_customer_checks = customer_checks.loc[
    ~customer_checks["passed"].astype(bool)
]
if not failed_customer_checks.empty:
    raise AssertionError(
        "Customer Task 2 validation failed:\n"
        + failed_customer_checks.to_string(index=False)
    )


customer_row_flow = pd.DataFrame(
    [
        {
            "stage": "JSON customerProfiles[]",
            "rows": len(json_customers),
            "unique_customer_id": customer_staging["customer_id"].nunique(),
        },
        {
            "stage": "standardised customers",
            "rows": len(customers),
            "unique_customer_id": customers["customer_id"].nunique(),
        },
    ]
)

show(customer_row_flow, "Customer row flow (derived, not hard-coded)")
show(
    customer_checks.drop(columns="passed"),
    "Customer requirement validation",
)
show(customers.head(), "Standardised customers sample")



Customer row flow (derived, not hard-coded)
                  stage  rows  unique_customer_id
JSON customerProfiles[]   500                 500
 standardised customers   500                 500

Customer requirement validation
              check_id                                         requirement                                            observed status
T2-CUSTOMERS-REQ-01-04                      exact grain/schema/order/types              rows=500, columns=20, exact_order=True   PASS
   T2-CUSTOMERS-REQ-03             non-null unique customer_id primary key                                 null=0, duplicate=0   PASS
   T2-CUSTOMERS-REQ-06           identifier/case/leading-zero preservation         customer_id_match=True, postcode_match=True   PASS
   T2-CUSTOMERS-REQ-07                     signup_date is valid YYYY-MM-DD                         format=True, parseable=True   PASS
   T2-CUSTOMERS-REQ-08            marketing_consent contains real booleans                            

### 4.4 `deliveries`

Build and reconcile the 19 target fields that do not depend on another
member. Retain the structurally parsed delivery note as a helper input;
the twentieth target field is created automatically only when the real
shared cleaner is integrated. No partial submitted CSV is written.


In [12]:
# EVIDENCE: SEC-4.4-DELIVERY-NORMALISATION
import re
import unicodedata
from datetime import datetime
from decimal import Decimal

DELIVERY_SOURCE_FIELDS = {
    "delivery_id": ("deliveryID", "Delivery_ID"),
    "order_id": ("orderID", "Order_ID"),
    "dispatch_date": ("dispatchDate", "Dispatch_Date"),
    "promised_date": ("promisedDate", "Promised_Date"),
    "delivered_date": ("deliveredDate", "Delivered_Date"),
    "carrier": ("carrier", "Carrier"),
    "service_level": ("serviceLevel", "Service_Level"),
    "delivery_status": ("deliveryStatus", "Delivery_Status"),
    "delay_days": ("delayDays", "Delay_Days"),
    "on_time_in_full": ("onTimeInFull", "On_Time_In_Full"),
    "fulfilment_hours": ("fulfilmentHours", "Fulfilment_Hours"),
    "delivery_cost": ("deliveryCost", "Delivery_Cost"),
    "delay_reason": ("delayReason", "Delay_Reason"),
    "promised_days": ("promisedDays", "Promised_Days"),
    "tracking_event_count": ("trackingEventCount", "Tracking_Event_Count"),
    "delivery_window": ("deliveryWindow", "Delivery_Window"),
    "shipping_distance_km": ("shippingDistanceKm", "Shipping_Distance_Km"),
    "signature_required": ("signatureRequired", "Signature_Required"),
    "estimated_carbon_kg": ("estimatedCarbonKg", "Estimated_Carbon_Kg"),
    "delivery_note_clean": ("deliveryNoteClean", "Delivery_Note_Clean"),
}
DELIVERY_TARGET_COLUMNS = list(DELIVERY_SOURCE_FIELDS)
DELIVERY_PREINTEGRATION_COLUMNS = (
    DELIVERY_TARGET_COLUMNS[:-1] + ["_delivery_note_raw"]
)
DELIVERY_DATE_FIELDS = {
    "dispatch_date",
    "promised_date",
    "delivered_date",
}
DELIVERY_BOOLEAN_FIELDS = {"on_time_in_full", "signature_required"}
DELIVERY_INTEGER_FIELDS = {
    "delay_days",
    "fulfilment_hours",
    "promised_days",
    "tracking_event_count",
}
DELIVERY_FLOAT_FIELDS = {
    "delivery_cost",
    "shipping_distance_km",
    "estimated_carbon_kg",
}

def _required_delivery_text(value, field_name):
    """NFC-normalise and trim a required structured delivery value."""

    if value is None:
        raise ValueError(f"Required {field_name} is missing")
    result = unicodedata.normalize("NFC", str(value)).strip()
    if not result:
        raise ValueError(f"Required {field_name} is empty")
    return result

def _normalise_delivery_date(value, source_name, field_name):
    source_format = "%Y-%m-%d" if source_name == "JSON" else "%d/%m/%Y"
    text = _required_delivery_text(value, field_name)
    return datetime.strptime(text, source_format).strftime("%Y-%m-%d")

def _normalise_delivery_boolean(value, field_name):
    if isinstance(value, bool):
        return value
    token = _required_delivery_text(value, field_name).upper()
    if token in {"Y", "TRUE"}:
        return True
    if token in {"N", "FALSE"}:
        return False
    raise ValueError(
        f"Unexpected boolean token for {field_name}: {value!r}"
    )

def _normalise_delivery_decimal(value, field_name):
    token = _required_delivery_text(value, field_name)
    token = re.sub(r"^[A-Za-z]{3}\s*", "", token).replace(",", "")
    return Decimal(token)

def _normalise_delivery_record(raw_record, source_name):
    source_position = 0 if source_name == "JSON" else 1
    result = {}
    for target_field, source_fields in DELIVERY_SOURCE_FIELDS.items():
        value = raw_record[source_fields[source_position]]
        if target_field in DELIVERY_DATE_FIELDS:
            result[target_field] = _normalise_delivery_date(
                value, source_name, target_field
            )
        elif target_field in DELIVERY_BOOLEAN_FIELDS:
            result[target_field] = _normalise_delivery_boolean(
                value, target_field
            )
        elif target_field in DELIVERY_INTEGER_FIELDS:
            number = _normalise_delivery_decimal(value, target_field)
            if number != number.to_integral_value():
                raise ValueError(
                    f"Non-integral value for {target_field}: {value!r}"
                )
            result[target_field] = int(number)
        elif target_field in DELIVERY_FLOAT_FIELDS:
            number = _normalise_delivery_decimal(value, target_field)
            result[target_field] = (
                float(round(number, 2))
                if target_field == "delivery_cost"
                else float(number)
            )
        elif target_field == "delivery_note_clean":
            result["_delivery_note_raw"] = _required_delivery_text(
                value, target_field
            )
        else:
            result[target_field] = _required_delivery_text(
                value, target_field
            )
    return result

deliveries_json_normalised = pd.DataFrame(
    [
        _normalise_delivery_record(record, "JSON")
        for record in json_deliveries
    ],
    columns=DELIVERY_PREINTEGRATION_COLUMNS,
)
deliveries_xml_normalised = pd.DataFrame(
    [
        _normalise_delivery_record(record, "XML")
        for record in xml_delivery_records
    ],
    columns=DELIVERY_PREINTEGRATION_COLUMNS,
)

delivery_source_normalisation_summary = pd.DataFrame(
    [
        {
            "source": "JSON",
            "raw_rows": len(deliveries_json_normalised),
            "unique_delivery_ids": deliveries_json_normalised[
                "delivery_id"
            ].nunique(),
            "missing_required_values": int(
                deliveries_json_normalised.isna().sum().sum()
            ),
        },
        {
            "source": "XML",
            "raw_rows": len(deliveries_xml_normalised),
            "unique_delivery_ids": deliveries_xml_normalised[
                "delivery_id"
            ].nunique(),
            "missing_required_values": int(
                deliveries_xml_normalised.isna().sum().sum()
            ),
        },
    ]
)
show(
    delivery_source_normalisation_summary,
    "Delivery source normalisation",
)



Delivery source normalisation
source  raw_rows  unique_delivery_ids  missing_required_values
  JSON      2818                 2750                        0
   XML      2818                 2750                        0


### 4.5 `products`


In [13]:
# Integrated member component: products
# EVIDENCE: SEC-4.5-PRODUCTS
# TASK SCOPE: Build only the Task 2 standardised products table.
# This cell does not export a CSV or implement the Task 3 deliverable module.

from datetime import datetime
from decimal import Decimal, InvalidOperation, ROUND_HALF_UP
import html
import re
import unicodedata

import pandas as pd


def _normalise_required_product_text(value, field_name):
    """
    Usage:
        _normalise_required_product_text(raw_value, "Product_ID")

    Reason:
        Required product identifiers and structured categories must be trimmed
        and NFC-normalised while preserving leading zeros and source case.

    Expected result:
        A non-empty string. Missing or empty input raises ValueError.
    """
    if value is None:
        raise ValueError(f"{field_name}: required text is missing")

    result = unicodedata.normalize("NFC", str(value)).strip()

    if not result:
        raise ValueError(f"{field_name}: required text is empty")

    return result


def _parse_product_currency(value, field_name):
    """
    Usage:
        _parse_product_currency("AUD 2,686.14", "Unit_Price")

    Reason:
        XML prices contain an AUD label and thousands separators that must be
        removed before numeric conversion.

    Expected result:
        A float rounded to two decimal places, such as 2686.14.
    """
    text = _normalise_required_product_text(value, field_name)
    numeric_text = re.sub(r"(?i)\bAUD\b", "", text)
    numeric_text = numeric_text.replace(",", "").strip()

    try:
        amount = Decimal(numeric_text)
    except InvalidOperation as exc:
        raise ValueError(
            f"{field_name}: invalid currency value {value!r}"
        ) from exc

    return float(
        amount.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
    )


def _parse_product_integer(value, field_name):
    """
    Usage:
        _parse_product_integer("24", "Warranty_Months")

    Reason:
        Integer product attributes must be converted without silently rounding
        non-integral values.

    Expected result:
        A Python int. Invalid or non-integral input raises ValueError.
    """
    text = _normalise_required_product_text(value, field_name)

    try:
        number = Decimal(text)
    except InvalidOperation as exc:
        raise ValueError(
            f"{field_name}: invalid integer value {value!r}"
        ) from exc

    if number != number.to_integral_value():
        raise ValueError(
            f"{field_name}: expected an integer, got {value!r}"
        )

    return int(number)


def _parse_product_number(value, field_name):
    """
    Usage:
        _parse_product_number("1.551", "Weight_Kg")

    Reason:
        Numeric measurements must be converted from XML text while retaining
        their published precision.

    Expected result:
        A Python float, such as 1.551.
    """
    text = _normalise_required_product_text(value, field_name)

    try:
        return float(Decimal(text))
    except InvalidOperation as exc:
        raise ValueError(
            f"{field_name}: invalid numeric value {value!r}"
        ) from exc


def _parse_product_date(value, field_name):
    """
    Usage:
        _parse_product_date("26/06/2014", "Launch_Date")

    Reason:
        XML Product launch dates use DD/MM/YYYY, while the target requires
        YYYY-MM-DD.

    Expected result:
        A date string such as "2014-06-26".
    """
    text = _normalise_required_product_text(value, field_name)

    try:
        return datetime.strptime(text, "%d/%m/%Y").strftime("%Y-%m-%d")
    except ValueError as exc:
        raise ValueError(
            f"{field_name}: invalid DD/MM/YYYY date {value!r}"
        ) from exc


def _parse_product_boolean(value, field_name):
    """
    Usage:
        _parse_product_boolean("Y", "Active_Flag")

    Reason:
        XML represents Product booleans as Y/N; the target requires True/False.

    Expected result:
        True for Y and False for N. Other values raise ValueError.
    """
    text = _normalise_required_product_text(
        value, field_name
    ).upper()

    boolean_mapping = {
        "Y": True,
        "N": False,
    }

    if text not in boolean_mapping:
        raise ValueError(
            f"{field_name}: expected Y or N, got {value!r}"
        )

    return boolean_mapping[text]


def _remove_product_description_emoji(text):
    """
    Usage:
        _remove_product_description_emoji(normalised_text)

    Reason:
        The published narrative-cleaning sequence removes emoji. This private
        helper is limited to the Product table and is not the assessed Task 3
        function.

    Expected result:
        The same string with common emoji code-point ranges removed.
    """
    emoji_ranges = (
        (0x1F000, 0x1FAFF),
        (0x2600, 0x27BF),
        (0x2300, 0x23FF),
        (0x2B00, 0x2BFF),
        (0xFE00, 0xFE0F),
        (0x1F1E6, 0x1F1FF),
    )

    return "".join(
        character
        for character in text
        if character not in {"\u200d", "\u20e3"}
        and not any(
            start <= ord(character) <= end
            for start, end in emoji_ranges
        )
    )


def _clean_product_description_for_task2(value):
    """
    Usage:
        _clean_product_description_for_task2(raw_product_description)

    Reason:
        Task 2 requires product_description_clean now, but the shared assessed
        clean_narrative_text function belongs to Task 3 and may not exist yet.

        If clean_narrative_text has already been loaded by the final combined
        workflow, this helper delegates to it. Otherwise, it applies the
        published cleaning sequence privately for the Product table only.

    Expected result:
        Lower-case cleaned text, or the literal string "NaN" when no readable
        text remains.
    """
    shared_cleaner = globals().get("clean_narrative_text")

    if callable(shared_cleaner):
        result = shared_cleaner(value)

        if not isinstance(result, str) or result == "":
            raise ValueError(
                "clean_narrative_text must return non-empty text or "
                "the literal string 'NaN'."
            )

        return result

    if value is None:
        return "NaN"

    # 1. Decode HTML entities and apply Unicode NFC normalisation.
    text = html.unescape(str(value))
    text = unicodedata.normalize("NFC", text)

    # 2. Remove HTML/XML-like tags while preserving readable content.
    text = re.sub(r"<[^>]*>", " ", text)

    # 3. Remove fixed and parameterised published markers.
    text = re.sub(
        r"\[(?:SYSTEM|CATALOGUE|VERIFIED_PURCHASE)\]",
        " ",
        text,
        flags=re.IGNORECASE,
    )
    text = re.sub(
        r"\[SOURCE:\s*[^\]]*\]",
        " ",
        text,
        flags=re.IGNORECASE,
    )
    text = re.sub(
        r"\[RATING:\s*[0-5]\s*/\s*5\]",
        " ",
        text,
        flags=re.IGNORECASE,
    )
    text = re.sub(
        r"(?<![\w-])(?:#verified-buyer|@store_support)(?![\w-])",
        " ",
        text,
        flags=re.IGNORECASE,
    )

    # 4. Remove URLs.
    text = re.sub(
        r"(?i)\b(?:https?://|www\.)\S+",
        " ",
        text,
    )

    # 5. Remove emoji.
    text = _remove_product_description_emoji(text)

    # 6. Remove a complete review reference wrapper if one occurs.
    text = re.sub(
        (
            r"(?i)(?<![A-Z0-9])"
            r"Reference:\s*(?:HORD|CORD)\d{6}"
            r"\s*[|,;/]\s*"
            r"SKU:\s*SKU-[A-Z0-9]+"
            r"(?![A-Z0-9-])"
        ),
        " ",
        text,
    )

    # 7. Remove PROMO: together with a valid promotion code.
    text = re.sub(
        (
            r"(?i)(?<![A-Z0-9])"
            r"PROMO:\s*B[1-5]SAVE-\d{2}"
            r"(?![A-Z0-9-])"
        ),
        " ",
        text,
    )

    # 8. Collapse whitespace, trim and lower-case the cleaned narrative.
    text = re.sub(r"\s+", " ", text).strip().lower()

    # 9. Use the literal missing-string sentinel when no text remains.
    return text if text else "NaN"


def _standardise_product_record(record):
    """
    Usage:
        _standardise_product_record(one_xml_product_record)

    Reason:
        Converts one structured XML Product record to the exact target fields.
        XML structure has already been parsed in Task 1.

    Expected result:
        One dictionary containing exactly the 21 Product target fields.
    """
    return {
        "product_id": _normalise_required_product_text(
            record.get("Product_ID"),
            "Product_ID",
        ),
        "product_name": _normalise_required_product_text(
            record.get("Product_Name"),
            "Product_Name",
        ),
        "category": _normalise_required_product_text(
            record.get("Category"),
            "Category",
        ),
        "brand": _normalise_required_product_text(
            record.get("Brand"),
            "Brand",
        ),
        "unit_price": _parse_product_currency(
            record.get("Unit_Price"),
            "Unit_Price",
        ),
        "unit_cost": _parse_product_currency(
            record.get("Unit_Cost"),
            "Unit_Cost",
        ),
        "launch_year": _parse_product_integer(
            record.get("Launch_Year"),
            "Launch_Year",
        ),
        "warranty_months": _parse_product_integer(
            record.get("Warranty_Months"),
            "Warranty_Months",
        ),
        "weight_kg": _parse_product_number(
            record.get("Weight_Kg"),
            "Weight_Kg",
        ),
        "product_sku": _normalise_required_product_text(
            record.get("Product_Sku"),
            "Product_Sku",
        ),
        "subcategory": _normalise_required_product_text(
            record.get("Subcategory"),
            "Subcategory",
        ),
        "model_family": _normalise_required_product_text(
            record.get("Model_Family"),
            "Model_Family",
        ),
        "colour": _normalise_required_product_text(
            record.get("Colour"),
            "Colour",
        ),
        "supplier_id": _normalise_required_product_text(
            record.get("Supplier_ID"),
            "Supplier_ID",
        ),
        "supplier_country": _normalise_required_product_text(
            record.get("Supplier_Country"),
            "Supplier_Country",
        ),
        "launch_date": _parse_product_date(
            record.get("Launch_Date"),
            "Launch_Date",
        ),
        "tax_category": _normalise_required_product_text(
            record.get("Tax_Category"),
            "Tax_Category",
        ),
        "package_type": _normalise_required_product_text(
            record.get("Package_Type"),
            "Package_Type",
        ),
        "recyclable_packaging": _parse_product_boolean(
            record.get("Recyclable_Packaging"),
            "Recyclable_Packaging",
        ),
        "active_flag": _parse_product_boolean(
            record.get("Active_Flag"),
            "Active_Flag",
        ),
        "product_description_clean": (
            _clean_product_description_for_task2(
                record.get("Product_Description")
            )
        ),
    }


def _reconcile_product_candidates(frame):
    """
    Usage:
        products, conflicts = _reconcile_product_candidates(candidates)

    Reason:
        Product rows must be reconciled by product_id after normalisation.
        Conflicting non-missing values must be recorded rather than resolved
        using arbitrary row or source precedence.

    Expected result:
        One canonical row per product_id and a separate conflict DataFrame.
    """
    canonical_rows = []
    conflict_rows = []

    for product_id, group in frame.groupby(
        "product_id",
        sort=True,
        dropna=False,
    ):
        canonical = {"product_id": product_id}

        for column in frame.columns:
            if column == "product_id":
                continue

            non_missing_values = [
                value
                for value in group[column].tolist()
                if not pd.isna(value)
            ]

            # Remove duplicate values while preserving deterministic order.
            distinct_values = list(dict.fromkeys(non_missing_values))

            if len(distinct_values) > 1:
                conflict_rows.append(
                    {
                        "product_id": product_id,
                        "field": column,
                        "normalised_values": distinct_values,
                    }
                )

            canonical[column] = (
                distinct_values[0]
                if distinct_values
                else None
            )

        canonical_rows.append(canonical)

    canonical_frame = pd.DataFrame(
        canonical_rows,
        columns=frame.columns,
    )
    conflict_frame = pd.DataFrame(
        conflict_rows,
        columns=[
            "product_id",
            "field",
            "normalised_values",
        ],
    )

    return canonical_frame, conflict_frame


# Read the required Product schema and field order from the public dictionary.
# This avoids maintaining a second manual output schema.
product_columns = (
    data_dictionary.loc[
        data_dictionary["output_table"].eq("products")
    ]
    .assign(
        _position=lambda frame: frame["position"].astype(int)
    )
    .sort_values("_position")["field_name"]
    .tolist()
)

# xml_product_records was created using the structured XML parser in Task 1.
product_candidates = pd.DataFrame(
    [
        _standardise_product_record(record)
        for record in xml_product_records
    ],
    columns=product_columns,
)

# Reconcile by the published Product primary key.
products, product_reconciliation_conflicts = (
    _reconcile_product_candidates(product_candidates)
)

# Do not silently select a value when a conflict is found.
if not product_reconciliation_conflicts.empty:
    raise ValueError(
        "Conflicting non-missing normalised Product values were found:\n"
        + product_reconciliation_conflicts.to_string(index=False)
    )

# Keep only the required fields and make row ordering deterministic.
products = (
    products.loc[:, product_columns]
    .sort_values("product_id", kind="stable")
    .reset_index(drop=True)
)

# Immediate Task 2 transformation guards.
# These are local construction guards for the table.
if list(products.columns) != product_columns:
    raise AssertionError(
        "Product columns do not match the public data dictionary."
    )

if products["product_id"].isna().any():
    raise AssertionError(
        "products.product_id contains missing values."
    )

if not products["product_id"].is_unique:
    raise AssertionError(
        "products.product_id is not unique."
    )

if products.isna().any().any():
    missing_counts = products.isna().sum()
    missing_counts = missing_counts[missing_counts.gt(0)]

    raise AssertionError(
        "Products contains prohibited missing values:\n"
        + missing_counts.to_string()
    )

product_string_columns = (
    data_dictionary.loc[
        data_dictionary["output_table"].eq("products")
        & data_dictionary["data_type"].eq("string"),
        "field_name",
    ]
    .tolist()
)

blank_string_counts = {
    column: int(
        products[column]
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )
    for column in product_string_columns
}
blank_string_counts = {
    column: count
    for column, count in blank_string_counts.items()
    if count > 0
}

if blank_string_counts:
    raise AssertionError(
        "Products contains empty required strings: "
        f"{blank_string_counts}"
    )

# Display Task 2 row-flow evidence without hard-coding expected row counts.
product_row_flow = pd.DataFrame(
    [
        {
            "stage": "structured XML product records",
            "rows": len(xml_product_records),
        },
        {
            "stage": "normalised Product candidates",
            "rows": len(product_candidates),
        },
        {
            "stage": "canonical products",
            "rows": len(products),
        },
    ]
)

show(
    product_row_flow,
    "Product transformation row flow",
)
show(
    products.head(),
    "Standardised products preview",
)

cleaner_source = (
    "shared clean_narrative_text"
    if callable(globals().get("clean_narrative_text"))
    else "private Task 2 Product fallback"
)

print(
    "\nProduct table ready:"
    f" rows={len(products)},"
    f" columns={len(products.columns)},"
    f" reconciliation_conflicts="
    f"{len(product_reconciliation_conflicts)},"
    f" cleaner={cleaner_source}"
)


Product transformation row flow
                         stage  rows
structured XML product records  1000
 normalised Product candidates  1000
            canonical products  1000

Standardised products preview
product_id     product_name           category  brand  unit_price  unit_cost  launch_year  warranty_months  weight_kg  product_sku         subcategory model_family   colour supplier_id supplier_country launch_date tax_category package_type  recyclable_packaging  active_flag                                                                                                                        product_description_clean
   PRD0001 Candle Bloom 100             Laptop Candle     2686.14    1498.92         2014               24      1.551 SKU-CAN00001           Ultrabook          Arc Graphite      SUP001        Australia  2014-06-26 GST_STANDARD Recycled box                  True         True             ultrabook designed for portable document work and video meetings, in the arc fami

### 4.6 `product_reviews`


In [14]:
# Integrated member component: product_reviews
# EVIDENCE: SEC-4.6-PRODUCT-REVIEWS
# TASK SCOPE: Build only the Task 2 standardised product_reviews table.
# This cell builds the assigned Task 2 table only.

from datetime import datetime
from decimal import Decimal, InvalidOperation
import html
import re
import unicodedata

import pandas as pd


# ---------------------------------------------------------------------------
# 1. Task 1 prerequisite check
# ---------------------------------------------------------------------------

# Usage:
#   Run this Product Review cell after all Task 1 cells and the Product cell.
# Reason:
#   Product Reviews reuses the structured JSON/XML objects and dictionary
#   produced by Task 1 instead of parsing the source documents again.
# Expected result:
#   No exception. If an object is missing, the message identifies what must
#   be run first.
required_product_review_inputs = [
    "data_dictionary",
    "json_reviews",
    "xml_review_records",
    "show",
]

missing_product_review_inputs = [
    name
    for name in required_product_review_inputs
    if name not in globals()
]

if missing_product_review_inputs:
    raise RuntimeError(
        "Run all Task 1 cells before the Product Review cell. "
        f"Missing objects: {missing_product_review_inputs}"
    )


# ---------------------------------------------------------------------------
# 2. General Product Review standardisation helpers
# ---------------------------------------------------------------------------

def _normalise_required_review_text(value, field_name):
    """
    Usage:
        _normalise_required_review_text(raw_value, "Review_ID")

    Reason:
        Required identifiers and structured review attributes must be trimmed
        and Unicode NFC-normalised without changing their source case or
        removing identifier leading zeros.

    Expected result:
        A non-empty string. Missing or empty input raises ValueError.
    """
    if value is None:
        raise ValueError(f"{field_name}: required text is missing")

    result = unicodedata.normalize("NFC", str(value)).strip()

    if not result:
        raise ValueError(f"{field_name}: required text is empty")

    return result


def _parse_required_review_integer(value, field_name):
    """
    Usage:
        _parse_required_review_integer("5", "Rating")

    Reason:
        Rating and helpful-vote values are represented as numbers in JSON but
        text in XML. Both sources must produce the same target type.

    Expected result:
        A Python int. Invalid or non-integral values raise ValueError.
    """
    text = _normalise_required_review_text(value, field_name)

    try:
        number = Decimal(text)
    except InvalidOperation as exc:
        raise ValueError(
            f"{field_name}: invalid integer value {value!r}"
        ) from exc

    if number != number.to_integral_value():
        raise ValueError(
            f"{field_name}: expected an integer, got {value!r}"
        )

    return int(number)


def _parse_review_timestamp(value, field_name, source_format):
    """
    Usage:
        _parse_review_timestamp(value, "Review_Timestamp", "XML")

    Reason:
        JSON timestamps use YYYY-MM-DD HH:MM:SS, while XML timestamps use
        DD/MM/YYYY HH:MM:SS. The target requires YYYY-MM-DD HH:MM:SS.

    Expected result:
        A standardised timestamp string such as "2018-06-16 18:58:00".
    """
    text = _normalise_required_review_text(value, field_name)

    source_patterns = {
        "JSON": "%Y-%m-%d %H:%M:%S",
        "XML": "%d/%m/%Y %H:%M:%S",
    }

    if source_format not in source_patterns:
        raise ValueError(
            f"Unsupported review source format: {source_format!r}"
        )

    try:
        parsed = datetime.strptime(
            text,
            source_patterns[source_format],
        )
    except ValueError as exc:
        raise ValueError(
            f"{field_name}: invalid {source_format} timestamp {value!r}"
        ) from exc

    return parsed.strftime("%Y-%m-%d %H:%M:%S")


def _parse_review_boolean(value, field_name, source_format):
    """
    Usage:
        _parse_review_boolean("Y", "Verified_Purchase", "XML")

    Reason:
        JSON uses real booleans, while XML uses Y/N. The standardised output
        must contain Python True or False.

    Expected result:
        A Python bool. Unsupported values raise ValueError.
    """
    if source_format == "JSON":
        if type(value) is not bool:
            raise ValueError(
                f"{field_name}: expected a JSON boolean, got {value!r}"
            )

        return value

    if source_format == "XML":
        text = _normalise_required_review_text(
            value,
            field_name,
        ).upper()

        xml_boolean_mapping = {
            "Y": True,
            "N": False,
        }

        if text not in xml_boolean_mapping:
            raise ValueError(
                f"{field_name}: expected Y or N, got {value!r}"
            )

        return xml_boolean_mapping[text]

    raise ValueError(
        f"Unsupported review source format: {source_format!r}"
    )


# ---------------------------------------------------------------------------
# 3. Shared Task 3 text functions
#
# The six fixed-interface functions were imported and tested in Section 3.
# Product Review transformation calls those shared functions directly.
# ---------------------------------------------------------------------------

# ---------------------------------------------------------------------------
# 4. Convert one structured JSON/XML review to target fields
# ---------------------------------------------------------------------------

def _standardise_product_review_record(record, source_format):
    """
    Usage:
        _standardise_product_review_record(record, "JSON")
        _standardise_product_review_record(record, "XML")

    Reason:
        Maps the two source-specific representations to the same 21-field
        Product Review target schema.

    Expected result:
        One normalised dictionary at review grain with no helper fields.
    """
    source_field_maps = {
        "JSON": {
            "review_id": "reviewID",
            "order_id": "orderID",
            "order_item_id": "orderItemID",
            "product_id": "productID",
            "customer_id": "customerID",
            "review_timestamp": "reviewTimestamp",
            "language_code": "languageCode",
            "rating": "rating",
            "review_title": "reviewTitle",
            "review_text": "reviewText",
            "verified_purchase": "verifiedPurchase",
            "helpful_votes": "helpfulVotes",
            "delivery_experience": "deliveryExperience",
            "value_experience": "valueExperience",
            "writing_style": "writingStyle",
        },
        "XML": {
            "review_id": "Review_ID",
            "order_id": "Order_ID",
            "order_item_id": "Order_Item_ID",
            "product_id": "Product_ID",
            "customer_id": "Customer_ID",
            "review_timestamp": "Review_Timestamp",
            "language_code": "Language_Code",
            "rating": "Rating",
            "review_title": "Review_Title",
            "review_text": "Review_Text",
            "verified_purchase": "Verified_Purchase",
            "helpful_votes": "Helpful_Votes",
            "delivery_experience": "Delivery_Experience",
            "value_experience": "Value_Experience",
            "writing_style": "Writing_Style",
        },
    }

    if source_format not in source_field_maps:
        raise ValueError(
            f"Unsupported review source format: {source_format!r}"
        )

    fields = source_field_maps[source_format]
    raw_review = record.get(fields["review_text"])

    # Extract references from the raw review before cleaning.
    extracted_order_reference = extract_order_reference(raw_review)
    extracted_product_sku = extract_product_sku(raw_review)

    # Clean the multilingual review after extraction.
    review_body_clean = clean_narrative_text(raw_review)

    if not isinstance(review_body_clean, str) or review_body_clean == "":
        raise ValueError(
            "clean_narrative_text must return cleaned text or "
            "the literal string 'NaN'."
        )

    if (
        not isinstance(extracted_order_reference, str)
        or extracted_order_reference == ""
    ):
        raise ValueError(
            "extract_order_reference must return a reference or 'NaN'."
        )

    if (
        not isinstance(extracted_product_sku, str)
        or extracted_product_sku == ""
    ):
        raise ValueError(
            "extract_product_sku must return a SKU or 'NaN'."
        )

    # Multilingual fields must be derived from review_body_clean.
    review_body_latin_analysis = build_latin_analysis(review_body_clean)
    review_contains_non_latin = contains_non_latin_script(review_body_clean)

    if (
        not isinstance(review_body_latin_analysis, str)
        or review_body_latin_analysis == ""
    ):
        raise ValueError(
            "build_latin_analysis must return text or the literal 'NaN'."
        )

    if type(review_contains_non_latin) is not bool:
        raise ValueError(
            "contains_non_latin_script must return a Python bool."
        )

    if review_body_clean == "NaN":
        review_length_chars = 0
        review_word_count = 0
    else:
        review_length_chars = len(review_body_clean)
        review_word_count = len(review_body_clean.split())

    return {
        "review_id": _normalise_required_review_text(
            record.get(fields["review_id"]),
            fields["review_id"],
        ),
        "order_id": _normalise_required_review_text(
            record.get(fields["order_id"]),
            fields["order_id"],
        ),
        "order_item_id": _normalise_required_review_text(
            record.get(fields["order_item_id"]),
            fields["order_item_id"],
        ),
        "product_id": _normalise_required_review_text(
            record.get(fields["product_id"]),
            fields["product_id"],
        ),
        "customer_id": _normalise_required_review_text(
            record.get(fields["customer_id"]),
            fields["customer_id"],
        ),
        "review_timestamp": _parse_review_timestamp(
            record.get(fields["review_timestamp"]),
            fields["review_timestamp"],
            source_format,
        ),
        "language_code": _normalise_required_review_text(
            record.get(fields["language_code"]),
            fields["language_code"],
        ),
        "rating": _parse_required_review_integer(
            record.get(fields["rating"]),
            fields["rating"],
        ),
        "review_title": _normalise_required_review_text(
            record.get(fields["review_title"]),
            fields["review_title"],
        ),
        "review_body_clean": review_body_clean,
        "review_body_latin_analysis": review_body_latin_analysis,
        "verified_purchase": _parse_review_boolean(
            record.get(fields["verified_purchase"]),
            fields["verified_purchase"],
            source_format,
        ),
        "helpful_votes": _parse_required_review_integer(
            record.get(fields["helpful_votes"]),
            fields["helpful_votes"],
        ),
        "review_length_chars": review_length_chars,
        "review_word_count": review_word_count,
        "contains_non_latin_script": review_contains_non_latin,
        "extracted_order_reference": extracted_order_reference,
        "extracted_product_sku": extracted_product_sku,
        "delivery_experience": _normalise_required_review_text(
            record.get(fields["delivery_experience"]),
            fields["delivery_experience"],
        ),
        "value_experience": _normalise_required_review_text(
            record.get(fields["value_experience"]),
            fields["value_experience"],
        ),
        "writing_style": _normalise_required_review_text(
            record.get(fields["writing_style"]),
            fields["writing_style"],
        ),
    }


# ---------------------------------------------------------------------------
# 5. Reconcile JSON/XML candidates by review_id
# ---------------------------------------------------------------------------

def _reconcile_product_review_candidates(
    candidate_frame,
    output_columns,
):
    """
    Usage:
        canonical, conflicts = _reconcile_product_review_candidates(
            product_review_candidates,
            product_review_columns,
        )

    Reason:
        Within-source duplicates and cross-source overlap must be compared
        field by field after normalisation. Conflicts must be recorded instead
        of applying JSON-over-XML or XML-over-JSON precedence.

    Expected result:
        One canonical row per review_id and a separate conflict DataFrame.
    """
    canonical_rows = []
    conflict_rows = []

    for review_id, group in candidate_frame.groupby(
        "review_id",
        sort=True,
        dropna=False,
    ):
        canonical = {"review_id": review_id}

        for column in output_columns:
            if column == "review_id":
                continue

            non_missing_values = [
                value
                for value in group[column].tolist()
                if not pd.isna(value)
            ]

            distinct_values = list(dict.fromkeys(non_missing_values))

            if len(distinct_values) > 1:
                conflict_rows.append(
                    {
                        "review_id": review_id,
                        "field": column,
                        "normalised_values": distinct_values,
                        "source_rows": (
                            group["_source"]
                            .value_counts()
                            .to_dict()
                        ),
                    }
                )

            canonical[column] = (
                distinct_values[0]
                if distinct_values
                else None
            )

        canonical_rows.append(canonical)

    canonical_frame = pd.DataFrame(
        canonical_rows,
        columns=output_columns,
    )
    conflict_frame = pd.DataFrame(
        conflict_rows,
        columns=[
            "review_id",
            "field",
            "normalised_values",
            "source_rows",
        ],
    )

    return canonical_frame, conflict_frame


# Obtain the exact output field order from the public data dictionary.
product_review_columns = (
    data_dictionary.loc[
        data_dictionary["output_table"].eq("product_reviews")
    ]
    .assign(
        _position=lambda frame: frame["position"].astype(int)
    )
    .sort_values("_position")["field_name"]
    .tolist()
)

json_product_review_candidates = pd.DataFrame(
    [
        _standardise_product_review_record(record, "JSON")
        for record in json_reviews
    ],
    columns=product_review_columns,
).assign(_source="JSON")

xml_product_review_candidates = pd.DataFrame(
    [
        _standardise_product_review_record(record, "XML")
        for record in xml_review_records
    ],
    columns=product_review_columns,
).assign(_source="XML")

product_review_candidates = pd.concat(
    [
        json_product_review_candidates,
        xml_product_review_candidates,
    ],
    ignore_index=True,
)

product_reviews, product_review_reconciliation_conflicts = (
    _reconcile_product_review_candidates(
        product_review_candidates,
        product_review_columns,
    )
)

# Stop instead of silently resolving a conflicting normalized value.
if not product_review_reconciliation_conflicts.empty:
    raise ValueError(
        "Conflicting non-missing Product Review values were found:\n"
        + product_review_reconciliation_conflicts.to_string(index=False)
    )

product_reviews = (
    product_reviews.loc[:, product_review_columns]
    .sort_values("review_id", kind="stable")
    .reset_index(drop=True)
)


# ---------------------------------------------------------------------------
# 6. Immediate Task 2 transformation guards
#
# These are local construction guards for the table.
# ---------------------------------------------------------------------------

if list(product_reviews.columns) != product_review_columns:
    raise AssertionError(
        "Product Review columns do not match the public data dictionary."
    )

if product_reviews["review_id"].isna().any():
    raise AssertionError(
        "product_reviews.review_id contains missing values."
    )

if not product_reviews["review_id"].is_unique:
    raise AssertionError(
        "product_reviews.review_id is not unique."
    )

if product_reviews.isna().any().any():
    missing_counts = product_reviews.isna().sum()
    missing_counts = missing_counts[missing_counts.gt(0)]

    raise AssertionError(
        "Product Reviews contains prohibited missing values:\n"
        + missing_counts.to_string()
    )

product_review_string_columns = (
    data_dictionary.loc[
        data_dictionary["output_table"].eq("product_reviews")
        & data_dictionary["data_type"].eq("string"),
        "field_name",
    ]
    .tolist()
)

blank_string_counts = {
    column: int(
        product_reviews[column]
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )
    for column in product_review_string_columns
}
blank_string_counts = {
    column: count
    for column, count in blank_string_counts.items()
    if count > 0
}

if blank_string_counts:
    raise AssertionError(
        "Product Reviews contains empty required strings: "
        f"{blank_string_counts}"
    )

for boolean_column in [
    "verified_purchase",
    "contains_non_latin_script",
]:
    invalid_boolean_count = int(
        product_reviews[boolean_column]
        .map(lambda value: type(value) is not bool)
        .sum()
    )

    if invalid_boolean_count:
        raise AssertionError(
            f"{boolean_column} contains "
            f"{invalid_boolean_count} non-boolean values."
        )

invalid_timestamp_count = int(
    (
        ~product_reviews["review_timestamp"].str.fullmatch(
            r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}"
        )
    ).sum()
)

if invalid_timestamp_count:
    raise AssertionError(
        f"Product Reviews contains {invalid_timestamp_count} "
        "invalid timestamp formats."
    )


# Check the Product foreign key now because your products table is complete.
# The remaining parent-table checks require teammates' standardised tables.
if "products" in globals():
    missing_product_fk_values = sorted(
        set(product_reviews["product_id"])
        - set(products["product_id"])
    )

    if missing_product_fk_values:
        raise AssertionError(
            "Product Review product_id values are missing from products: "
            f"{missing_product_fk_values[:10]}"
        )

    product_fk_status = "PASS"
else:
    missing_product_fk_values = []
    product_fk_status = "PENDING - run the Product cell first"


# ---------------------------------------------------------------------------
# 7. Task 2 row-flow and preview evidence
# ---------------------------------------------------------------------------

product_review_row_flow = pd.DataFrame(
    [
        {
            "stage": "structured JSON review rows",
            "rows": len(json_reviews),
        },
        {
            "stage": "structured XML review rows",
            "rows": len(xml_review_records),
        },
        {
            "stage": "combined normalised candidates",
            "rows": len(product_review_candidates),
        },
        {
            "stage": "canonical product reviews",
            "rows": len(product_reviews),
        },
    ]
)

show(
    product_review_row_flow,
    "Product Review transformation row flow",
)

product_review_preview = product_reviews[
    [
        "review_id",
        "order_id",
        "order_item_id",
        "product_id",
        "customer_id",
        "review_timestamp",
        "language_code",
        "rating",
        "verified_purchase",
        "helpful_votes",
        "review_length_chars",
        "review_word_count",
        "contains_non_latin_script",
        "extracted_order_reference",
        "extracted_product_sku",
    ]
].head()

show(
    product_review_preview,
    "Standardised Product Review preview",
)

product_review_function_sources = pd.DataFrame(
    [
        {
            "required function": function_name,
            "current source": (
                "shared teammate function"
                if callable(globals().get(function_name))
                else "private Task 2 fallback"
            ),
        }
        for function_name in [
            "clean_narrative_text",
            "extract_order_reference",
            "extract_product_sku",
            "build_latin_analysis",
            "contains_non_latin_script",
        ]
    ]
)

show(
    product_review_function_sources,
    "Product Review function integration status",
)

print(
    "\nProduct Review table ready:"
    f" rows={len(product_reviews)},"
    f" columns={len(product_reviews.columns)},"
    f" reconciliation_conflicts="
    f"{len(product_review_reconciliation_conflicts)},"
    f" product_fk={product_fk_status}"
)

# Final output timestamp representation required by the public dictionary.
product_reviews["review_timestamp"] = pd.to_datetime(
    product_reviews["review_timestamp"]
).dt.strftime("%Y-%m-%d %H:%M:%S")



Product Review transformation row flow
                         stage  rows
   structured JSON review rows  3946
    structured XML review rows  3946
combined normalised candidates  7892
     canonical product reviews  7000

Standardised Product Review preview
 review_id   order_id order_item_id product_id customer_id    review_timestamp language_code  rating  verified_purchase  helpful_votes  review_length_chars  review_word_count  contains_non_latin_script extracted_order_reference extracted_product_sku
HREV000001 HORD000001   HITM0000001    PRD0837    CUS00191 2018-12-16 11:01:00            ru       4               True              4                 1398                210                       True                HORD000001          SKU-CAN00837
HREV000002 HORD000002   HITM0000004    PRD0726    CUS00096 2018-05-23 12:02:00            en       5               True             15                  741                120                      False                HORD000002          S

## 5. Reconcile overlap and verify relationships

Records are compared by stable business key before canonicalisation. The
executable delivery example below records within-source and cross-source
conflicts, retains one consistent row per `delivery_id`, and avoids silent
source precedence.


In [15]:
# EVIDENCE: SEC-5-DELIVERY-RECONCILIATION
def _find_delivery_conflicts(frame, scope):
    comparison_columns = [
        column
        for column in frame.columns
        if column not in {"delivery_id", "_source"}
    ]
    distinct_counts = frame.groupby("delivery_id", sort=False)[
        comparison_columns
    ].nunique(dropna=False)
    conflict_keys = distinct_counts.index[
        distinct_counts.gt(1).any(axis=1)
    ]
    records = []
    for delivery_id in conflict_keys:
        group = frame.loc[
            frame["delivery_id"].eq(delivery_id),
            comparison_columns,
        ]
        fields = [
            column
            for column in comparison_columns
            if group[column].nunique(dropna=False) > 1
        ]
        values = {
            column: group[column]
            .drop_duplicates()
            .astype(str)
            .tolist()[:3]
            for column in fields
        }
        records.append(
            {
                "scope": scope,
                "delivery_id": delivery_id,
                "conflicting_fields": " | ".join(fields),
                "observed_values": repr(values),
            }
        )
    return pd.DataFrame(
        records,
        columns=[
            "scope",
            "delivery_id",
            "conflicting_fields",
            "observed_values",
        ],
    )

delivery_within_source_conflicts = pd.concat(
    [
        _find_delivery_conflicts(
            deliveries_json_normalised, "within JSON"
        ),
        _find_delivery_conflicts(
            deliveries_xml_normalised, "within XML"
        ),
    ],
    ignore_index=True,
)
if not delivery_within_source_conflicts.empty:
    show(
        delivery_within_source_conflicts,
        "Within-source delivery conflicts",
    )
    raise ValueError(
        "Within-source delivery conflicts require resolution before "
        "canonicalisation"
    )

deliveries_json_unique = deliveries_json_normalised.drop_duplicates(
    "delivery_id", keep="first"
).copy()
deliveries_xml_unique = deliveries_xml_normalised.drop_duplicates(
    "delivery_id", keep="first"
).copy()

delivery_overlap_ids = set(
    deliveries_json_unique["delivery_id"]
) & set(deliveries_xml_unique["delivery_id"])
delivery_cross_source_candidates = pd.concat(
    [
        deliveries_json_unique.assign(_source="JSON"),
        deliveries_xml_unique.assign(_source="XML"),
    ],
    ignore_index=True,
)
delivery_cross_source_conflicts = _find_delivery_conflicts(
    delivery_cross_source_candidates.drop(columns="_source"),
    "cross-source JSON/XML",
)
delivery_conflicts = pd.concat(
    [
        delivery_within_source_conflicts,
        delivery_cross_source_conflicts,
    ],
    ignore_index=True,
)
if not delivery_conflicts.empty:
    show(delivery_conflicts, "Normalised delivery conflicts")
    raise ValueError(
        "Cross-source delivery conflicts require resolution before "
        "canonicalisation"
    )

deliveries_preintegration = (
    delivery_cross_source_candidates.drop(columns="_source")
    .drop_duplicates("delivery_id", keep="first")
    .sort_values("delivery_id", kind="mergesort")
    .reset_index(drop=True)
)
expected_delivery_union_count = len(
    set(deliveries_json_unique["delivery_id"])
    | set(deliveries_xml_unique["delivery_id"])
)

delivery_row_flow = pd.DataFrame(
    [
        {
            "stage": "JSON raw",
            "rows": len(deliveries_json_normalised),
            "unique_delivery_ids": deliveries_json_normalised[
                "delivery_id"
            ].nunique(),
            "duplicate_extra_rows": len(deliveries_json_normalised)
            - deliveries_json_normalised["delivery_id"].nunique(),
        },
        {
            "stage": "XML raw",
            "rows": len(deliveries_xml_normalised),
            "unique_delivery_ids": deliveries_xml_normalised[
                "delivery_id"
            ].nunique(),
            "duplicate_extra_rows": len(deliveries_xml_normalised)
            - deliveries_xml_normalised["delivery_id"].nunique(),
        },
        {
            "stage": "Cross-source overlap",
            "rows": len(delivery_overlap_ids),
            "unique_delivery_ids": len(delivery_overlap_ids),
            "duplicate_extra_rows": 0,
        },
        {
            "stage": "Canonical pre-integration",
            "rows": len(deliveries_preintegration),
            "unique_delivery_ids": deliveries_preintegration[
                "delivery_id"
            ].nunique(),
            "duplicate_extra_rows": len(deliveries_preintegration)
            - deliveries_preintegration["delivery_id"].nunique(),
        },
    ]
)

def _clean_delivery_source(frame):
    cleaned = frame.drop(columns="_delivery_note_raw").copy()
    cleaned["delivery_note_clean"] = frame[
        "_delivery_note_raw"
    ].map(clean_narrative_text)
    return cleaned.loc[:, DELIVERY_TARGET_COLUMNS]


deliveries_json_cleaned = _clean_delivery_source(
    deliveries_json_unique
)
deliveries_xml_cleaned = _clean_delivery_source(
    deliveries_xml_unique
)
cleaned_delivery_candidates = pd.concat(
    [
        deliveries_json_cleaned.assign(_source="JSON"),
        deliveries_xml_cleaned.assign(_source="XML"),
    ],
    ignore_index=True,
)
delivery_cleaned_conflicts = _find_delivery_conflicts(
    cleaned_delivery_candidates,
    "cleaned cross-source JSON/XML",
)
if not delivery_cleaned_conflicts.empty:
    show(delivery_cleaned_conflicts, "Cleaned delivery conflicts")
    raise ValueError(
        "Cleaned cross-source delivery conflicts require resolution before "
        "canonicalisation"
    )

deliveries = (
    cleaned_delivery_candidates.drop(columns="_source")
    .drop_duplicates("delivery_id", keep="first")
    .sort_values("delivery_id", kind="mergesort")
    .reset_index(drop=True)
    .loc[:, DELIVERY_TARGET_COLUMNS]
)
if len(deliveries) != expected_delivery_union_count:
    raise ValueError(
        "Canonical delivery row count does not match the data-derived key union"
    )

show(delivery_row_flow, "Delivery row-flow evidence")
print(f"Normalised delivery conflicts: {len(delivery_conflicts)}")
print(f"TASK2_DELIVERIES_READY rows={len(deliveries)}")



Delivery row-flow evidence
                    stage  rows  unique_delivery_ids  duplicate_extra_rows
                 JSON raw  2818                 2750                    68
                  XML raw  2818                 2750                    68
     Cross-source overlap   500                  500                     0
Canonical pre-integration  5000                 5000                     0
Normalised delivery conflicts: 0
TASK2_DELIVERIES_READY rows=5000


## 6. Validation register

Each executable check has a stable `VAL-...` identifier and records its observed
result, `PASS`/`FAIL` status, evidence and resolution or interpretation. Genuine
failures remain visible rather than being converted or hard-coded into passes.

The register below integrates the four allocated owner scopes:

- Lucy — orders/order items keys and arithmetic;
- Jason — schema/types, customers and source coverage;
- Xingao — deliveries and temporal validation;
- Keshu — products/reviews, text and final register integration.


### 6.1 Schema and type checks (`VAL-SCHEMA-...`)


In [16]:
# EVIDENCE: SEC-6.1-SCHEMA-TYPES
# OWNER: Jason — schema, types and missing-value representation.

from numbers import Number
import re
import unicodedata


TASK4_TABLE_ORDER = [
    "orders",
    "order_items",
    "customers",
    "deliveries",
    "products",
    "product_reviews",
]
TASK4_PRIMARY_KEYS = {
    "orders": "order_id",
    "order_items": "order_item_id",
    "customers": "customer_id",
    "deliveries": "delivery_id",
    "products": "product_id",
    "product_reviews": "review_id",
}
TASK4_TABLES = {
    table_name: globals().get(table_name)
    for table_name in TASK4_TABLE_ORDER
}

task4_validation_rows = []


def _add_task4_validation(
    validation_id,
    check,
    observed_result,
    passed,
    pass_resolution,
    fail_resolution,
):
    """Append one auditable validation row with a stable ID."""
    task4_validation_rows.append(
        {
            "validation_id": validation_id,
            "check": check,
            "observed_result": observed_result,
            "status": "PASS" if bool(passed) else "FAIL",
            "resolution_or_interpretation": (
                pass_resolution if bool(passed) else fail_resolution
            ),
        }
    )


def _dictionary_contract(table_name):
    """Return one table's public dictionary rows in published position order."""
    return (
        data_dictionary.loc[
            data_dictionary["output_table"].eq(table_name)
        ]
        .assign(_position=lambda frame: frame["position"].astype(int))
        .sort_values("_position", kind="stable")
        .drop(columns="_position")
        .reset_index(drop=True)
    )


def _value_matches_published_type(value, published_type):
    """Check one non-missing scalar against a dictionary data type."""
    if published_type == "string":
        return isinstance(value, str)
    if published_type == "boolean":
        return isinstance(value, (bool, np.bool_))
    if published_type == "number":
        return (
            isinstance(value, Number)
            and not isinstance(value, (bool, np.bool_))
            and bool(np.isfinite(value))
        )
    if published_type == "date":
        if not isinstance(value, str) or not re.fullmatch(
            r"\d{4}-\d{2}-\d{2}", value
        ):
            return False
        return not pd.isna(
            pd.to_datetime(value, format="%Y-%m-%d", errors="coerce")
        )
    if published_type == "datetime":
        if not isinstance(value, str) or not re.fullmatch(
            r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}", value
        ):
            return False
        return not pd.isna(
            pd.to_datetime(
                value,
                format="%Y-%m-%d %H:%M:%S",
                errors="coerce",
            )
        )
    return False


def _field_type_violations(series, published_type):
    """Count invalid non-missing values and retain only small audit samples."""
    invalid_values = [
        value
        for value in series.tolist()
        if not pd.isna(value)
        and not _value_matches_published_type(value, published_type)
    ]
    return {
        "invalid_count": len(invalid_values),
        "sample": [repr(value) for value in invalid_values[:3]],
    }


schema_section_start = len(task4_validation_rows)
for table_number, table_name in enumerate(TASK4_TABLE_ORDER, start=1):
    frame = TASK4_TABLES[table_name]
    contract = _dictionary_contract(table_name)
    expected_columns = contract["field_name"].tolist()

    table_present = isinstance(frame, pd.DataFrame)
    actual_columns = list(frame.columns) if table_present else []
    schema_passed = table_present and actual_columns == expected_columns
    _add_task4_validation(
        f"VAL-SCHEMA-{table_number:02d}",
        f"{table_name}: exact public-dictionary fields and field order",
        {
            "table_present": table_present,
            "expected_column_count": len(expected_columns),
            "actual_column_count": len(actual_columns),
            "missing_columns": [
                field for field in expected_columns if field not in actual_columns
            ],
            "unexpected_columns": [
                field for field in actual_columns if field not in expected_columns
            ],
            "exact_order": actual_columns == expected_columns,
        },
        schema_passed,
        "The canonical table contains only the published fields in the exact order.",
        "Align the table to public_data_dictionary.csv and remove helper columns before export.",
    )

    type_problems = {}
    if table_present:
        for dictionary_row in contract.itertuples(index=False):
            if dictionary_row.field_name not in frame.columns:
                type_problems[dictionary_row.field_name] = {
                    "invalid_count": len(frame),
                    "sample": ["field missing"],
                }
                continue
            result = _field_type_violations(
                frame[dictionary_row.field_name],
                dictionary_row.data_type,
            )
            if result["invalid_count"]:
                type_problems[dictionary_row.field_name] = result
    type_passed = table_present and not type_problems
    _add_task4_validation(
        f"VAL-TYPE-{table_number:02d}",
        f"{table_name}: values conform to published data types and formats",
        {
            "rows_checked": len(frame) if table_present else 0,
            "fields_checked": len(contract),
            "problem_fields": type_problems,
        },
        type_passed,
        "All non-missing values conform to the dictionary type; dates and datetimes also match the published formats.",
        "Correct the listed field conversions or standardised date/datetime formats and rerun.",
    )

    # Only two fields are nullable in the dictionary, and both prescribe the
    # literal string "NaN". Therefore the six outputs should contain neither
    # Python/pandas missing values nor empty strings in any exported field.
    actual_missing_count = int(frame.isna().sum().sum()) if table_present else 0
    string_fields = contract.loc[
        contract["data_type"].isin(["string", "date", "datetime"]),
        "field_name",
    ].tolist()
    empty_string_count = 0
    literal_nan_count = 0
    if table_present:
        for field in string_fields:
            if field not in frame.columns:
                continue
            values = frame[field].astype("string")
            empty_string_count += int(values.str.strip().eq("").sum())
            literal_nan_count += int(values.eq("NaN").sum())
    missing_passed = (
        table_present
        and actual_missing_count == 0
        and empty_string_count == 0
    )
    _add_task4_validation(
        f"VAL-MISSING-{table_number:02d}",
        f"{table_name}: distinguish literal 'NaN' from empty/None/pandas missing",
        {
            "actual_missing_count": actual_missing_count,
            "empty_string_count": empty_string_count,
            "literal_NaN_count": literal_nan_count,
        },
        missing_passed,
        "No empty or Python/pandas missing output remains; prescribed missing strings use the literal sentinel.",
        "Replace prescribed missing strings with literal 'NaN' and resolve missing required values.",
    )


task4_schema_type_results = pd.DataFrame(
    task4_validation_rows[schema_section_start:]
)
show(task4_schema_type_results, "Task 4 schema/type/missing validations")



Task 4 schema/type/missing validations
 validation_id                                                                     check                                                                                                                                       observed_result status                                                                                 resolution_or_interpretation
 VAL-SCHEMA-01                    orders: exact public-dictionary fields and field order {'table_present': True, 'expected_column_count': 23, 'actual_column_count': 23, 'missing_columns': [], 'unexpected_columns': [], 'exact_order': True}   PASS                                   The canonical table contains only the published fields in the exact order.
   VAL-TYPE-01                orders: values conform to published data types and formats                                                                                    {'rows_checked': 5000, 'fields_checked': 23, 'problem_fields': {}}   PASS All 

In [17]:
# EVIDENCE: SEC-6.1-KESHU-VALIDATION-HELPERS
from datetime import datetime as _validation_datetime
import numbers


KESHU_VALIDATION_COLUMNS = [
    "validation_id",
    "check",
    "observed_result",
    "status",
    "evidence",
    "resolution",
]
keshu_validation_rows = []


def _record_keshu_validation(
    validation_id,
    check,
    passed,
    observed_result,
    evidence,
    pass_interpretation,
    failure_resolution,
):
    """
    Usage:
        _record_keshu_validation("VAL-PROD-PK-01", "...", True, "...", ...)

    Reason:
        Task 4 requires every executable check to have a stable ID, observed
        evidence, PASS/FAIL status, and a resolution or interpretation.

    Expected result:
        One dictionary is appended to `keshu_validation_rows`; validation
        failures remain visible and do not stop the rest of the register.
    """
    if not isinstance(validation_id, str) or not validation_id.startswith("VAL-"):
        raise ValueError("Every validation ID must be a stable VAL-... string.")

    keshu_validation_rows.append(
        {
            "validation_id": validation_id,
            "check": check,
            "observed_result": str(observed_result),
            "status": "PASS" if bool(passed) else "FAIL",
            "evidence": evidence,
            "resolution": (
                pass_interpretation if bool(passed) else failure_resolution
            ),
        }
    )


def _semantic_type_issue_count(series, target_type):
    """
    Usage:
        _semantic_type_issue_count(products["unit_price"], "number")

    Reason:
        CSV-oriented tables can use different pandas storage dtypes while still
        meeting the public dictionary's semantic string, number, date,
        datetime, or boolean contract.

    Expected result:
        The number of values that do not satisfy the requested semantic type.
        Literal `NaN` is accepted only as a string, never as a number/boolean.
    """
    if target_type == "string":
        return int((~series.map(lambda value: isinstance(value, str))).sum())

    if target_type == "boolean":
        return int(
            (~series.map(lambda value: isinstance(value, (bool, np.bool_)))).sum()
        )

    if target_type == "number":
        valid = series.map(
            lambda value: (
                isinstance(value, numbers.Number)
                and not isinstance(value, (bool, np.bool_))
                and bool(np.isfinite(value))
            )
        )
        return int((~valid).sum())

    formats = {
        "date": "%Y-%m-%d",
        "datetime": "%Y-%m-%d %H:%M:%S",
    }
    if target_type in formats:
        parsed = pd.to_datetime(
            series,
            format=formats[target_type],
            errors="coerce",
        )
        exact_text = series.map(lambda value: isinstance(value, str))
        return int((parsed.isna() | ~exact_text).sum())

    raise ValueError(f"Unsupported public data type: {target_type!r}")


def _table_contract_observation(frame, table_name):
    """
    Usage:
        result = _table_contract_observation(products, "products")

    Reason:
        Product and Product Review columns, order, requiredness and semantic
        types must be checked directly against the public data dictionary.

    Expected result:
        A dictionary containing expected/actual columns, missing counts and
        per-field semantic type-issue counts for the requested table.
    """
    dictionary_rows = (
        data_dictionary.loc[
            data_dictionary["output_table"].eq(table_name)
        ]
        .assign(_position=lambda value: value["position"].astype(int))
        .sort_values("_position")
    )
    expected_columns = dictionary_rows["field_name"].tolist()
    type_issues = {
        row.field_name: _semantic_type_issue_count(
            frame[row.field_name],
            row.data_type,
        )
        for row in dictionary_rows.itertuples()
        if row.field_name in frame.columns
    }
    return {
        "expected_columns": expected_columns,
        "actual_columns": frame.columns.tolist(),
        "missing_columns": [
            column for column in expected_columns if column not in frame.columns
        ],
        "extra_columns": [
            column for column in frame.columns if column not in expected_columns
        ],
        "python_missing_values": int(frame.isna().sum().sum()),
        "type_issue_counts": {
            column: count for column, count in type_issues.items() if count
        },
    }


print("KESHU_TASK4_HELPERS_READY")


KESHU_TASK4_HELPERS_READY


In [18]:
# EVIDENCE: SEC-6.1-KESHU-CONTRACTS-TYPES
product_contract = _table_contract_observation(products, "products")
review_contract = _table_contract_observation(product_reviews, "product_reviews")

for table_label, contract, validation_prefix in [
    ("products", product_contract, "PROD"),
    ("product_reviews", review_contract, "REV"),
]:
    schema_passed = (
        not contract["missing_columns"]
        and not contract["extra_columns"]
        and contract["actual_columns"] == contract["expected_columns"]
    )
    _record_keshu_validation(
        f"VAL-{validation_prefix}-SCHEMA-01",
        f"{table_label} contains exactly the public-dictionary columns in order",
        schema_passed,
        {
            "missing_columns": contract["missing_columns"],
            "extra_columns": contract["extra_columns"],
            "column_order_matches": (
                contract["actual_columns"] == contract["expected_columns"]
            ),
        },
        "SEC-6.1-KESHU-CONTRACTS-TYPES",
        "Required Product/Product Review columns and order are present.",
        "Correct the table projection using the ordered public dictionary rows.",
    )
    _record_keshu_validation(
        f"VAL-{validation_prefix}-TYPE-01",
        f"{table_label} values meet public semantic types and required missingness",
        (
            contract["python_missing_values"] == 0
            and not contract["type_issue_counts"]
        ),
        {
            "python_missing_values": contract["python_missing_values"],
            "type_issue_counts": contract["type_issue_counts"],
        },
        "SEC-6.1-KESHU-CONTRACTS-TYPES",
        "Required values are populated and match their semantic target types.",
        "Trace the listed fields to their source parser and conversion rule; do not fill required numeric or boolean values with the string 'NaN'.",
    )

show(
    pd.DataFrame(keshu_validation_rows).tail(4),
    "Keshu Task 4 - Product/Product Review schema and types",
)



Keshu Task 4 - Product/Product Review schema and types
     validation_id                                                                      check                                                            observed_result status                      evidence                                                           resolution
VAL-PROD-SCHEMA-01           products contains exactly the public-dictionary columns in order {'missing_columns': [], 'extra_columns': [], 'column_order_matches': True}   PASS SEC-6.1-KESHU-CONTRACTS-TYPES       Required Product/Product Review columns and order are present.
  VAL-PROD-TYPE-01        products values meet public semantic types and required missingness                      {'python_missing_values': 0, 'type_issue_counts': {}}   PASS SEC-6.1-KESHU-CONTRACTS-TYPES Required values are populated and match their semantic target types.
 VAL-REV-SCHEMA-01    product_reviews contains exactly the public-dictionary columns in order {'missing_columns': [], '

**Observed result/status/interpretation:** The executable rows
displayed immediately above are derived from the current run and are also
retained in the consolidated register in Section 6.7. A failed row remains
visible with its proposed resolution.


### 6.2 Primary- and foreign-key checks (`VAL-PK-...`, `VAL-FK-...`)


In [19]:
# EVIDENCE: SEC-6.2-LUCY-ORDER-KEYS
# OWNER: Lucy — orders/order_items keys and arithmetic.

CURRENCY_TOLERANCE = 0.01
FLOAT_EPSILON = 1e-9

LUCY_VALIDATION_COLUMNS = [
    "validation_id",
    "owner_scope",
    "check",
    "observed_result",
    "status",
    "evidence",
    "resolution_or_interpretation",
]
lucy_validation_rows = []


def record_lucy_validation(
    validation_id,
    check,
    observed_result,
    passed,
    evidence,
    resolution_or_interpretation,
):
    """Record one Lucy-owned executable validation result."""
    lucy_validation_rows.append(
        {
            "validation_id": validation_id,
            "owner_scope": "orders/order_items keys and arithmetic",
            "check": check,
            "observed_result": observed_result,
            "status": "PASS" if bool(passed) else "FAIL",
            "evidence": evidence,
            "resolution_or_interpretation": resolution_or_interpretation,
        }
    )


def show_lucy_validation_results(start_index, title):
    show(pd.DataFrame(lucy_validation_rows[start_index:]), title)


def _key_missing_count(series):
    values = series.astype("string")
    return int(
        (
            values.isna()
            | values.str.strip().isin(["", "NaN"])
        ).sum()
    )


section_start = len(lucy_validation_rows)

for validation_id, table_name, frame, key_column in [
    ("VAL-PK-ORDERS-01", "orders", orders, "order_id"),
    (
        "VAL-PK-ORDER-ITEMS-01",
        "order_items",
        order_items,
        "order_item_id",
    ),
]:
    missing_count = _key_missing_count(frame[key_column])
    duplicate_rows = int(frame[key_column].duplicated(keep=False).sum())
    record_lucy_validation(
        validation_id,
        f"{table_name}.{key_column} is complete and unique",
        {
            "rows": len(frame),
            "missing_count": missing_count,
            "duplicate_rows": duplicate_rows,
        },
        missing_count == 0 and duplicate_rows == 0,
        f"{table_name}.{key_column}",
        (
            "The published business key is complete and unique."
            if missing_count == 0 and duplicate_rows == 0
            else "Trace missing or duplicate keys to source reconciliation; do not invent replacement IDs."
        ),
    )

for validation_id, child_column, parent_name, parent_frame, parent_column in [
    (
        "VAL-FK-ORDER-ITEMS-01",
        "order_id",
        "orders",
        orders,
        "order_id",
    ),
    (
        "VAL-FK-ORDER-ITEMS-02",
        "product_id",
        "products",
        products,
        "product_id",
    ),
]:
    parent_values = set(parent_frame[parent_column].dropna())
    orphan_mask = ~order_items[child_column].isin(parent_values)
    orphan_values = sorted(
        set(order_items.loc[orphan_mask, child_column].astype(str))
    )
    record_lucy_validation(
        validation_id,
        f"order_items.{child_column} references {parent_name}.{parent_column}",
        {
            "child_rows": len(order_items),
            "orphan_rows": int(orphan_mask.sum()),
            "orphan_key_sample": orphan_values[:5],
        },
        not orphan_mask.any(),
        f"order_items.{child_column} -> {parent_name}.{parent_column}",
        (
            "Every order-item foreign key resolves to its canonical parent."
            if not orphan_mask.any()
            else "Investigate the reported orphan keys before export; do not silently drop or replace them."
        ),
    )

show_lucy_validation_results(
    section_start,
    "Lucy Task 4 - orders/order_items key validation",
)



Lucy Task 4 - orders/order_items key validation
        validation_id                            owner_scope                                                 check                                                  observed_result status                                      evidence                                   resolution_or_interpretation
     VAL-PK-ORDERS-01 orders/order_items keys and arithmetic                orders.order_id is complete and unique          {'rows': 5000, 'missing_count': 0, 'duplicate_rows': 0}   PASS                               orders.order_id             The published business key is complete and unique.
VAL-PK-ORDER-ITEMS-01 orders/order_items keys and arithmetic      order_items.order_item_id is complete and unique         {'rows': 15711, 'missing_count': 0, 'duplicate_rows': 0}   PASS                     order_items.order_item_id             The published business key is complete and unique.
VAL-FK-ORDER-ITEMS-01 orders/order_items keys and arithmetic   

In [20]:
# EVIDENCE: SEC-6.2-CUSTOMERS
# OWNER: Jason — customer primary key, semantics and customer relationships.

customer_section_start = len(task4_validation_rows)

customer_pk_missing = int(customers["customer_id"].isna().sum())
customer_pk_duplicates = int(customers["customer_id"].duplicated().sum())
_add_task4_validation(
    "VAL-PK-CUSTOMERS-01",
    "customers.customer_id is a complete unique primary key",
    {
        "rows": len(customers),
        "missing_count": customer_pk_missing,
        "duplicate_count": customer_pk_duplicates,
    },
    customer_pk_missing == 0 and customer_pk_duplicates == 0,
    "One canonical row exists for each non-null customer_id.",
    "Investigate missing IDs or duplicate canonical customer rows before export.",
)


invalid_signup_dates = int(
    (~customers["signup_date"].str.fullmatch(r"\d{4}-\d{2}-\d{2}")).sum()
)
unparseable_signup_dates = int(
    pd.to_datetime(
        customers["signup_date"],
        format="%Y-%m-%d",
        errors="coerce",
    ).isna().sum()
)
invalid_boolean_count = int(
    (~customers["marketing_consent"].map(
        lambda value: isinstance(value, (bool, np.bool_))
    )).sum()
)
negative_prior_orders = int(customers["prior_12m_orders"].lt(0).sum())
negative_lifetime_value = int(
    customers["lifetime_value_before_period"].lt(0).sum()
)
_add_task4_validation(
    "VAL-CUSTOMERS-01",
    "Customer date, boolean and non-negative numeric semantics",
    {
        "invalid_date_format": invalid_signup_dates,
        "unparseable_dates": unparseable_signup_dates,
        "invalid_boolean_values": invalid_boolean_count,
        "negative_prior_12m_orders": negative_prior_orders,
        "negative_lifetime_values": negative_lifetime_value,
    },
    invalid_signup_dates == 0
    and unparseable_signup_dates == 0
    and invalid_boolean_count == 0
    and negative_prior_orders == 0
    and negative_lifetime_value == 0,
    "Customer dates, booleans and numeric measures satisfy their published meanings.",
    "Review the listed customer conversions or invalid negative measures.",
)


normalised_source_customer_ids = {
    unicodedata.normalize("NFC", str(record.get("customerID"))).strip()
    for record in json_customers
}
output_customer_ids = set(customers["customer_id"].astype(str))
missing_customer_profiles = normalised_source_customer_ids - output_customer_ids
unexpected_customers = output_customer_ids - normalised_source_customer_ids

source_postcode_by_customer = {
    unicodedata.normalize("NFC", str(record.get("customerID"))).strip():
    unicodedata.normalize("NFC", str(record.get("homePostcode"))).strip()
    for record in json_customers
}
postcode_mismatch_ids = [
    row.customer_id
    for row in customers[["customer_id", "home_postcode"]].itertuples(index=False)
    if row.customer_id not in source_postcode_by_customer
    or row.home_postcode != source_postcode_by_customer[row.customer_id]
]
_add_task4_validation(
    "VAL-CUSTOMERS-02",
    "Customer source IDs and postcode strings preserve case/leading zeroes",
    {
        "source_unique_ids": len(normalised_source_customer_ids),
        "output_unique_ids": len(output_customer_ids),
        "missing_source_ids": len(missing_customer_profiles),
        "unexpected_output_ids": len(unexpected_customers),
        "postcode_mismatch_count": len(postcode_mismatch_ids),
        "sample_postcode_mismatch_ids": postcode_mismatch_ids[:5],
    },
    not missing_customer_profiles
    and not unexpected_customers
    and not postcode_mismatch_ids,
    "The JSON-only customer population is complete and stable strings were preserved.",
    "Reconcile missing/unexpected customer IDs or restore postcode strings without numeric coercion.",
)


customer_conflict_count = (
    len(customer_conflict_register)
    if "customer_conflict_register" in globals()
    else None
)
_add_task4_validation(
    "VAL-CUSTOMERS-03",
    "Customer duplicate reconciliation has no unresolved field conflicts",
    {
        "conflict_register_present": "customer_conflict_register" in globals(),
        "conflict_count": customer_conflict_count,
    },
    customer_conflict_count == 0,
    "No conflicting non-missing values were silently resolved for duplicate customer IDs.",
    "Review customer_conflict_register and document a justified resolution before canonicalisation.",
)


orders_customer_ids = set(orders["customer_id"].astype(str))
reviews_customer_ids = set(product_reviews["customer_id"].astype(str))
unmatched_order_customers = orders_customer_ids - output_customer_ids
unmatched_review_customers = reviews_customer_ids - output_customer_ids
_add_task4_validation(
    "VAL-FK-CUSTOMERS-01",
    "Final orders and product_reviews customer foreign keys resolve to customers",
    {
        "orders_distinct_customer_ids": len(orders_customer_ids),
        "reviews_distinct_customer_ids": len(reviews_customer_ids),
        "unmatched_order_customer_ids": len(unmatched_order_customers),
        "unmatched_review_customer_ids": len(unmatched_review_customers),
        "sample_unmatched": sorted(
            unmatched_order_customers | unmatched_review_customers
        )[:5],
    },
    not unmatched_order_customers and not unmatched_review_customers,
    "All customer foreign keys in the two referencing canonical tables resolve.",
    "Investigate orphan customer IDs or missing customer profiles before export.",
)


task4_customer_results = pd.DataFrame(
    task4_validation_rows[customer_section_start:]
)
show(task4_customer_results, "Task 4 customer validations")



Task 4 customer validations
      validation_id                                                                       check                                                                                                                                                             observed_result status                                                         resolution_or_interpretation
VAL-PK-CUSTOMERS-01                      customers.customer_id is a complete unique primary key                                                                                                                     {'rows': 500, 'missing_count': 0, 'duplicate_count': 0}   PASS                              One canonical row exists for each non-null customer_id.
   VAL-CUSTOMERS-01                   Customer date, boolean and non-negative numeric semantics                              {'invalid_date_format': 0, 'unparseable_dates': 0, 'invalid_boolean_values': 0, 'negative_prior_12m_orders': 0, 'negative_life

In [21]:
product_pk_observed = {
    "rows": len(products),
    "missing_product_id": int(products["product_id"].isna().sum()),
    "duplicate_product_id_rows": int(products["product_id"].duplicated(keep=False).sum()),
}
_record_keshu_validation(
    "VAL-PROD-PK-01",
    "products.product_id is complete and unique at product grain",
    (
        product_pk_observed["missing_product_id"] == 0
        and product_pk_observed["duplicate_product_id_rows"] == 0
    ),
    product_pk_observed,
    "SEC-6.2-KESHU-KEYS-FKS",
    "There is one canonical row per product_id.",
    "Inspect Product parsing and reconciliation for missing or repeated business keys.",
)

review_pk_observed = {
    "rows": len(product_reviews),
    "missing_review_id": int(product_reviews["review_id"].isna().sum()),
    "duplicate_review_id_rows": int(product_reviews["review_id"].duplicated(keep=False).sum()),
}
_record_keshu_validation(
    "VAL-REV-PK-01",
    "product_reviews.review_id is complete and unique at canonical review grain",
    (
        review_pk_observed["missing_review_id"] == 0
        and review_pk_observed["duplicate_review_id_rows"] == 0
    ),
    review_pk_observed,
    "SEC-6.2-KESHU-KEYS-FKS",
    "There is one canonical row per review_id.",
    "Inspect JSON/XML review reconciliation for missing or repeated business keys.",
)

review_parent_specs = {
    "order_id": ("orders", "order_id"),
    "order_item_id": ("order_items", "order_item_id"),
    "product_id": ("products", "product_id"),
    "customer_id": ("customers", "customer_id"),
}
review_fk_observed = {}
for child_column, (parent_name, parent_column) in review_parent_specs.items():
    parent_frame = globals().get(parent_name)
    if not isinstance(parent_frame, pd.DataFrame) or parent_column not in parent_frame:
        review_fk_observed[child_column] = {
            "dependency": parent_name,
            "dependency_available": False,
            "orphan_value_count": None,
        }
        continue
    orphan_values = set(product_reviews[child_column]) - set(parent_frame[parent_column])
    review_fk_observed[child_column] = {
        "dependency": parent_name,
        "dependency_available": True,
        "orphan_value_count": len(orphan_values),
        "sample_orphans": sorted(orphan_values)[:5],
    }

review_fk_passed = all(
    item["dependency_available"] and item["orphan_value_count"] == 0
    for item in review_fk_observed.values()
)
_record_keshu_validation(
    "VAL-REV-FK-01",
    "Product Review order, item, product and customer foreign keys resolve",
    review_fk_passed,
    review_fk_observed,
    "SEC-6.2-KESHU-KEYS-FKS",
    "Every Product Review child key resolves to its parent table.",
    "Run the missing parent contribution or trace each reported orphan to source reconciliation; do not invent replacement keys.",
)

show(
    pd.DataFrame(keshu_validation_rows).tail(3),
    "Keshu Task 4 - Product/Product Review keys and foreign keys",
)



Keshu Task 4 - Product/Product Review keys and foreign keys
 validation_id                                                                      check                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  observed_result status               evidence                                                   resolution
VAL-PROD-PK-01                products.product_id is complete and unique at product grain                                                                                                                                                                                            

In [22]:
# EVIDENCE: SEC-6.2-DELIVERY-TEMPORAL-SETUP
DELIVERY_VALIDATION_COLUMNS = [
    "validation_id",
    "owner_scope",
    "check",
    "observed_result",
    "status",
    "resolution_or_interpretation",
    "evidence",
]
delivery_temporal_validation_rows = []


def record_delivery_validation(
    validation_id,
    check,
    observed_result,
    passed,
    resolution_or_interpretation,
    evidence,
):
    """Append one delivery/temporal check in the shared register format."""
    delivery_temporal_validation_rows.append(
        {
            "validation_id": validation_id,
            "owner_scope": "deliveries and temporal",
            "check": check,
            "observed_result": json.dumps(
                observed_result,
                ensure_ascii=False,
                sort_keys=True,
                default=str,
            ),
            "status": "PASS" if bool(passed) else "FAIL",
            "resolution_or_interpretation": resolution_or_interpretation,
            "evidence": evidence,
        }
    )


def _missing_required_text(series):
    text = series.astype("string")
    return text.isna() | text.str.strip().eq("")


def _duplicate_group_count(series):
    duplicated = series[series.duplicated(keep=False)]
    return int(duplicated.nunique(dropna=False))


def _sample_ids(frame, mask, id_column, limit=5):
    return frame.loc[mask, id_column].astype(str).head(limit).tolist()


delivery_validation = deliveries.copy()
order_validation = orders[
    ["order_id", "order_status", "order_timestamp"]
].copy()
review_validation = product_reviews[
    ["review_id", "order_id", "review_timestamp"]
].copy()

# Preserve text forms for exact public-format checks before parsing copies.
delivery_date_text = delivery_validation[
    ["dispatch_date", "promised_date", "delivered_date"]
].astype("string")
order_timestamp_text = order_validation["order_timestamp"].astype("string")
review_timestamp_text = review_validation["review_timestamp"].astype("string")

for column in ["dispatch_date", "promised_date", "delivered_date"]:
    delivery_validation[f"_{column}_dt"] = pd.to_datetime(
        delivery_validation[column],
        format="%Y-%m-%d",
        errors="coerce",
    )
order_validation["_order_timestamp_dt"] = pd.to_datetime(
    order_validation["order_timestamp"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce",
)
review_validation["_review_timestamp_dt"] = pd.to_datetime(
    review_validation["review_timestamp"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce",
)

# Deduplicated parent views prevent accidental row multiplication in evidence
# joins. Duplicate delivery/order keys are reported separately below.
order_parent_view = order_validation.drop_duplicates(
    "order_id",
    keep="first",
)
delivery_parent_view = delivery_validation[
    ["order_id", "_delivered_date_dt"]
].drop_duplicates("order_id", keep="first")

delivery_order_validation = delivery_validation.merge(
    order_parent_view,
    on="order_id",
    how="left",
    indicator=True,
)
review_delivery_validation = review_validation.merge(
    delivery_parent_view,
    on="order_id",
    how="left",
    indicator=True,
)

print(
    "DELIVERY_TEMPORAL_VALIDATION_SCOPE "
    f"deliveries={len(delivery_validation)}; "
    f"orders={len(order_validation)}; reviews={len(review_validation)}"
)


DELIVERY_TEMPORAL_VALIDATION_SCOPE deliveries=5000; orders=5000; reviews=7000


In [23]:
# EVIDENCE: SEC-6.2-DELIVERY-CHECKS
missing_delivery_id = int(
    _missing_required_text(delivery_validation["delivery_id"]).sum()
)
missing_delivery_order_id = int(
    _missing_required_text(delivery_validation["order_id"]).sum()
)
duplicate_delivery_id_groups = _duplicate_group_count(
    delivery_validation["delivery_id"]
)
duplicate_delivery_order_groups = _duplicate_group_count(
    delivery_validation["order_id"]
)
record_delivery_validation(
    "VAL-DEL-01",
    "Delivery primary key and one-delivery-per-order grain",
    {
        "delivery_rows": int(len(delivery_validation)),
        "missing_delivery_id": missing_delivery_id,
        "missing_order_id": missing_delivery_order_id,
        "duplicate_delivery_id_groups": duplicate_delivery_id_groups,
        "duplicate_order_id_groups": duplicate_delivery_order_groups,
    },
    missing_delivery_id == 0
    and missing_delivery_order_id == 0
    and duplicate_delivery_id_groups == 0
    and duplicate_delivery_order_groups == 0,
    (
        "Each canonical delivery has a required unique delivery_id and a "
        "required order_id used once at the completed-order delivery grain."
    ),
    "deliveries: delivery_id, order_id",
)

completed_order_ids = set(
    order_validation.loc[
        order_validation["order_status"].eq("Completed"),
        "order_id",
    ]
)
delivery_order_ids = set(delivery_validation["order_id"])
missing_parent_count = int(
    delivery_order_validation["_merge"].ne("both").sum()
)
non_completed_parent_count = int(
    delivery_order_validation["order_status"].ne("Completed").sum()
)
completed_orders_without_delivery = len(
    completed_order_ids - delivery_order_ids
)
deliveries_without_completed_order = len(
    delivery_order_ids - completed_order_ids
)
record_delivery_validation(
    "VAL-DEL-02",
    "Delivery-to-completed-order relationship",
    {
        "missing_order_parent": missing_parent_count,
        "non_completed_order_parent": non_completed_parent_count,
        "completed_orders_without_delivery": completed_orders_without_delivery,
        "delivery_orders_not_in_completed_set": deliveries_without_completed_order,
    },
    missing_parent_count == 0
    and non_completed_parent_count == 0
    and completed_orders_without_delivery == 0
    and deliveries_without_completed_order == 0,
    (
        "The delivery order_id foreign key resolves to a completed order, and "
        "the two data-derived completed-order key sets agree."
    ),
    "deliveries.order_id -> orders.order_id; orders.order_status",
)

numeric_delay = pd.to_numeric(
    delivery_validation["delay_days"],
    errors="coerce",
)
late_mask = numeric_delay.gt(0)
on_time_mismatch = int(
    (
        delivery_validation["on_time_in_full"]
        != numeric_delay.eq(0)
    ).sum()
)
non_delivered_status = int(
    (~delivery_validation["delivery_status"].eq("Delivered")).sum()
)
reason_tokens = (
    delivery_validation["delay_reason"]
    .astype("string")
    .str.strip()
    .str.lower()
)
late_reason_missing = int(
    (
        late_mask
        & reason_tokens.isin(["", "nan", "none"])
    ).sum()
)
record_delivery_validation(
    "VAL-DEL-03",
    "Completed-delivery status, on-time flag and delayed-reason consistency",
    {
        "delivered_status_violations": non_delivered_status,
        "on_time_flag_mismatches": on_time_mismatch,
        "late_rows": int(late_mask.sum()),
        "late_rows_without_reason": late_reason_missing,
    },
    non_delivered_status == 0
    and on_time_mismatch == 0
    and late_reason_missing == 0,
    (
        "The completed-delivery grain is represented as Delivered; "
        "on_time_in_full agrees with zero delay, and delayed rows retain a reason."
    ),
    (
        "deliveries: delivery_status, delay_days, on_time_in_full, "
        "delay_reason"
    ),
)

range_fields = [
    "delay_days",
    "fulfilment_hours",
    "delivery_cost",
    "promised_days",
    "tracking_event_count",
    "shipping_distance_km",
    "estimated_carbon_kg",
]
range_observed = {}
range_invalid_count = 0
for field in range_fields:
    numeric = pd.to_numeric(
        delivery_validation[field],
        errors="coerce",
    )
    invalid = numeric.isna() | numeric.lt(0)
    invalid_count = int(invalid.sum())
    range_invalid_count += invalid_count
    range_observed[field] = {
        "min": None if numeric.dropna().empty else float(numeric.min()),
        "max": None if numeric.dropna().empty else float(numeric.max()),
        "non_numeric_or_negative": invalid_count,
    }
record_delivery_validation(
    "VAL-DEL-04",
    "Sensible non-negative operational delivery ranges",
    {
        "fields": range_observed,
        "total_invalid": range_invalid_count,
    },
    range_invalid_count == 0,
    (
        "Published operational measures are numeric and non-negative. No "
        "unpublished upper bound or certified expected value is hard-coded."
    ),
    "deliveries: operational numeric fields",
)

delivery_integrity_results = pd.DataFrame(
    delivery_temporal_validation_rows,
    columns=DELIVERY_VALIDATION_COLUMNS,
)
show(
    delivery_integrity_results,
    "Xingao Task 4 - delivery integrity validation",
)



Xingao Task 4 - delivery integrity validation
validation_id             owner_scope                                                                  check                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  observed_result status                                                                                                        resolution_or_interpretation                                                               evidence
   VAL-DEL-01 deliveries and temporal                  De

**Observed result/status/interpretation:** The executable rows
displayed immediately above are derived from the current run and are also
retained in the consolidated register in Section 6.7. A failed row remains
visible with its proposed resolution.


### 6.3 Source coverage and reconciliation checks (`VAL-FLOW-...`)


In [24]:
# EVIDENCE: SEC-6.3-SOURCE-COVERAGE
# OWNER: Jason — six-table source-key coverage.


def _stable_source_id(value):
    """Normalise a source business key without changing case or zeroes."""
    if value is None:
        return None
    result = unicodedata.normalize("NFC", str(value)).strip()
    return result if result else None


source_key_components = {
    "orders": {
        "JSON": {
            _stable_source_id(order["header"].get("orderID"))
            for order in json_orders
        },
        "XML": {
            _stable_source_id(record.get("Order_ID"))
            for record in xml_header_records
        },
    },
    "order_items": {
        "JSON": {
            _stable_source_id(record.get("orderItemID"))
            for record in json_items
        },
        "XML": {
            _stable_source_id(record.get("Order_Item_ID"))
            for record in xml_item_records
        },
    },
    "customers": {
        "JSON": {
            _stable_source_id(record.get("customerID"))
            for record in json_customers
        },
        "XML": set(),  # No full XML customer collection is published.
    },
    "deliveries": {
        "JSON": {
            _stable_source_id(record.get("deliveryID"))
            for record in json_deliveries
        },
        "XML": {
            _stable_source_id(record.get("Delivery_ID"))
            for record in xml_delivery_records
        },
    },
    "products": {
        "JSON": set(),  # JSON has references, not a full product catalogue.
        "XML": {
            _stable_source_id(record.get("Product_ID"))
            for record in xml_product_records
        },
    },
    "product_reviews": {
        "JSON": {
            _stable_source_id(record.get("reviewID"))
            for record in json_reviews
        },
        "XML": {
            _stable_source_id(record.get("Review_ID"))
            for record in xml_review_records
        },
    },
}

flow_section_start = len(task4_validation_rows)
source_coverage_detail_rows = []
for table_number, table_name in enumerate(TASK4_TABLE_ORDER, start=1):
    frame = TASK4_TABLES[table_name]
    primary_key = TASK4_PRIMARY_KEYS[table_name]
    json_keys = source_key_components[table_name]["JSON"] - {None}
    xml_keys = source_key_components[table_name]["XML"] - {None}
    expected_union = json_keys | xml_keys
    overlap = json_keys & xml_keys
    output_keys = set(frame[primary_key].astype(str))
    missing_expected = expected_union - output_keys
    unexpected_output = output_keys - expected_union
    duplicate_output_keys = int(frame[primary_key].duplicated().sum())

    observed = {
        "json_unique_keys": len(json_keys),
        "xml_unique_keys": len(xml_keys),
        "cross_source_overlap": len(overlap),
        "expected_union": len(expected_union),
        "output_rows": len(frame),
        "output_unique_keys": len(output_keys),
        "duplicate_output_keys": duplicate_output_keys,
        "missing_expected_keys": len(missing_expected),
        "unexpected_output_keys": len(unexpected_output),
        "sample_missing": sorted(missing_expected)[:5],
        "sample_unexpected": sorted(unexpected_output)[:5],
    }
    coverage_passed = (
        not missing_expected
        and not unexpected_output
        and duplicate_output_keys == 0
        and len(frame) == len(expected_union)
    )
    _add_task4_validation(
        f"VAL-FLOW-{table_number:02d}",
        f"{table_name}: canonical primary keys equal applicable source-key union",
        observed,
        coverage_passed,
        "All applicable source business keys appear exactly once in the canonical output.",
        "Investigate missing, unexpected or duplicated output keys and repeat field-level reconciliation.",
    )
    source_coverage_detail_rows.append(
        {
            "table": table_name,
            **{
                key: value
                for key, value in observed.items()
                if not key.startswith("sample_")
            },
            "status": "PASS" if coverage_passed else "FAIL",
        }
    )


task4_source_coverage_results = pd.DataFrame(
    task4_validation_rows[flow_section_start:]
)
task4_source_coverage_detail = pd.DataFrame(source_coverage_detail_rows)
show(task4_source_coverage_detail, "Task 4 source-coverage detail")
show(task4_source_coverage_results, "Task 4 source-coverage validations")



Task 4 source-coverage detail
          table  json_unique_keys  xml_unique_keys  cross_source_overlap  expected_union  output_rows  output_unique_keys  duplicate_output_keys  missing_expected_keys  unexpected_output_keys status
         orders              2750             2750                   500            5000         5000                5000                      0                      0                       0   PASS
    order_items              8666             8622                  1577           15711        15711               15711                      0                      0                       0   PASS
      customers               500                0                     0             500          500                 500                      0                      0                       0   PASS
     deliveries              2750             2750                   500            5000         5000                5000                      0                      0      

In [25]:
# EVIDENCE: SEC-6.3-KESHU-RANGES-FLOW-RECONCILIATION
product_launch_dates = pd.to_datetime(
    products["launch_date"],
    format="%Y-%m-%d",
    errors="coerce",
)
product_range_issues = {
    "unit_price_not_positive": int(products["unit_price"].le(0).sum()),
    "unit_cost_negative": int(products["unit_cost"].lt(0).sum()),
    "unit_cost_above_price": int(products["unit_cost"].gt(products["unit_price"]).sum()),
    "weight_not_positive": int(products["weight_kg"].le(0).sum()),
    "warranty_negative": int(products["warranty_months"].lt(0).sum()),
    "launch_date_invalid": int(product_launch_dates.isna().sum()),
    "launch_year_date_mismatch": int(
        products["launch_year"].ne(product_launch_dates.dt.year).sum()
    ),
    "sku_format_invalid": int(
        (~products["product_sku"].str.fullmatch(r"SKU-[A-Z0-9]+", na=False)).sum()
    ),
}
_record_keshu_validation(
    "VAL-PROD-RANGE-01",
    "Product numeric values, launch year/date and SKU format are sensible",
    not any(product_range_issues.values()),
    product_range_issues,
    "SEC-6.3-KESHU-RANGES-FLOW-RECONCILIATION",
    "Product measures and structured formats satisfy their business invariants.",
    "Trace each failing Product field to its XML conversion; report genuine source anomalies rather than fabricating a passing value.",
)

review_range_issues = {
    "rating_outside_1_to_5": int((~product_reviews["rating"].between(1, 5)).sum()),
    "rating_non_integral": int(
        product_reviews["rating"].map(lambda value: float(value).is_integer()).eq(False).sum()
    ),
    "helpful_votes_negative": int(product_reviews["helpful_votes"].lt(0).sum()),
    "review_length_negative": int(product_reviews["review_length_chars"].lt(0).sum()),
    "review_word_count_negative": int(product_reviews["review_word_count"].lt(0).sum()),
}
_record_keshu_validation(
    "VAL-REV-RANGE-01",
    "Product Review ratings, votes and derived counts are in sensible ranges",
    not any(review_range_issues.values()),
    review_range_issues,
    "SEC-6.3-KESHU-RANGES-FLOW-RECONCILIATION",
    "Review measures satisfy the published and non-negative range rules.",
    "Trace invalid values to the structured JSON/XML review parser and preserve any genuine anomaly in the register.",
)

# Build source-derived allowed sets instead of hard-coding assessed categories.
product_category_sources = {
    "category": "Category",
    "brand": "Brand",
    "subcategory": "Subcategory",
    "model_family": "Model_Family",
    "colour": "Colour",
    "supplier_id": "Supplier_ID",
    "supplier_country": "Supplier_Country",
    "tax_category": "Tax_Category",
    "package_type": "Package_Type",
}
product_category_issues = {}
for target_column, source_column in product_category_sources.items():
    allowed_values = {
        unicodedata.normalize("NFC", str(record[source_column])).strip()
        for record in xml_product_records
    }
    unexpected = set(products[target_column]) - allowed_values
    product_category_issues[target_column] = {
        "allowed_count_from_source": len(allowed_values),
        "unexpected_count": len(unexpected),
        "sample_unexpected": sorted(unexpected)[:5],
    }

_record_keshu_validation(
    "VAL-PROD-CAT-01",
    "Product structured categories are supported by parsed XML source values",
    all(item["unexpected_count"] == 0 for item in product_category_issues.values()),
    product_category_issues,
    "SEC-6.3-KESHU-RANGES-FLOW-RECONCILIATION",
    "No Product category was invented or changed outside published normalisation.",
    "Review the reported field mapping and normalisation against the structured XML record.",
)

review_category_sources = {
    "language_code": ("languageCode", "Language_Code"),
    "delivery_experience": ("deliveryExperience", "Delivery_Experience"),
    "value_experience": ("valueExperience", "Value_Experience"),
    "writing_style": ("writingStyle", "Writing_Style"),
}
review_category_issues = {}
for target_column, (json_column, xml_column) in review_category_sources.items():
    allowed_values = {
        unicodedata.normalize("NFC", str(record[json_column])).strip()
        for record in json_reviews
    } | {
        unicodedata.normalize("NFC", str(record[xml_column])).strip()
        for record in xml_review_records
    }
    unexpected = set(product_reviews[target_column]) - allowed_values
    review_category_issues[target_column] = {
        "allowed_count_from_sources": len(allowed_values),
        "unexpected_count": len(unexpected),
        "sample_unexpected": sorted(unexpected)[:5],
    }

_record_keshu_validation(
    "VAL-REV-CAT-01",
    "Product Review structured categories are supported by parsed source values",
    all(item["unexpected_count"] == 0 for item in review_category_issues.values()),
    review_category_issues,
    "SEC-6.3-KESHU-RANGES-FLOW-RECONCILIATION",
    "No Review category or language code was inferred from cleaned text.",
    "Review the reported structured field mapping and JSON/XML reconciliation.",
)

product_source_ids = {
    unicodedata.normalize("NFC", str(record["Product_ID"])).strip()
    for record in xml_product_records
}
product_flow_observed = {
    "structured_xml_rows": len(xml_product_records),
    "normalised_candidate_rows": len(product_candidates),
    "unique_source_product_ids": len(product_source_ids),
    "canonical_product_rows": len(products),
}
product_flow_passed = (
    len(product_candidates) == len(xml_product_records)
    and len(products) == len(product_source_ids)
    and set(products["product_id"]) == product_source_ids
)
_record_keshu_validation(
    "VAL-PROD-FLOW-01",
    "Product source rows flow to one canonical row per source-derived product key",
    product_flow_passed,
    product_flow_observed,
    "SEC-6.3-KESHU-RANGES-FLOW-RECONCILIATION",
    "Product row flow is complete and the canonical count is derived from source keys.",
    "Trace missing or additional product_id values through structured XML parsing and reconciliation.",
)

product_conflict_count = len(product_reconciliation_conflicts)
_record_keshu_validation(
    "VAL-PROD-RECON-01",
    "Product duplicate reconciliation has no unresolved field-level conflicts",
    product_conflict_count == 0,
    {"field_level_conflicts": product_conflict_count},
    "SEC-6.3-KESHU-RANGES-FLOW-RECONCILIATION",
    "Matching Product keys are consistent after normalisation.",
    "Retain the conflict evidence and agree a justified treatment; do not apply arbitrary source precedence.",
)

json_review_ids = set(json_product_review_candidates["review_id"])
xml_review_ids = set(xml_product_review_candidates["review_id"])
review_union_ids = json_review_ids | xml_review_ids
review_flow_observed = {
    "structured_json_rows": len(json_reviews),
    "structured_xml_rows": len(xml_review_records),
    "combined_candidate_rows": len(product_review_candidates),
    "json_unique_keys": len(json_review_ids),
    "xml_unique_keys": len(xml_review_ids),
    "cross_source_overlap_keys": len(json_review_ids & xml_review_ids),
    "json_within_source_duplicate_rows": len(json_product_review_candidates) - len(json_review_ids),
    "xml_within_source_duplicate_rows": len(xml_product_review_candidates) - len(xml_review_ids),
    "source_union_keys": len(review_union_ids),
    "canonical_review_rows": len(product_reviews),
}
review_flow_passed = (
    len(product_review_candidates) == len(json_reviews) + len(xml_review_records)
    and len(product_reviews) == len(review_union_ids)
    and set(product_reviews["review_id"]) == review_union_ids
)
_record_keshu_validation(
    "VAL-REV-FLOW-01",
    "Product Review row flow accounts for within-source duplicates and cross-source overlap",
    review_flow_passed,
    review_flow_observed,
    "SEC-6.3-KESHU-RANGES-FLOW-RECONCILIATION",
    "The canonical review count equals the runtime union of parsed source keys.",
    "Trace missing/additional review_id values through structured parsing and key-based reconciliation.",
)

review_conflict_count = len(product_review_reconciliation_conflicts)
_record_keshu_validation(
    "VAL-REV-RECON-01",
    "Product Review overlap has no unresolved normalised field-level conflicts",
    review_conflict_count == 0,
    {"field_level_conflicts": review_conflict_count},
    "SEC-6.3-KESHU-RANGES-FLOW-RECONCILIATION",
    "Matching review_id rows agree after published normalisation.",
    "Retain conflict evidence and agree a justified treatment; do not silently prefer JSON or XML.",
)

show(
    pd.DataFrame(keshu_validation_rows).tail(8),
    "Keshu Task 4 - ranges, categories, row flow and reconciliation",
)



Keshu Task 4 - ranges, categories, row flow and reconciliation
    validation_id                                                                                  check                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

**Observed result/status/interpretation:** The executable rows
displayed immediately above are derived from the current run and are also
retained in the consolidated register in Section 6.7. A failed row remains
visible with its proposed resolution.


### 6.4 Arithmetic checks (`VAL-ARITH-...`)


In [26]:
# EVIDENCE: SEC-6.4-ARITHMETIC
section_start = len(lucy_validation_rows)

expected_line_revenue = (
    order_items["quantity"] * order_items["unit_price"]
).round(2)
line_revenue_difference = (
    order_items["line_revenue"] - expected_line_revenue
).abs()
line_revenue_bad = line_revenue_difference.gt(
    CURRENCY_TOLERANCE + FLOAT_EPSILON
)
record_lucy_validation(
    "VAL-ARITH-01",
    "order_items.line_revenue equals quantity multiplied by unit_price",
    {
        "lines_checked": len(order_items),
        "mismatches": int(line_revenue_bad.sum()),
        "maximum_absolute_difference": float(line_revenue_difference.max()),
        "tolerance": CURRENCY_TOLERANCE,
    },
    not line_revenue_bad.any(),
    "recalculation from canonical order-item fields",
    "If FAIL, inspect quantity/unit parsing and rounding; preserve source evidence.",
)

item_price_by_order = (
    order_items.groupby("order_id", as_index=False)["line_revenue"]
    .sum()
    .rename(columns={"line_revenue": "item_line_sum"})
)
price_check = orders[["order_id", "order_price"]].merge(
    item_price_by_order,
    on="order_id",
    how="left",
    validate="one_to_one",
)
order_price_difference = (
    price_check["order_price"] - price_check["item_line_sum"].round(2)
).abs()
order_price_bad = (
    price_check["item_line_sum"].isna()
    | order_price_difference.gt(CURRENCY_TOLERANCE + FLOAT_EPSILON)
)
record_lucy_validation(
    "VAL-ARITH-02",
    "orders.order_price reconciles to summed order-item line revenue",
    {
        "orders_checked": len(price_check),
        "mismatches": int(order_price_bad.sum()),
        "maximum_absolute_difference": float(order_price_difference.max()),
        "tolerance": CURRENCY_TOLERANCE,
    },
    not order_price_bad.any(),
    "orders joined to grouped canonical order_items",
    "If FAIL, identify missing/duplicate lines or a source arithmetic disagreement.",
)

expected_order_total = (
    orders["order_price"] * (1 - orders["coupon_discount"] / 100)
    + orders["delivery_charges"]
).round(2)
order_total_difference = (orders["order_total"] - expected_order_total).abs()
order_total_bad = order_total_difference.gt(
    CURRENCY_TOLERANCE + FLOAT_EPSILON
)
record_lucy_validation(
    "VAL-ARITH-03",
    "orders.order_total equals discounted price plus delivery charges",
    {
        "orders_checked": len(orders),
        "mismatches": int(order_total_bad.sum()),
        "maximum_absolute_difference": float(order_total_difference.max()),
        "formula": "round(order_price * (1 - coupon_discount / 100) + delivery_charges, 2)",
        "tolerance": CURRENCY_TOLERANCE,
    },
    not order_total_bad.any(),
    "independent recalculation from canonical order fields",
    "If FAIL, verify whether coupon_discount is percentage points and review source rounding.",
)

expected_included_gst = (orders["order_price"] / 11).round(2)
gst_difference = (orders["tax_amount"] - expected_included_gst).abs()
gst_bad = gst_difference.gt(CURRENCY_TOLERANCE + FLOAT_EPSILON)
record_lucy_validation(
    "VAL-ARITH-04",
    "tax_amount is the GST component already included in order_price",
    {
        "orders_checked": len(orders),
        "mismatches": int(gst_bad.sum()),
        "maximum_absolute_difference": float(gst_difference.max()),
        "formula": "round(order_price / 11, 2)",
        "tolerance": CURRENCY_TOLERANCE,
    },
    not gst_bad.any(),
    "Australian 10% GST included-component formula",
    "If FAIL, verify whether the source price is GST-inclusive before changing tax treatment.",
)

incorrect_total_with_tax_added_again = (
    expected_order_total + orders["tax_amount"]
).round(2)
looks_like_tax_added_again = orders["order_total"].sub(
    incorrect_total_with_tax_added_again
).abs().le(CURRENCY_TOLERANCE + FLOAT_EPSILON)
record_lucy_validation(
    "VAL-ARITH-05",
    "GST was not added again when calculating order_total",
    {
        "orders_checked": len(orders),
        "rows_matching_incorrect_tax_added_formula": int(looks_like_tax_added_again.sum()),
        "incorrect_formula": "discounted price + delivery + tax_amount",
        "tolerance": CURRENCY_TOLERANCE,
    },
    not looks_like_tax_added_again.any(),
    "comparison with an explicit incorrect double-GST alternative",
    "If FAIL, remove the second tax addition only after confirming the source total contract.",
)

show_lucy_validation_results(section_start, "Lucy Task 4 - currency and arithmetic validation")



Lucy Task 4 - currency and arithmetic validation
validation_id                            owner_scope                                                             check                                                                                                                                                                       observed_result status                                                     evidence                                                             resolution_or_interpretation
 VAL-ARITH-01 orders/order_items keys and arithmetic order_items.line_revenue equals quantity multiplied by unit_price                                                                                      {'lines_checked': 15711, 'mismatches': 0, 'maximum_absolute_difference': 0.0, 'tolerance': 0.01}   PASS               recalculation from canonical order-item fields           If FAIL, inspect quantity/unit parsing and rounding; preserve source evidence.
 VAL-ARITH-02 orders/order_items k

**Observed result/status/interpretation:** The executable rows
displayed immediately above are derived from the current run and are also
retained in the consolidated register in Section 6.7. A failed row remains
visible with its proposed resolution.


### 6.5 Temporal checks (`VAL-TIME-...`)


In [27]:
# EVIDENCE: SEC-6.5-TEMPORAL-CHECKS
date_format_issues = {
    column: int(
        (~delivery_date_text[column].str.fullmatch(
            r"\d{4}-\d{2}-\d{2}",
            na=False,
        )).sum()
    )
    for column in ["dispatch_date", "promised_date", "delivered_date"]
}
datetime_format_issues = {
    "order_timestamp": int(
        (~order_timestamp_text.str.fullmatch(
            r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}",
            na=False,
        )).sum()
    ),
    "review_timestamp": int(
        (~review_timestamp_text.str.fullmatch(
            r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}",
            na=False,
        )).sum()
    ),
}
parse_issues = {
    "dispatch_date": int(delivery_validation["_dispatch_date_dt"].isna().sum()),
    "promised_date": int(delivery_validation["_promised_date_dt"].isna().sum()),
    "delivered_date": int(delivery_validation["_delivered_date_dt"].isna().sum()),
    "order_timestamp": int(order_validation["_order_timestamp_dt"].isna().sum()),
    "review_timestamp": int(review_validation["_review_timestamp_dt"].isna().sum()),
}
temporal_format_issue_count = (
    sum(date_format_issues.values())
    + sum(datetime_format_issues.values())
    + sum(parse_issues.values())
)
record_delivery_validation(
    "VAL-TIME-01",
    "Published delivery dates and related timestamps are exact and parseable",
    {
        "date_format_issues": date_format_issues,
        "datetime_format_issues": datetime_format_issues,
        "parse_issues": parse_issues,
    },
    temporal_format_issue_count == 0,
    (
        "Delivery dates use YYYY-MM-DD; order/review timestamps use "
        "YYYY-MM-DD HH:MM:SS before temporal comparisons are made."
    ),
    (
        "orders.order_timestamp; deliveries dispatch/promised/delivered dates; "
        "product_reviews.review_timestamp"
    ),
)

order_after_dispatch = (
    delivery_order_validation["_order_timestamp_dt"].dt.normalize()
    > delivery_order_validation["_dispatch_date_dt"]
)
dispatch_after_promised = (
    delivery_order_validation["_dispatch_date_dt"]
    > delivery_order_validation["_promised_date_dt"]
)
dispatch_after_delivered = (
    delivery_order_validation["_dispatch_date_dt"]
    > delivery_order_validation["_delivered_date_dt"]
)
sequence_violation_count = int(
    order_after_dispatch.sum()
    + dispatch_after_promised.sum()
    + dispatch_after_delivered.sum()
)
record_delivery_validation(
    "VAL-TIME-02",
    "Order, dispatch, promised and delivered temporal ordering",
    {
        "order_calendar_date_after_dispatch": int(order_after_dispatch.sum()),
        "dispatch_after_promised": int(dispatch_after_promised.sum()),
        "dispatch_after_delivered": int(dispatch_after_delivered.sum()),
        "sample_order_after_dispatch_ids": _sample_ids(
            delivery_order_validation,
            order_after_dispatch,
            "delivery_id",
        ),
        "sample_dispatch_after_promised_ids": _sample_ids(
            delivery_order_validation,
            dispatch_after_promised,
            "delivery_id",
        ),
        "sample_dispatch_after_delivered_ids": _sample_ids(
            delivery_order_validation,
            dispatch_after_delivered,
            "delivery_id",
        ),
    },
    sequence_violation_count == 0,
    (
        "Order-to-dispatch comparison uses the order calendar date because "
        "dispatch is supplied only as a date. Exact fulfilment_hours cannot be "
        "recomputed without a dispatch time and is therefore range-checked only."
    ),
    (
        "orders.order_timestamp joined to deliveries on order_id; "
        "dispatch_date, promised_date, delivered_date"
    ),
)

expected_promised_days = (
    delivery_validation["_promised_date_dt"]
    - delivery_validation["_dispatch_date_dt"]
).dt.days
signed_delivery_lateness = (
    delivery_validation["_delivered_date_dt"]
    - delivery_validation["_promised_date_dt"]
).dt.days
expected_delay_days = signed_delivery_lateness.clip(lower=0)
expected_on_time = signed_delivery_lateness.le(0)
reported_promised_days = pd.to_numeric(
    delivery_validation["promised_days"],
    errors="coerce",
)
reported_delay_days = pd.to_numeric(
    delivery_validation["delay_days"],
    errors="coerce",
)
promised_days_mismatch = int(
    reported_promised_days.ne(expected_promised_days).sum()
)
delay_days_mismatch = int(
    reported_delay_days.ne(expected_delay_days).sum()
)
on_time_date_mismatch = int(
    delivery_validation["on_time_in_full"].ne(expected_on_time).sum()
)
record_delivery_validation(
    "VAL-TIME-03",
    "Promised-days, delay-days and on-time temporal derivations",
    {
        "promised_days_mismatches": promised_days_mismatch,
        "delay_days_mismatches": delay_days_mismatch,
        "on_time_date_mismatches": on_time_date_mismatch,
        "early_delivery_rows": int(signed_delivery_lateness.lt(0).sum()),
        "on_promised_date_rows": int(signed_delivery_lateness.eq(0).sum()),
        "late_delivery_rows": int(signed_delivery_lateness.gt(0).sum()),
    },
    promised_days_mismatch == 0
    and delay_days_mismatch == 0
    and on_time_date_mismatch == 0,
    (
        "promised_days equals promised minus dispatch days; delay_days is the "
        "positive part of delivered minus promised days, so early delivery is "
        "correctly represented as zero delay."
    ),
    "deliveries: dispatch_date, promised_date, delivered_date and derived fields",
)

review_missing_delivery = review_delivery_validation["_merge"].ne("both")
review_before_delivery = (
    review_delivery_validation["_review_timestamp_dt"]
    < review_delivery_validation["_delivered_date_dt"]
)
valid_review_gaps = (
    review_delivery_validation["_review_timestamp_dt"]
    - review_delivery_validation["_delivered_date_dt"]
).dt.total_seconds().div(3600)
review_temporal_issue_count = int(
    review_missing_delivery.sum() + review_before_delivery.sum()
)
record_delivery_validation(
    "VAL-TIME-04",
    "Product reviews occur after the completed delivery for their order",
    {
        "review_rows": int(len(review_delivery_validation)),
        "reviews_without_delivery_parent": int(review_missing_delivery.sum()),
        "reviews_before_delivered_date": int(review_before_delivery.sum()),
        "minimum_observed_gap_hours": (
            None
            if valid_review_gaps.dropna().empty
            else round(float(valid_review_gaps.min()), 3)
        ),
        "sample_early_review_ids": _sample_ids(
            review_delivery_validation,
            review_before_delivery,
            "review_id",
        ),
    },
    review_temporal_issue_count == 0,
    (
        "Each review resolves to one delivery through order_id and is timestamped "
        "on or after the delivered date. The many-review-to-one-delivery join "
        "uses a deduplicated delivery parent view to prevent row multiplication."
    ),
    "product_reviews.order_id -> deliveries.order_id; review_timestamp, delivered_date",
)


In [28]:
# EVIDENCE: SEC-6.5-DELIVERY-TEMPORAL-REGISTER
delivery_temporal_validation_register = pd.DataFrame(
    delivery_temporal_validation_rows,
    columns=DELIVERY_VALIDATION_COLUMNS,
)
delivery_temporal_status_summary = (
    delivery_temporal_validation_register["status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="checks")
)

show(
    delivery_temporal_validation_register,
    "Delivery and temporal validation register",
)
show(
    delivery_temporal_status_summary,
    "Delivery and temporal validation status summary",
)

failed_delivery_temporal_checks = delivery_temporal_validation_register.loc[
    delivery_temporal_validation_register["status"].eq("FAIL")
]
print(
    "DELIVERY_TEMPORAL_VALIDATION_COMPLETE "
    f"checks={len(delivery_temporal_validation_register)}; "
    f"pass={int(delivery_temporal_validation_register['status'].eq('PASS').sum())}; "
    f"fail={len(failed_delivery_temporal_checks)}"
)
if not failed_delivery_temporal_checks.empty:
    print(
        "Review the recorded evidence and interpretation; do not fabricate "
        "values solely to force a PASS."
    )



Delivery and temporal validation register
validation_id             owner_scope                                                                   check                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  observed_result status                                                                                                                                                                                          resolution_or_interpretation                                                  

**Temporal interpretation notes**

- `fulfilment_hours` is range-checked but not recomputed: `dispatch_date` has
  day precision while `order_timestamp` has second precision, so an exact
  dispatch timestamp is unavailable.
- Early delivery is valid. It gives `delay_days = 0` and
  `on_time_in_full = True`; only delivery after the promised date creates a
  positive delay.


**Observed result/status/interpretation:** The executable rows
displayed immediately above are derived from the current run and are also
retained in the consolidated register in Section 6.7. A failed row remains
visible with its proposed resolution.


### 6.6 Text and multilingual checks (`VAL-TEXT-...`)


In [29]:
# EVIDENCE: SEC-6.6-KESHU-TEXT-MULTILINGUAL-NAN
task3_test_observed = {
    "public_passed": int(task3_public_results["passed"].sum()),
    "public_total": len(task3_public_results),
    "student_passed": int(task3_student_results["passed"].sum()),
    "student_total": len(task3_student_results),
    "interface_passed": int(task3_interface_results["passed"].sum()),
    "interface_total": len(task3_interface_results),
}
task3_tests_passed = (
    task3_test_observed["public_passed"] == task3_test_observed["public_total"]
    and task3_test_observed["student_passed"] == task3_test_observed["student_total"]
    and task3_test_observed["interface_passed"] == task3_test_observed["interface_total"]
)
_record_keshu_validation(
    "VAL-TEXT-TEST-01",
    "All public, student-designed and fixed-interface Task 3 text tests pass",
    task3_tests_passed,
    task3_test_observed,
    "SEC-3.2-TEXT-FUNCTION-TESTS",
    "The shared text module meets the published examples and additional boundary cases.",
    "Correct the shared pure function named by the failing case, then rerun from a fresh kernel.",
)

clean_bodies = product_reviews["review_body_clean"]
expected_lengths = clean_bodies.map(
    lambda value: 0 if value == "NaN" else len(value)
)
expected_word_counts = clean_bodies.map(
    lambda value: 0 if value == "NaN" else len(value.split())
)
derived_measure_issues = {
    "review_length_mismatches": int(
        product_reviews["review_length_chars"].ne(expected_lengths).sum()
    ),
    "review_word_count_mismatches": int(
        product_reviews["review_word_count"].ne(expected_word_counts).sum()
    ),
}
_record_keshu_validation(
    "VAL-TEXT-DERIVED-01",
    "Review character and whitespace-token counts are derived from review_body_clean",
    not any(derived_measure_issues.values()),
    derived_measure_issues,
    "SEC-6.6-KESHU-TEXT-MULTILINGUAL-NAN",
    "Review measures use the cleaned multilingual field and preserve sentinel behavior.",
    "Recalculate counts from review_body_clean, treating the literal 'NaN' as a missing-string result rather than three ordinary letters.",
)

expected_review_skus = product_reviews["product_id"].map(
    products.set_index("product_id")["product_sku"]
)
order_reference_present = product_reviews["extracted_order_reference"].ne("NaN")
sku_reference_present = product_reviews["extracted_product_sku"].ne("NaN")
reference_issues = {
    "invalid_order_reference_format": int(
        (
            order_reference_present
            & ~product_reviews["extracted_order_reference"].str.fullmatch(
                r"(?:HORD|CORD)[0-9]{6}", na=False
            )
        ).sum()
    ),
    "present_order_reference_mismatch": int(
        (
            order_reference_present
            & product_reviews["extracted_order_reference"].ne(product_reviews["order_id"])
        ).sum()
    ),
    "invalid_sku_reference_format": int(
        (
            sku_reference_present
            & ~product_reviews["extracted_product_sku"].str.fullmatch(
                r"SKU-[A-Z0-9]+", na=False
            )
        ).sum()
    ),
    "present_sku_reference_mismatch": int(
        (
            sku_reference_present
            & product_reviews["extracted_product_sku"].ne(expected_review_skus)
        ).sum()
    ),
    "absent_order_reference_sentinels": int((~order_reference_present).sum()),
    "absent_sku_reference_sentinels": int((~sku_reference_present).sum()),
}
reference_passed = all(
    reference_issues[key] == 0
    for key in [
        "invalid_order_reference_format",
        "present_order_reference_mismatch",
        "invalid_sku_reference_format",
        "present_sku_reference_mismatch",
    ]
)
_record_keshu_validation(
    "VAL-TEXT-REF-01",
    "Extracted review order/SKU references are bounded, upper-case and consistent when present",
    reference_passed,
    reference_issues,
    "SEC-6.6-KESHU-TEXT-MULTILINGUAL-NAN",
    "Present embedded references match their structured order and product evidence; absence uses literal 'NaN'.",
    "Inspect raw review extraction boundaries before cleaning; do not convert malformed near-matches into valid references.",
)

recomputed_latin = clean_bodies.map(build_latin_analysis)
recomputed_non_latin = clean_bodies.map(contains_non_latin_script)
multilingual_issues = {
    "latin_analysis_mismatches": int(
        product_reviews["review_body_latin_analysis"].ne(recomputed_latin).sum()
    ),
    "non_latin_indicator_mismatches": int(
        product_reviews["contains_non_latin_script"].ne(recomputed_non_latin).sum()
    ),
    "non_latin_reviews_erased_to_nan": int(
        (
            product_reviews["contains_non_latin_script"]
            & product_reviews["review_body_clean"].eq("NaN")
        ).sum()
    ),
    "reviews_flagged_non_latin": int(
        product_reviews["contains_non_latin_script"].sum()
    ),
    "latin_analysis_nan_results": int(
        product_reviews["review_body_latin_analysis"].eq("NaN").sum()
    ),
}
multilingual_passed = all(
    multilingual_issues[key] == 0
    for key in [
        "latin_analysis_mismatches",
        "non_latin_indicator_mismatches",
        "non_latin_reviews_erased_to_nan",
    ]
)
_record_keshu_validation(
    "VAL-TEXT-MULTI-01",
    "Multilingual review text is preserved and Latin/non-Latin derivatives are reproducible",
    multilingual_passed,
    multilingual_issues,
    "SEC-6.6-KESHU-TEXT-MULTILINGUAL-NAN",
    "review_body_clean preserves multilingual letters and separate derived fields follow the published contract.",
    "Rebuild both derived fields from review_body_clean; do not infer language_code or erase a review because it contains non-Latin letters.",
)

product_string_columns = data_dictionary.loc[
    data_dictionary["output_table"].eq("products")
    & data_dictionary["data_type"].eq("string"),
    "field_name",
].tolist()
review_string_columns = data_dictionary.loc[
    data_dictionary["output_table"].eq("product_reviews")
    & data_dictionary["data_type"].eq("string"),
    "field_name",
].tolist()
non_string_columns = [
    column
    for column in products.columns
    if column not in product_string_columns
] + [
    column
    for column in product_reviews.columns
    if column not in review_string_columns
]

from io import StringIO

# Section 6 precedes final export in the official template. Use an in-memory
# UTF-8-compatible CSV round-trip so this check is reproducible from a clean
# directory and does not depend on stale output files.
_products_csv_buffer = StringIO()
products.to_csv(_products_csv_buffer, index=False)
_products_csv_buffer.seek(0)
products_csv_roundtrip = pd.read_csv(
    _products_csv_buffer,
    keep_default_na=False,
)

_reviews_csv_buffer = StringIO()
product_reviews.to_csv(_reviews_csv_buffer, index=False)
_reviews_csv_buffer.seek(0)
reviews_csv_roundtrip = pd.read_csv(
    _reviews_csv_buffer,
    keep_default_na=False,
)

sentinel_columns = {
    "products.product_description_clean": (
        products["product_description_clean"],
        products_csv_roundtrip["product_description_clean"],
    ),
    "product_reviews.review_body_clean": (
        product_reviews["review_body_clean"],
        reviews_csv_roundtrip["review_body_clean"],
    ),
    "product_reviews.review_body_latin_analysis": (
        product_reviews["review_body_latin_analysis"],
        reviews_csv_roundtrip["review_body_latin_analysis"],
    ),
    "product_reviews.extracted_order_reference": (
        product_reviews["extracted_order_reference"],
        reviews_csv_roundtrip["extracted_order_reference"],
    ),
    "product_reviews.extracted_product_sku": (
        product_reviews["extracted_product_sku"],
        reviews_csv_roundtrip["extracted_product_sku"],
    ),
}
sentinel_roundtrip_mismatches = {
    column: abs(int(memory.eq("NaN").sum()) - int(csv.eq("NaN").sum()))
    for column, (memory, csv) in sentinel_columns.items()
}
literal_nan_issues = {
    "python_missing_in_product_strings": int(products[product_string_columns].isna().sum().sum()),
    "python_missing_in_review_strings": int(product_reviews[review_string_columns].isna().sum().sum()),
    "blank_product_strings": int(
        products[product_string_columns].apply(
            lambda series: series.str.strip().eq("").sum()
        ).sum()
    ),
    "blank_review_strings": int(
        product_reviews[review_string_columns].apply(
            lambda series: series.str.strip().eq("").sum()
        ).sum()
    ),
    "literal_nan_in_numeric_or_boolean_fields": int(
        sum(
            frame[column].map(lambda value: value == "NaN").sum()
            for frame in [products, product_reviews]
            for column in frame.columns
            if column not in (
                product_string_columns if frame is products else review_string_columns
            )
        )
    ),
    "sentinel_csv_roundtrip_count_mismatch": int(
        sum(sentinel_roundtrip_mismatches.values())
    ),
    "nan_clean_body_with_nonzero_measure": int(
        (
            product_reviews["review_body_clean"].eq("NaN")
            & (
                product_reviews["review_length_chars"].ne(0)
                | product_reviews["review_word_count"].ne(0)
            )
        ).sum()
    ),
}
_record_keshu_validation(
    "VAL-TEXT-NAN-01",
    "Prescribed missing strings use literal 'NaN' and survive CSV round-trip",
    not any(literal_nan_issues.values()),
    {
        **literal_nan_issues,
        "literal_nan_counts_in_memory": {
            column: int(memory.eq("NaN").sum())
            for column, (memory, _) in sentinel_columns.items()
        },
    },
    "SEC-6.6-KESHU-TEXT-MULTILINGUAL-NAN",
    "No empty/pandas-missing string is used and literal sentinels remain visible with keep_default_na=False.",
    "Trace the affected string field and write exactly the three characters 'NaN'; never place that string in a required numeric or boolean field.",
)

show(
    pd.DataFrame(keshu_validation_rows).tail(5),
    "Keshu Task 4 - text, references, multilingual and literal NaN",
)



Keshu Task 4 - text, references, multilingual and literal NaN
      validation_id                                                                                     check                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             observed_result status                            evidence                                                                                                  resolution
   VAL-TEXT-TEST-01                   All public, student-designed and fixed-interface Task 3 text tests pass                                           

**Observed result/status/interpretation:** The executable rows
displayed immediately above are derived from the current run and are also
retained in the consolidated register in Section 6.7. A failed row remains
visible with its proposed resolution.


### 6.7 Literal `NaN` reminder

For prescribed missing string outputs, the expected value is the three text
characters `NaN`, not an empty field, Python `None` or a floating-point NaN.
Use `pandas.read_csv(path, keep_default_na=False)` when validating that sentinel.


In [30]:
# EVIDENCE: SEC-6.7-CONSOLIDATED-VALIDATION-REGISTER
VALIDATION_REGISTER_COLUMNS = [
    "validation_id",
    "owner_scope",
    "check",
    "observed_result",
    "status",
    "evidence",
    "resolution_or_interpretation",
]


def _serialise_observed_result(value):
    if isinstance(value, str):
        return value
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        default=str,
    )


def _normalise_component_register(
    frame,
    owner_scope,
    default_evidence,
):
    result = frame.copy()
    if "check_id" in result.columns and "validation_id" not in result.columns:
        result = result.rename(columns={"check_id": "validation_id"})
    if (
        "resolution" in result.columns
        and "resolution_or_interpretation" not in result.columns
    ):
        result = result.rename(
            columns={
                "resolution": "resolution_or_interpretation",
            }
        )
    result["owner_scope"] = owner_scope
    if "evidence" not in result.columns:
        result["evidence"] = default_evidence
    else:
        result["evidence"] = result["evidence"].replace("", pd.NA).fillna(
            default_evidence
        )
    result["observed_result"] = result["observed_result"].map(
        _serialise_observed_result
    )
    return result.loc[:, VALIDATION_REGISTER_COLUMNS]


task4_lucy_validation_register = pd.DataFrame(
    lucy_validation_rows,
    columns=LUCY_VALIDATION_COLUMNS,
)
task4_jason_validation_register = pd.DataFrame(task4_validation_rows)
task4_keshu_validation_register = pd.DataFrame(
    keshu_validation_rows,
    columns=KESHU_VALIDATION_COLUMNS,
)

component_registers = [
    _normalise_component_register(
        task4_lucy_validation_register,
        "orders/order_items keys and arithmetic",
        "SEC-6.2 and SEC-6.4 Lucy contribution",
    ),
    _normalise_component_register(
        task4_jason_validation_register,
        "schema/types, customers and source coverage",
        "SEC-6.1 to SEC-6.3 Jason contribution",
    ),
    _normalise_component_register(
        delivery_temporal_validation_register,
        "deliveries and temporal",
        "SEC-6.2 and SEC-6.5 Xingao contribution",
    ),
    _normalise_component_register(
        task4_keshu_validation_register,
        "products/reviews, text and final-register integration",
        "SEC-6.1 to SEC-6.6 Keshu contribution",
    ),
]

validation_register = (
    pd.concat(component_registers, ignore_index=True)
    .sort_values("validation_id", kind="stable")
    .reset_index(drop=True)
)

duplicate_validation_ids = validation_register[
    "validation_id"
].duplicated(keep=False)
if duplicate_validation_ids.any():
    raise ValueError(
        "Duplicate validation IDs: "
        + repr(
            validation_register.loc[
                duplicate_validation_ids,
                "validation_id",
            ].tolist()
        )
    )

show(validation_register, "Complete Task 4 validation register")

validation_status_summary = (
    validation_register.groupby(
        ["owner_scope", "status"],
        dropna=False,
    )
    .size()
    .rename("checks")
    .reset_index()
)
show(
    validation_status_summary,
    "Task 4 validation status by owner scope",
)

validation_failures = validation_register.loc[
    validation_register["status"].eq("FAIL")
]
if not validation_failures.empty:
    show(
        validation_failures,
        "Genuine failures requiring interpretation or resolution",
    )

print(
    "TASK4_VALIDATION_COMPLETE "
    f"checks={len(validation_register)}; "
    f"pass={int(validation_register['status'].eq('PASS').sum())}; "
    f"fail={int(validation_register['status'].eq('FAIL').sum())}; "
    f"unique_ids={validation_register['validation_id'].is_unique}"
)



Complete Task 4 validation register
        validation_id                                           owner_scope                                                                                     check                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

## 7. Export the six CSV files

Export exactly the public filenames and dictionary field order. UTF-8 preserves
multilingual text, while the literal `NaN` sentinel is written as three text
characters.


In [31]:
# EVIDENCE: SEC-7-FINAL-EXPORT
TASK2_TABLES = {
    "orders": orders,
    "order_items": order_items,
    "customers": customers,
    "deliveries": deliveries,
    "products": products,
    "product_reviews": product_reviews,
}

export_manifest_rows = []
for table_name, frame in TASK2_TABLES.items():
    output_path = OUTPUT_DIR / f"{GROUP_ID}_{table_name}_standardised.csv"
    frame.to_csv(
        output_path,
        index=False,
        encoding="utf-8",
    )
    export_manifest_rows.append(
        {
            "table": table_name,
            "file": output_path.name,
            "rows": len(frame),
            "columns": len(frame.columns),
        }
    )

task2_export_manifest = pd.DataFrame(export_manifest_rows)
show(task2_export_manifest, "Task 2 export manifest")
print("TASK2_EXPORT_COMPLETE 6/6")



Task 2 export manifest
          table                                      file  rows  columns
         orders          Group050_orders_standardised.csv  5000       23
    order_items     Group050_order_items_standardised.csv 15711        6
      customers       Group050_customers_standardised.csv   500       20
     deliveries      Group050_deliveries_standardised.csv  5000       20
       products        Group050_products_standardised.csv  1000       21
product_reviews Group050_product_reviews_standardised.csv  7000       21
TASK2_EXPORT_COMPLETE 6/6


## 8. Final reproducibility record

The read-back check compares each exported file with its in-memory table using
`keep_default_na=False`, which preserves the literal `NaN` sentinel during
verification. Counts are derived from the current run.


In [32]:
# EVIDENCE: SEC-8-REPRODUCIBILITY
from datetime import datetime, timezone

export_readback_rows = []
for table_name, frame in TASK2_TABLES.items():
    output_path = OUTPUT_DIR / f"{GROUP_ID}_{table_name}_standardised.csv"
    exported = pd.read_csv(
        output_path,
        keep_default_na=False,
        dtype=str,
        encoding="utf-8",
    )
    export_readback_rows.append(
        {
            "table": table_name,
            "file_exists": output_path.is_file(),
            "rows_in_memory": len(frame),
            "rows_exported": len(exported),
            "columns_match": list(exported.columns) == list(frame.columns),
            "literal_NaN_cells_preserved": int(exported.eq("NaN").sum().sum()),
            "readback_status": (
                "PASS"
                if len(exported) == len(frame)
                and list(exported.columns) == list(frame.columns)
                else "FAIL"
            ),
        }
    )

export_readback = pd.DataFrame(export_readback_rows)
show(export_readback, "Final UTF-8 CSV read-back")

reproducibility_record = pd.DataFrame(
    [
        {
            "run_completed_utc": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC"),
            "Python": platform.python_version(),
            "pandas": pd.__version__,
            "NumPy": np.__version__,
            "tables_exported": len(export_readback),
            "export_readback_pass": export_readback["readback_status"].eq("PASS").all(),
            "validation_checks": len(validation_register),
            "validation_pass": int(validation_register["status"].eq("PASS").sum()),
            "validation_fail": int(validation_register["status"].eq("FAIL").sum()),
        }
    ]
)
show(reproducibility_record, "Final reproducibility record")



Final UTF-8 CSV read-back
          table  file_exists  rows_in_memory  rows_exported  columns_match  literal_NaN_cells_preserved readback_status
         orders         True            5000           5000           True                         7520            PASS
    order_items         True           15711          15711           True                            0            PASS
      customers         True             500            500           True                            0            PASS
     deliveries         True            5000           5000           True                            0            PASS
       products         True            1000           1000           True                            0            PASS
product_reviews         True            7000           7000           True                            0            PASS

Final reproducibility record
      run_completed_utc Python pandas NumPy  tables_exported  export_readback_pass  validation_checks  